# Evo-1 v40 linear130 — LIBERO_10 resume-only notebook

Faithful copy of the previous research-rigor notebook, but configured only for **resuming the existing `libero_10` run**. It does not search Drive and does not create a new empty folder in resume mode. Cell 13 skips exact completed `.done.json` rows under `RUN_ROOT/results/<mode>/` and runs the missing rows.

Use this notebook to finish the current partial `libero_10` run only. Use the separate spatial-only notebook for a fresh spatial run.


In [1]:
# CELL 01 — GPU check
import torch, os, subprocess, sys, textwrap, json, re, time
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())
else:
    raise RuntimeError("No GPU. Runtime > Change runtime type > GPU")


CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
BF16 supported: True


In [2]:
# CELL 01B — LIBERO_10 resume setup, direct RUN_ROOT, no search
# Run before CELL 02. This points the notebook at the existing partial libero_10 folder.
import os
from pathlib import Path

EXACT_RUN_ROOT = Path("/content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a")
print("EXACT_RUN_ROOT:", EXACT_RUN_ROOT)
if not EXACT_RUN_ROOT.exists():
    raise RuntimeError(f"EXACT_RUN_ROOT does not exist: {EXACT_RUN_ROOT}")

RESULTS_CHECK = EXACT_RUN_ROOT / "results"
if not RESULTS_CHECK.exists():
    raise RuntimeError(f"Expected results/ folder missing: {RESULTS_CHECK}")

done_count = len(list(RESULTS_CHECK.glob("*/*.done.json")))
print("DONE_JSON_COUNT:", done_count)
if done_count == 0:
    raise RuntimeError("Wrong folder: zero .done.json files under results/<mode>/. Resume would skip nothing.")

os.environ["W8DYN_RUN_ROOT"] = str(EXACT_RUN_ROOT)
os.environ["W8DYN_USER_RUN_ID"] = EXACT_RUN_ROOT.name
os.environ["W8DYN_FORCE_NEW_RUN"] = "0"
os.environ["W8A8_SUITES"] = "libero_10"

print("OK_SET_LIBERO10_RESUME_RUN_ROOT")
print("W8DYN_RUN_ROOT =", os.environ["W8DYN_RUN_ROOT"])
print("W8A8_SUITES =", os.environ["W8A8_SUITES"])
print("NEXT: run Cell 02. Cell 13 should skip existing rows and continue with the missing libero_10 rows.")


EXACT_RUN_ROOT: /content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a
DONE_JSON_COUNT: 314
OK_SET_LIBERO10_RESUME_RUN_ROOT
W8DYN_RUN_ROOT = /content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a
W8A8_SUITES = libero_10
NEXT: run Cell 02. Cell 13 should skip existing rows and continue with the missing libero_10 rows.


In [3]:
# # CELL 01C — Optional checkpoint path override for shared/shortcut checkpoint folders
# # Run this before Cell 02 if Cell 02 cannot find /content/drive/MyDrive/Evo-1-checkpoints/Evo1_LIBERO.
# # It does not change results or run folders; it only helps Cell 02 find the checkpoint.

# from pathlib import Path
# import os, subprocess

# DEFAULT_CKPT = Path("/content/drive/MyDrive/Evo-1-checkpoints/Evo1_LIBERO")
# print("DEFAULT_CKPT:", DEFAULT_CKPT, "exists=", DEFAULT_CKPT.exists())

# # If you know the actual shared/shortcut path, set it here manually before Cell 02:
# # os.environ["EVO1_DRIVE_CKPT"] = "/content/drive/MyDrive/YOUR_ACTUAL_Evo1_LIBERO_PATH"

# if "EVO1_DRIVE_CKPT" in os.environ:
#     p = Path(os.environ["EVO1_DRIVE_CKPT"])
#     print("EVO1_DRIVE_CKPT already set:", p, "exists=", p.exists())
# else:
#     candidates = [
#         DEFAULT_CKPT,
#         Path("/content/drive/MyDrive/Evo1_LIBERO"),
#         Path("/content/drive/MyDrive/Evo-1/Evo1_LIBERO"),
#         Path("/content/Evo1_LIBERO"),
#     ]
#     found = [p for p in candidates if p.exists()]
#     if not found:
#         print("SEARCHING for Evo1_LIBERO under MyDrive, maxdepth 6...")
#         cmd = "find /content/drive/MyDrive -maxdepth 6 -type d -name Evo1_LIBERO 2>/dev/null | head -20"
#         out = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE).stdout.strip().splitlines()
#         found = [Path(x) for x in out if x.strip()]

#     if found:
#         os.environ["EVO1_DRIVE_CKPT"] = str(found[0])
#         print("SET EVO1_DRIVE_CKPT:", found[0])
#         if len(found) > 1:
#             print("OTHER CANDIDATES:")
#             for p in found[1:]:
#                 print(" ", p)
#     else:
#         print("NO CKPT FOUND. If the checkpoint is in a shared folder, add a shortcut to MyDrive or set EVO1_DRIVE_CKPT manually.")

# print("FINAL EVO1_DRIVE_CKPT:", os.environ.get("EVO1_DRIVE_CKPT", str(DEFAULT_CKPT)))


In [4]:
# CELL 02 — Workspace setup with strict fresh/resume folder policy, no automatic folder reuse
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os, time, json, subprocess, textwrap, glob, shutil, uuid

# Versioned experiment identity. Result folders must reflect this current notebook/code version.
# Default is LIBERO_10 resume-only in this notebook. Override before Cell 02 with env W8A8_SUITES if needed.
# Default: 1 suite × 10 tasks × 5 episodes × 8 modes = 400 rows.
EXPERIMENT_ID = "v40_linear130_split_ref5ep_windows_cumulative"
DEFAULT_SUITES = [s.strip() for s in os.environ.get("W8A8_SUITES", "libero_10").split(",") if s.strip()]
SUITE_TAG = "_".join(DEFAULT_SUITES).replace("libero_", "libero")
if set(DEFAULT_SUITES) == {"libero_10"}:
    SUITE_TAG = "libero10_resume"
elif set(DEFAULT_SUITES) == {"libero_spatial", "libero_10"}:
    SUITE_TAG = "spatial_libero10"
elif set(DEFAULT_SUITES) == {"libero_spatial", "libero_object", "libero_goal", "libero_10"}:
    SUITE_TAG = "all_libero_suites"
NOTEBOOK_BUILD_ID = f"freshfolder_manifestchecked_fixed_episode_seed_{SUITE_TAG}_researchrigor"
RUN_VERSION = f"{EXPERIMENT_ID}__{NOTEBOOK_BUILD_ID}"

SOURCE_REPO = Path("/content/drive/MyDrive/Evo-1")
DRIVE_CKPT = Path(os.environ.get("EVO1_DRIVE_CKPT", "/content/drive/MyDrive/Evo-1-checkpoints/Evo1_LIBERO"))
BASE_RESULTS = Path(os.environ.get("W8DYN_BASE_RESULTS", "/content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation"))
BASE_RESULTS.mkdir(parents=True, exist_ok=True)

# User controls:
# - Fresh two-suite run: leave W8DYN_RUN_ROOT/W8DYN_USER_RUN_ID unset and W8DYN_FORCE_NEW_RUN unset/1.
# - Resume/add selected suites to an existing folder: preferred path is W8DYN_RUN_ROOT=exact_existing_run_root.
#   This is required for shared folders/shortcuts where BASE_RESULTS/RUN_ID may create a new empty folder.
# - W8DYN_USER_RUN_ID is kept only as a fallback for non-shared folders under BASE_RESULTS.
# - There is intentionally NO automatic "resume latest" behavior.
# - Cell 13 skips only exact done.json rows that match the current manifest/config/result_stem.
_EXACT_RUN_ROOT_RAW = os.environ.get("W8DYN_RUN_ROOT", "").strip()
EXACT_RUN_ROOT = Path(_EXACT_RUN_ROOT_RAW).expanduser().resolve() if _EXACT_RUN_ROOT_RAW else None
_USER_RUN_ID_RAW = os.environ.get("W8DYN_USER_RUN_ID", "").strip()
USER_RUN_ID = _USER_RUN_ID_RAW or None
_FORCE_DEFAULT = "0" if (EXACT_RUN_ROOT or USER_RUN_ID) else "1"
FORCE_NEW_RUN = os.environ.get("W8DYN_FORCE_NEW_RUN", _FORCE_DEFAULT).strip().lower() in {"1", "true", "yes", "y"}
RESUME_LATEST_RUN = False   # disabled by design; do not silently reuse another run folder
CLEAN_RUN_ROOT_ON_FORCE_NEW = True

if RESUME_LATEST_RUN:
    raise RuntimeError(
        "RESUME_LATEST_RUN is disabled in this fixed notebook. "
        "Use W8DYN_RUN_ROOT='exact_existing_run_root' with FORCE_NEW_RUN=False, "
        "or FORCE_NEW_RUN=True for a new folder."
    )

if FORCE_NEW_RUN and (EXACT_RUN_ROOT or USER_RUN_ID):
    raise RuntimeError(
        "Ambiguous controls: resume path/id is set but FORCE_NEW_RUN=True. "
        "For a fresh run, unset W8DYN_RUN_ROOT/W8DYN_USER_RUN_ID. "
        "To resume that folder, set W8DYN_FORCE_NEW_RUN=0."
    )

if FORCE_NEW_RUN:
    # New folder always reflects the current experiment/version and wall-clock start time.
    RUN_ID = f"{RUN_VERSION}__{time.strftime('%Y%m%d_%H%M%S')}__{uuid.uuid4().hex[:8]}"
    RUN_ROOT = (BASE_RESULTS / RUN_ID).resolve()
    RUN_MODE = "fresh_new_folder"
elif EXACT_RUN_ROOT:
    # Critical shared-folder behavior: use the exact existing folder path. Never rebuild BASE_RESULTS/RUN_ID.
    RUN_ROOT = EXACT_RUN_ROOT
    RUN_ID = RUN_ROOT.name
    RUN_MODE = "explicit_resume_exact_run_root"
elif USER_RUN_ID:
    # Fallback for normal MyDrive folders only.
    RUN_ID = str(USER_RUN_ID)
    RUN_ROOT = (BASE_RESULTS / RUN_ID).resolve()
    RUN_MODE = "explicit_resume_base_results_run_id"
else:
    raise RuntimeError(
        "No run selected. Set FORCE_NEW_RUN=True for a fresh versioned folder, "
        "or set W8DYN_RUN_ROOT to the exact existing folder to resume/add suites."
    )

WORK_REPO = SOURCE_REPO  # source import path, no full repo copy
GENERATED = RUN_ROOT / "generated"
REFS = RUN_ROOT / "refs"
LOGS = RUN_ROOT / "logs"
DRIFT = RUN_ROOT / "drift_jsonl"
SUMMARIES = RUN_ROOT / "summaries"
RESULTS = RUN_ROOT / "results"

# Safety check before any cleaning/creation.
if FORCE_NEW_RUN:
    _allowed_parent = BASE_RESULTS.resolve()
    if _allowed_parent not in RUN_ROOT.parents:
        raise RuntimeError(f"Refusing suspicious fresh RUN_ROOT outside BASE_RESULTS: {RUN_ROOT}")
    if RUN_ROOT == _allowed_parent:
        raise RuntimeError("Refusing to use BASE_RESULTS itself as RUN_ROOT")
else:
    # Resume must point to an existing Drive folder. Do not create a new root by accident.
    drive_root = Path("/content/drive").resolve()
    if drive_root not in RUN_ROOT.parents and RUN_ROOT != drive_root:
        raise RuntimeError(f"Refusing resume RUN_ROOT outside /content/drive: {RUN_ROOT}")
    if not RUN_ROOT.exists():
        raise RuntimeError(
            f"RESUME_ABORT: exact RUN_ROOT does not exist: {RUN_ROOT}\n"
            "For shared folders, run CELL 01B to auto-find the real existing folder, or set W8DYN_RUN_ROOT manually."
        )
    if not RESULTS.exists():
        raise RuntimeError(f"RESUME_ABORT: RUN_ROOT exists but has no results/ folder: {RESULTS}")
    existing_done = len(list(RESULTS.rglob("*.done.json")))
    if existing_done == 0:
        raise RuntimeError(
            f"RESUME_ABORT: RUN_ROOT/results contains zero .done.json files: {RESULTS}\n"
            "This would not skip completed runs. You are probably pointing at a newly-created empty folder, not the shared results folder."
        )

# Fresh run means a clean run folder. This removes only this versioned timestamp folder.
if FORCE_NEW_RUN and CLEAN_RUN_ROOT_ON_FORCE_NEW and RUN_ROOT.exists():
    print("FORCE_NEW_RUN_CLEAN_EXISTING_RUN_ROOT:", RUN_ROOT)
    shutil.rmtree(RUN_ROOT)

# In resume mode, this creates only missing subfolders inside the existing exact RUN_ROOT, not a new run root.
for p in [RUN_ROOT, GENERATED, REFS, LOGS, DRIFT, SUMMARIES, RESULTS]:
    p.mkdir(parents=True, exist_ok=True)

# Pin this runtime to this exact folder so later re-bootstrap cells cannot pick a different one.
os.environ["W8DYN_RUN_ID"] = RUN_ID
os.environ["W8DYN_RUN_ROOT"] = str(RUN_ROOT)
os.environ["W8DYN_BASE_RESULTS"] = str(BASE_RESULTS)
os.environ["FLOWA8_RUN_ROOT"] = str(RUN_ROOT)
os.environ["W8DYN_RUN_MODE"] = RUN_MODE
os.environ["W8DYN_EXPERIMENT_ID"] = EXPERIMENT_ID

print("RUN_MODE:", RUN_MODE)
print("DEFAULT_SUITES:", DEFAULT_SUITES)
print("SUITE_TAG:", SUITE_TAG)
print("RUN_ID:", RUN_ID)
print("RUN_ROOT:", RUN_ROOT)
print("SOURCE_REPO:", SOURCE_REPO)
print("WORK_REPO:", WORK_REPO)
print("DRIVE_CKPT:", DRIVE_CKPT)
print("GENERATED:", GENERATED)
print("RESULTS:", RESULTS)
print("DONE_JSON_COUNT_AT_START:", len(list(RESULTS.rglob('*.done.json'))))
print("DRIFT_JSONL_COUNT_AT_START:", len(list(DRIFT.rglob('*.jsonl'))))

if FORCE_NEW_RUN:
    assert len(list(RESULTS.rglob('*.done.json'))) == 0, "Fresh run results folder is not empty"
    assert len(list(DRIFT.rglob('*.jsonl'))) == 0, "Fresh run drift_jsonl folder is not empty"
else:
    assert len(list(RESULTS.rglob('*.done.json'))) > 0, "Resume run cannot see existing done.json files"

if not SOURCE_REPO.exists():
    raise RuntimeError(f"Missing source repo: {SOURCE_REPO}")
if not DRIVE_CKPT.exists():
    raise RuntimeError(f"Missing checkpoint dir: {DRIVE_CKPT}")

print("\nSOURCE REPO STATUS, proof only:")
subprocess.run(["git", "-C", str(SOURCE_REPO), "status", "--short"], text=True)
try:
    subprocess.run(["git", "-C", str(SOURCE_REPO), "rev-parse", "--short", "HEAD"], text=True)
except Exception:
    pass

required = [
    WORK_REPO / "Evo_1/scripts/Evo1.py",
    WORK_REPO / "Evo_1/model/action_head/flow_matching.py",
    WORK_REPO / "LIBERO_evaluation/libero_client_4tasks.py",
]
for p in required:
    print("REQUIRED", p, "exists=", p.exists())
    if not p.exists():
        raise RuntimeError(f"Missing required file: {p}")

run_manifest = {
    "run_id": RUN_ID,
    "run_mode": RUN_MODE,
    "run_root": str(RUN_ROOT),
    "source_repo": str(SOURCE_REPO),
    "work_repo": str(WORK_REPO),
    "generated": str(GENERATED),
    "refs": str(REFS),
    "logs": str(LOGS),
    "drift": str(DRIFT),
    "summaries": str(SUMMARIES),
    "results": str(RESULTS),
    "experiment_id": EXPERIMENT_ID,
    "notebook_build_id": NOTEBOOK_BUILD_ID,
    "run_version": RUN_VERSION,
    "experiment": "v40 clean linear130 W8A8 split; reference + ALL + R0/R1/R2/R3 + CUM_00_15/CUM_00_23; selected suites default libero_spatial + libero_10, 5 episodes/task",
    "folder_policy": "Fresh run creates a new versioned timestamp folder. Resume/add suites requires exact existing W8DYN_RUN_ROOT when using shared folders; Cell 02 aborts instead of creating an empty folder. Fixed per-episode seeds are shared across modes for paired comparison.",
}
(RUN_ROOT / "run_manifest.json").write_text(json.dumps(run_manifest, indent=2))
print("SAVED", RUN_ROOT / "run_manifest.json")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RUN_MODE: explicit_resume_exact_run_root
DEFAULT_SUITES: ['libero_10']
SUITE_TAG: libero10_resume
RUN_ID: v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a
RUN_ROOT: /content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a
SOURCE_REPO: /content/drive/MyDrive/Evo-1
WORK_REPO: /content/drive/MyDrive/Evo-1
DRIVE_CKPT: /content/drive/MyDrive/Evo-1-checkpoints/Evo1_LIBERO
GENERATED: /content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a/generated
RESULTS: /content/drive/MyDrive/Evo

In [20]:
# CELL 02A — FAST selected RUN_ROOT proof, no Drive search, no full JSON scan
# This cell only proves which folder the notebook will use.
# Cell 13 does the real exact-row skip by checking RESULTS/<mode>/<result_stem>.done.json.
from pathlib import Path

print("=" * 110)
print("CELL 02A — FAST RUN_ROOT PROOF")
print("=" * 110)

required = ["RUN_MODE", "RUN_ID", "RUN_ROOT", "RESULTS", "DRIFT", "LOGS", "SUMMARIES"]
missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(f"Missing variables {missing}. Run CELL 02 first.")

RUN_ROOT = Path(RUN_ROOT)
RESULTS = Path(RESULTS)
DRIFT = Path(DRIFT)
LOGS = Path(LOGS)
SUMMARIES = Path(SUMMARIES)

print("RUN_MODE:", RUN_MODE)
print("RUN_ID:", RUN_ID)
print("RUN_ROOT:", RUN_ROOT)
print("RESULTS:", RESULTS)
print("DRIFT:", DRIFT)
print("LOGS:", LOGS)
print("SUMMARIES:", SUMMARIES)

# Only count expected direct result files. No recursive Drive search.
done_count = len(list(RESULTS.glob("*/*.done.json"))) if RESULTS.exists() else 0
print("DONE_JSON_COUNT_UNDER_RESULTS_MODE_FOLDERS:", done_count)

if not FORCE_NEW_RUN and done_count == 0:
    raise RuntimeError(
        "RESUME_ABORT: selected RUN_ROOT/results has zero .done.json files. "
        "This would not skip anything. You are pointing at the wrong folder."
    )

if FORCE_NEW_RUN:
    print("FRESH_RUN_SELECTED: Cell 13 will start a new run in this folder.")
else:
    print("RESUME_SELECTED: Cell 13 will skip exact existing done rows and run missing rows.")

print("OK_FAST_RUN_ROOT_PROOF")


CELL 02A — FAST RUN_ROOT PROOF
RUN_MODE: explicit_resume_exact_run_root
RUN_ID: v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a
RUN_ROOT: /content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a
RESULTS: /content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a/results
DRIFT: /content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a/drift_jsonl
LOGS: /content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__fresh

# Run-control note — LIBERO_10 resume only

This notebook resumes only `libero_10` with **400 manifest rows**. Cell 01B sets `W8DYN_RUN_ROOT` to the exact existing partial run folder. Cell 13 skips exact existing `.done.json` rows and runs missing rows.

No Drive-wide search. No new empty resume folder.

In [6]:
# CELL 03 — Install micromamba and create envs; guarded, skip if already valid
MAMBA = "/content/micromamba/bin/micromamba"
MAMBA_ROOT = "/content/micromamba-root"

if not os.path.exists(MAMBA):
    !wget -qO /tmp/micromamba.tar.bz2 https://micro.mamba.pm/api/micromamba/linux-64/latest
    !mkdir -p /content/micromamba
    !tar -xjf /tmp/micromamba.tar.bz2 -C /content/micromamba bin/micromamba

!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} create -y -n Evo1 python=3.10 pip -c conda-forge || true
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} create -y -n libero python=3.8.13 pip -c conda-forge || true
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} env list


[+] 0.0s
Fetch Shard Index for conda-forge/linux-64                                                ✔ Done (0.1 sec)
Fetch Shard Index for conda-forge/noarch                                                  ✔ Done (0.1 sec)
Fetching and Parsing Packages' Shards                                                           ⧖ Startingwarning  libmamba Failed to fetch shard https://conda.anaconda.org/conda-forge/linux-64/a8870816a70dddd05552177d18606626c677bb753a0909ff763a01422a90d45c.msgpack.zst for package clang-22: Download error (28) Timeout was reached [https://conda.anaconda.org/conda-forge/linux-64/a8870816a70dddd05552177d18606626c677bb753a0909ff763a01422a90d45c.msgpack.zst]
    Operation too slow. Less than 30 bytes/sec transferred the last 60 seconds
Fetching and Parsing Packages' Shards                                                    ✔ Done (61.3 sec)
Using Cached Shard Index for conda-forge/linux-64                                                   ✔ Done
Using Cached Shard Index

In [7]:
# CELL 04 — Install/check Evo-1 server deps + torchao; guarded
# Minimal install guard:
# - Do not run requirements.txt every session if the same requirements were already installed.
# - Do not force-reinstall huggingface-hub if 0.36.2 is already present.
# - Do not upgrade/replace torch on A100/H100.
MAMBA = '/content/micromamba/bin/micromamba'
MAMBA_ROOT = '/content/micromamba-root'
EVO = str(WORK_REPO / 'Evo_1')
import os, subprocess, hashlib
from pathlib import Path

env = {**os.environ, 'MAMBA_ROOT_PREFIX': MAMBA_ROOT}
REQ = Path(EVO) / 'requirements.txt'
STAMP_DIR = RUN_ROOT / 'setup_stamps'
STAMP_DIR.mkdir(parents=True, exist_ok=True)
REQ_STAMP = STAMP_DIR / 'evo1_requirements_Evo1.sha256'
HF_PIN = '0.36.2'
FORCE_EVO1_DEPS = os.environ.get('FORCE_EVO1_DEPS', '0') == '1'

def _run(cmd, **kw):
    return subprocess.run(cmd, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, **kw)

def _show_run(cmd, check=True):
    r = _run(cmd)
    print(r.stdout)
    if check and r.returncode != 0:
        raise RuntimeError('Command failed: ' + ' '.join(map(str, cmd)))
    return r

# Keep pip tooling sane; this is small compared with requirements/flash-attn.
_show_run([MAMBA,'run','-n','Evo1','python','-m','pip','install','-U','pip','setuptools','wheel'])

# requirements.txt guard: install only if requirements hash changed or user forces it.
req_hash = hashlib.sha256(REQ.read_bytes()).hexdigest()
old_hash = REQ_STAMP.read_text().strip() if REQ_STAMP.exists() else None
# Sanity-check: if key packages are missing, reinstall even if hash matches
_probe = __import__('subprocess').run(
    [MAMBA,'run','-n','Evo1','python','-c','import transformers, torch'],
    env=env, stdout=__import__('subprocess').PIPE, stderr=__import__('subprocess').STDOUT
)
if _probe.returncode != 0:
    print('KEY_IMPORT_MISSING: forcing requirements reinstall')
    FORCE_EVO1_DEPS = True

if FORCE_EVO1_DEPS or old_hash != req_hash:
    print('EVO1_REQUIREMENTS_INSTALL: hash changed or force requested')
    print('  old:', old_hash)
    print('  new:', req_hash)
    _show_run([MAMBA,'run','-n','Evo1','python','-m','pip','install','-r',str(REQ)])
    REQ_STAMP.write_text(req_hash)
else:
    print('EVO1_REQUIREMENTS_SKIP: requirements.txt hash unchanged:', req_hash)

# huggingface-hub pin guard: do not force reinstall if already correct.
hf_probe = '''
import sys
try:
    import huggingface_hub
    v = huggingface_hub.__version__
    print("HF_VERSION", v)
    raise SystemExit(0 if v == "''' + HF_PIN + '''" else 42)
except ModuleNotFoundError:
    print("HF_MISSING")
    raise SystemExit(42)
'''
r = _run([MAMBA,'run','-n','Evo1','python','-c',hf_probe])
print(r.stdout)
if r.returncode == 42:
    print('HF_INSTALL_PIN:', HF_PIN)
    _show_run([MAMBA,'run','-n','Evo1','python','-m','pip','install','huggingface-hub=='+HF_PIN])
elif r.returncode != 0:
    raise RuntimeError('huggingface_hub version probe failed')
else:
    print('HF_PIN_OK_SKIP_INSTALL:', HF_PIN)

smi = subprocess.run(
    ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
major = int(smi.stdout.strip().splitlines()[0].split('.')[0]) if smi.returncode == 0 else 12
print('GPU_CAP:', major)

# TorchAO guard.
# Why this exists:
# - Evo-1 requirements pin torch==2.5.1 on A100/H100-style runtimes.
# - Latest torchao releases may require newer torch APIs such as torch.int1 and fail to import.
# - For this notebook we only need W8 weight-only quantization APIs, so we pin a torch-2.5-era torchao
#   on non-Blackwell paths and support both old and new TorchAO API names in the generated server.
TORCHAO_PIN = os.environ.get("TORCHAO_PIN", "0.8.0")

if major >= 12:
    # Blackwell path only. This is the only branch allowed to replace torch.
    print('BLACKWELL_DETECTED: installing CUDA 12.8 torch/torchao')
    subprocess.run([MAMBA,'run','-n','Evo1','python','-m','pip','install','--pre','--force-reinstall','torch','torchvision','torchaudio','torchao','--index-url','https://download.pytorch.org/whl/nightly/cu128'], env=env, check=True)
else:
    # A100/H100/L4 path: keep Evo-1 torch==2.5.1 and pin torchao to a compatible 0.8-era build.
    # Do not install latest torchao here: latest torchao can reference torch.int1 and fail on torch 2.5.1.
    torchao_probe = r'''
import traceback
try:
    import torchao
    from torchao.quantization import quantize_
    try:
        from torchao.quantization import Int8WeightOnlyConfig
        api = "Int8WeightOnlyConfig"
    except Exception:
        from torchao.quantization import int8_weight_only
        api = "int8_weight_only"
    print("TORCHAO_ALREADY_OK", getattr(torchao, "__version__", "unknown"), api, flush=True)
except Exception:
    print("TORCHAO_NEEDS_INSTALL_OR_REPAIR", flush=True)
    traceback.print_exc()
    raise SystemExit(42)
'''
    r = subprocess.run([MAMBA,'run','-n','Evo1','python','-c',torchao_probe], env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(r.stdout)
    if r.returncode == 42:
        pins = [p.strip() for p in os.environ.get("TORCHAO_PIN_CANDIDATES", TORCHAO_PIN + ",0.7.0").split(",") if p.strip()]
        last_out = ""
        for pin in pins:
            print("TORCHAO_INSTALL_PIN_TRY:", pin, "for torch 2.5.x path")
            subprocess.run([MAMBA,'run','-n','Evo1','python','-m','pip','uninstall','-y','torchao'], env=env, check=False)
            inst = subprocess.run([MAMBA,'run','-n','Evo1','python','-m','pip','install','--no-cache-dir','--force-reinstall','--no-deps','torchao=='+pin], env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
            print(inst.stdout)
            if inst.returncode != 0:
                last_out = inst.stdout
                continue
            r2 = subprocess.run([MAMBA,'run','-n','Evo1','python','-c',torchao_probe], env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
            print(r2.stdout)
            if r2.returncode == 0:
                print("TORCHAO_INSTALL_PIN_OK:", pin)
                break
            last_out = r2.stdout
        else:
            raise RuntimeError("Could not install a torchao version compatible with current torch. Last output:\n" + last_out)
    elif r.returncode != 0:
        raise RuntimeError('TorchAO probe failed unexpectedly; see output above.')

# Final import check: print the exact failing module if anything is broken.
check = r'''
import importlib, traceback
mods = ['torch', 'transformers', 'huggingface_hub', 'torchao']
for m in mods:
    print('IMPORT_TEST_START', m, flush=True)
    try:
        mod = importlib.import_module(m)
        print('IMPORT_OK', m, getattr(mod, '__version__', 'unknown'), flush=True)
    except Exception:
        print('IMPORT_FAIL', m, flush=True)
        traceback.print_exc()
        raise
try:
    from torchao.quantization import quantize_, Int8WeightOnlyConfig
    torchao_api = 'Int8WeightOnlyConfig'
except Exception:
    from torchao.quantization import quantize_, int8_weight_only
    torchao_api = 'int8_weight_only'
import torch, huggingface_hub, torchao
print('torch', torch.__version__, 'cuda', torch.version.cuda, flush=True)
print('cap', torch.cuda.get_device_capability() if torch.cuda.is_available() else None, flush=True)
print('torchao', getattr(torchao, '__version__', 'unknown'), 'api', torchao_api, flush=True)
print('hf', huggingface_hub.__version__, flush=True)
print('TORCHAO_API_OK', torchao_api, 'quantize_', flush=True)
'''
r = subprocess.run([MAMBA,'run','-n','Evo1','python','-c',check], env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(r.stdout)
if r.returncode != 0:
    raise RuntimeError('Evo1 import check failed above. Fix the failing import before starting the W8 dynamic A8 server.')



KEY_IMPORT_MISSING: forcing requirements reinstall
EVO1_REQUIREMENTS_INSTALL: hash changed or force requested
  old: None
  new: 9f3647334696e85c48375d3927fff61d32fad49f4e1edb61e9641125e7ad313f
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 38.7 MB/s  0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Installing build dependencies: started
  Installing build dependenc

In [8]:
                                                    # CELL 05 — FlashAttention check/build, guarded
# Goal:
# - import-test first; if FlashAttention already works, skip everything
# - if a matching wheel exists on Drive, install it and skip compile
# - only build once when no matching cached wheel exists
# - save the built wheel to Drive for future Colab sessions
# - avoids reinstalling when the cached/imported binary already works

import os, re, json, subprocess
from pathlib import Path

MAMBA = "/content/micromamba/bin/micromamba"
MAMBA_ROOT = "/content/micromamba-root"
EVO = "/content/drive/MyDrive/Evo-1/Evo_1"
FLASH_WHEEL_ROOT = Path("/content/drive/MyDrive/Evo-1/flash_attn_wheels")
FLASH_WHEEL_ROOT.mkdir(parents=True, exist_ok=True)

env = {**os.environ, "MAMBA_ROOT_PREFIX": MAMBA_ROOT}

def run_evo(cmd, check=False):
    r = subprocess.run(
        [MAMBA, "run", "-n", "Evo1", *cmd],
        cwd=EVO,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(r.stdout)
    if check and r.returncode != 0:
        raise RuntimeError("Command failed: " + " ".join(cmd))
    return r

def flash_import_ok():
    probe = r"""
import traceback
try:
    import flash_attn, flash_attn_2_cuda
    print("FLASH_ATTN_IMPORT_OK", getattr(flash_attn, "__version__", "unknown"), flush=True)
except Exception:
    print("FLASH_ATTN_IMPORT_FAIL", flush=True)
    traceback.print_exc()
    raise SystemExit(42)
"""
    r = run_evo(["python", "-c", probe], check=False)
    return r.returncode == 0

# 1) If current env already imports FlashAttention, do not reinstall/rebuild.
if flash_import_ok():
    print("FLASH_ATTN_SKIP: already importable in Evo1 env")
else:
    # 2) Get exact ABI/GPU key from the Evo1 env.
    info_code = r"""
import json, sys, torch
major, minor = torch.cuda.get_device_capability() if torch.cuda.is_available() else (-1, -1)
print(json.dumps({
    "python": f"{sys.version_info.major}.{sys.version_info.minor}",
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "arch": f"{major}.{minor}",
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
}))
"""
    r = run_evo(["python", "-c", info_code], check=True)
    info = json.loads(r.stdout.strip().splitlines()[-1])
    print("FLASH_ATTN_BUILD_KEY_INFO:", info)

    arch = info["arch"]
    if arch not in {"8.0", "9.0", "10.0", "12.0"}:
        raise RuntimeError(f"This FlashAttention cell is intended for A100/H100 only. Detected arch={arch}, device={info['device']}")

    key_raw = f"py{info['python']}_torch{info['torch']}_cu{info['cuda']}_sm{arch}"
    key = re.sub(r"[^A-Za-z0-9_.-]+", "_", key_raw)
    wheel_dir = FLASH_WHEEL_ROOT / key
    wheel_dir.mkdir(parents=True, exist_ok=True)
    print("FLASH_ATTN_WHEEL_DIR:", wheel_dir)

    cached = sorted(wheel_dir.glob("flash_attn*.whl"))
    if cached:
        print("FLASH_ATTN_CACHED_WHEEL_FOUND:", cached[-1])
        # Import failed, so remove broken package first. Reinstalls only after a failed import probe.
        run_evo(["python", "-m", "pip", "uninstall", "-y", "flash-attn", "flash_attn"], check=False)
        run_evo(["python", "-m", "pip", "install", "--no-deps", str(cached[-1])], check=True)
        if not flash_import_ok():
            raise RuntimeError("Cached FlashAttention wheel installed but import still failed; delete the cached wheel dir and rebuild.")
        print("FLASH_ATTN_CACHED_WHEEL_INSTALL_OK")
    else:
        print("FLASH_ATTN_NO_CACHED_WHEEL: building once, then saving wheel to Drive")
        build_script = f"""
set -euxo pipefail
python -m pip uninstall -y flash-attn flash_attn || true
python -m pip install -U packaging ninja
export MAX_JOBS=4
export TORCH_CUDA_ARCH_LIST="{arch}"
python -m pip wheel --no-build-isolation --no-deps --wheel-dir "{wheel_dir}" flash-attn
wheel="$(ls -t "{wheel_dir}"/flash_attn*.whl | head -n 1)"
python -m pip install --no-deps "$wheel"
python -c "import flash_attn, flash_attn_2_cuda; print('FLASH_ATTN_IMPORT_OK_AFTER_BUILD', flash_attn.__version__)"
"""
        r = run_evo(["bash", "-lc", build_script], check=False)
        if r.returncode != 0:
            raise RuntimeError("FlashAttention wheel build/install failed. See output above.")
        print("FLASH_ATTN_WHEEL_SAVED_TO_DRIVE:", wheel_dir)


FLASH_ATTN_IMPORT_FAIL
Traceback (most recent call last):
  File "<string>", line 4, in <module>
ModuleNotFoundError: No module named 'flash_attn'

{"python": "3.10", "torch": "2.12.0.dev20260407+cu128", "cuda": "12.8", "arch": "12.0", "device": "NVIDIA RTX PRO 6000 Blackwell Server Edition"}

FLASH_ATTN_BUILD_KEY_INFO: {'python': '3.10', 'torch': '2.12.0.dev20260407+cu128', 'cuda': '12.8', 'arch': '12.0', 'device': 'NVIDIA RTX PRO 6000 Blackwell Server Edition'}
FLASH_ATTN_WHEEL_DIR: /content/drive/MyDrive/Evo-1/flash_attn_wheels/py3.10_torch2.12.0.dev20260407_cu128_cu12.8_sm12.0
FLASH_ATTN_CACHED_WHEEL_FOUND: /content/drive/MyDrive/Evo-1/flash_attn_wheels/py3.10_torch2.12.0.dev20260407_cu128_cu12.8_sm12.0/flash_attn-2.8.3-cp310-cp310-linux_x86_64.whl

Processing /content/drive/MyDrive/Evo-1/flash_attn_wheels/py3.10_torch2.12.0.dev20260407_cu128_cu12.8_sm12.0/flash_attn-2.8.3-cp310-cp310-linux_x86_64.whl

FLASH_ATTN_IMPORT_OK 2.8.3

FLASH_ATTN_CACHED_WHEEL_INSTALL_OK


In [9]:
# CELL 06 — Install LIBERO env, guarded
MAMBA = "/content/micromamba/bin/micromamba"
MAMBA_ROOT = "/content/micromamba-root"
LIBERO_EVAL = str(WORK_REPO / "LIBERO_evaluation")

!cd "{LIBERO_EVAL}" && test -d LIBERO || git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n libero python -m pip install -U "pip<25.1" "setuptools<76" wheel
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n libero python -m pip install "numpy<1.24" "protobuf<4"
!cd "{LIBERO_EVAL}/LIBERO" && MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n libero python -m pip install -r requirements.txt
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n libero python -m pip install torch==1.11.0+cu113 torchvision==0.12.0+cu113 torchaudio==0.11.0 --extra-index-url https://download.pytorch.org/whl/cu113
!cd "{LIBERO_EVAL}/LIBERO" && MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n libero python -m pip install -e .
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n libero python -m pip install websockets==13.1 huggingface_hub imageio imageio-ffmpeg opencv-python


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 104.3 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.3.0
    Uninstalling setuptools-75.3.0:
      Successfully uninstalled setuptools-75.3.0
  Attempting uninstall: pip
    Found existing installation: pip 24.3.1
    Uninstalling pip-24.3.1:
      Successfully uninstalled pip-24.3.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 25.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 94.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.2/829.2 kB 47.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 734.5/734.5 kB 57.3 MB/s eta 0:00:00
  Installing build dependencies ... done

In [10]:

# CELL 07 — Checkpoint: reuse if present, download only if missing
from pathlib import Path
import os, subprocess

CKPT_DIR = Path("/content/drive/MyDrive/Evo-1-checkpoints/Evo1_LIBERO")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# A rough presence check: checkpoint folder should contain model shards / safetensors / bin / config files.
ckpt_files = list(CKPT_DIR.rglob("*.safetensors")) + list(CKPT_DIR.rglob("*.bin")) + list(CKPT_DIR.rglob("config*.json"))
print("CKPT_DIR:", CKPT_DIR)
print("EXISTING_CKPT_FILE_COUNT:", len(ckpt_files))

if len(ckpt_files) >= 3:
    print("CHECKPOINT_FOUND_DOWNLOAD_SKIPPED")
else:
    print("CHECKPOINT_MISSING_DOWNLOADING")
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id="MINT-SJTU/Evo1_LIBERO", local_dir=str(CKPT_DIR), local_dir_use_symlinks=False)

print("CHECKPOINT_PROOF_SAMPLE")
subprocess.run(f'find "{CKPT_DIR}" -maxdepth 2 -type f | sort | head -50', shell=True, text=True)


CKPT_DIR: /content/drive/MyDrive/Evo-1-checkpoints/Evo1_LIBERO
EXISTING_CKPT_FILE_COUNT: 1
CHECKPOINT_MISSING_DOWNLOADING


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

CHECKPOINT_PROOF_SAMPLE


CompletedProcess(args='find "/content/drive/MyDrive/Evo-1-checkpoints/Evo1_LIBERO" -maxdepth 2 -type f | sort | head -50', returncode=0)

In [11]:

# CELL 08 — LIBERO config
from pathlib import Path
import os, textwrap, subprocess

LIBERO_EVAL = WORK_REPO / "LIBERO_evaluation"
libero_pkg = LIBERO_EVAL / "LIBERO" / "libero" / "libero"
datasets = LIBERO_EVAL / "LIBERO" / "libero" / "datasets"
datasets.mkdir(parents=True, exist_ok=True)
Path.home().joinpath(".libero").mkdir(exist_ok=True)

cfg = f"""benchmark_root: {libero_pkg}
bddl_files: {libero_pkg}/bddl_files
init_states: {libero_pkg}/init_files
datasets: {datasets}
"""
(Path.home()/".libero/config.yaml").write_text(cfg)
print((Path.home()/".libero/config.yaml").read_text())


benchmark_root: /content/drive/MyDrive/Evo-1/LIBERO_evaluation/LIBERO/libero/libero
bddl_files: /content/drive/MyDrive/Evo-1/LIBERO_evaluation/LIBERO/libero/libero/bddl_files
init_states: /content/drive/MyDrive/Evo-1/LIBERO_evaluation/LIBERO/libero/libero/init_files
datasets: /content/drive/MyDrive/Evo-1/LIBERO_evaluation/LIBERO/libero/datasets



In [12]:
print("alive")

alive


In [13]:

# CELL 09 — Optional reference-code/provenance, download only missing Q-DiT quant.py.
# Optional: if this cell is skipped, the generated server v18 uses an internal A8 fake-quant fallback.
# Other methods are recorded as method references for grouping/validation design.
from pathlib import Path
import json, urllib.request

REFS.mkdir(parents=True, exist_ok=True)

refs_manifest = {
    "qdit": {
        "repo": "https://github.com/Juanerx/Q-DiT",
        "raw_quant_py": "https://raw.githubusercontent.com/Juanerx/Q-DiT/main/qdit/quant.py",
        "used_for": "Q-DiT quantize_activation_wrapper for A8 activation fake quantization.",
        "local_file": str(REFS / "qdit_quant_ref.py"),
    },
    "vidit_q": {
        "paper": "https://arxiv.org/abs/2406.02540",
        "repo": "https://github.com/thu-nics/ViDiT-Q",
        "used_for": "Four equal timestep ranges for sensitivity analysis; adapted here as four Evo-1 flow-step A8-only window ablations.",
        "adapted_range_rule": "32 Evo-1 flow steps -> four equal ranges: R0=0-7, R1=8-15, R2=16-23, R3=24-31.",
    },
    "adatsq": {
        "paper": "https://arxiv.org/abs/2602.09883",
        "repo": "https://github.com/Qiushao-E/AdaTSQ",
        "used_for": "End-to-end reconstruction-error-guided temporal sensitivity concept. We do not implement beam search here; we validate fixed 4-window A8 interventions.",
    },
    "taq_dit": {
        "paper": "https://arxiv.org/abs/2411.14172",
        "used_for": "Time-aware quantization motivation: reconstruction and layer/timestep-varying sensitivity matter.",
    },
    "tq_dit": {
        "paper": "https://arxiv.org/abs/2502.04056",
        "used_for": "Time-grouping quantization motivation: activation distributions vary across timesteps; group-specific treatment is needed.",
    },
    "qdrift": {
        "paper": "https://arxiv.org/abs/2603.18095",
        "used_for": "Paired FP-vs-quant timestep drift/variance calibration idea.",
        "note": "Official code not used here.",
    },
    "flash": {
        "paper": "https://arxiv.org/html/2605.13778v1",
        "used_for": "Intermediate flow-state consistency and gripper-risk idea.",
        "note": "Not imported by this notebook.",
    },
}

qdit_ref = REFS / "qdit_quant_ref.py"
url = refs_manifest["qdit"]["raw_quant_py"]
if not qdit_ref.exists():
    print("Downloading Q-DiT quant.py:", url)
    urllib.request.urlretrieve(url, qdit_ref)
else:
    print("Q-DiT quant.py already exists:", qdit_ref)

if not qdit_ref.exists() or qdit_ref.stat().st_size < 1000:
    raise RuntimeError("Q-DiT quant.py download failed or looks too small.")

(REFS / "reference_manifest.json").write_text(json.dumps(refs_manifest, indent=2))
print("SAVED", REFS / "reference_manifest.json")
print("QDIT_REF", qdit_ref, "size", qdit_ref.stat().st_size)
print(json.dumps(refs_manifest, indent=2))


Q-DiT quant.py already exists: /content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a/refs/qdit_quant_ref.py
SAVED /content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a/refs/reference_manifest.json
QDIT_REF /content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a/refs/qdit_quant_ref.py size 10057
{
  "qdit": {
    "repo": "https://github.com/Juanerx/Q-DiT",
    "raw_quant_py": "https://raw.githubusercontent.com/Juanerx/Q-DiT/main/qdit/quant.py",
    "used_for": "Q-DiT quantize_activation_wrapper for A8 activation fake quantization.",
    "local_fil

In [14]:
# CELL 09B — Experiment re-bootstrap for mid-notebook reruns after Colab runtime reset
# Fixed policy: never auto-resume latest. This cell only reconstructs the exact RUN_ID selected by Cell 02.
from pathlib import Path
import os, time, json, subprocess

SOURCE_REPO = Path(os.environ.get("EVO1_WORK_REPO", "/content/drive/MyDrive/Evo-1"))
WORK_REPO = SOURCE_REPO
DRIVE_CKPT = Path(os.environ.get("EVO1_DRIVE_CKPT", "/content/drive/MyDrive/Evo-1-checkpoints/Evo1_LIBERO"))
LOCAL_CKPT = Path(os.environ.get("EVO1_LOCAL_CKPT", "/content/Evo1_LIBERO"))
BASE_RESULTS = Path(os.environ.get("W8DYN_BASE_RESULTS", "/content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation"))
BASE_RESULTS.mkdir(parents=True, exist_ok=True)

EXPERIMENT_ID = os.environ.get("W8DYN_EXPERIMENT_ID", globals().get("EXPERIMENT_ID", "v40_linear130_split_ref5ep_windows_cumulative"))
RUN_ID = os.environ.get("W8DYN_RUN_ID", globals().get("RUN_ID", None))
if RUN_ID is None:
    raise RuntimeError(
        "RUN_ID is not set. Run CELL 02 first. "
        "This fixed notebook refuses to auto-create or auto-resume a folder from CELL 09B."
    )

RUN_ROOT = (BASE_RESULTS / RUN_ID).resolve()
_allowed_parent = BASE_RESULTS.resolve()
if _allowed_parent not in RUN_ROOT.parents:
    raise RuntimeError(f"Refusing suspicious RUN_ROOT outside BASE_RESULTS: {RUN_ROOT}")

GENERATED = RUN_ROOT / "generated"
RESULTS = RUN_ROOT / "results"
SUMMARIES = RUN_ROOT / "summaries"
LOGS = RUN_ROOT / "logs"
REFS = RUN_ROOT / "refs"
DRIFT = RUN_ROOT / "drift_jsonl"
for p in [RUN_ROOT, GENERATED, RESULTS, SUMMARIES, LOGS, REFS, DRIFT, LOCAL_CKPT]:
    p.mkdir(parents=True, exist_ok=True)

SERVER_SCRIPT = GENERATED / "Evo1_server_w8_dynamic_a8_ablation.py"
CLIENT_SCRIPT = GENERATED / "libero_client_w8_dynamic_a8_single_episode.py"
SERVER_LOG = LOGS / "server_w8_dynamic_a8_ablation.log"
MANIFEST_JSON = SUMMARIES / "manifest.json"

os.environ["EVO1_WORK_REPO"] = str(WORK_REPO)
os.environ["EVO1_DRIVE_CKPT"] = str(DRIVE_CKPT)
os.environ["EVO1_LOCAL_CKPT"] = str(LOCAL_CKPT)
os.environ["W8DYN_BASE_RESULTS"] = str(BASE_RESULTS)
os.environ["W8DYN_RUN_ID"] = RUN_ID
os.environ["FLOWA8_RUN_ROOT"] = str(RUN_ROOT)
os.environ["FLOWA8_SITE_MODE"] = "v40_linear130_split_116_global_14_windowed"
os.environ["EVO1_EXPECT_W8_TARGETS"] = "130"
os.environ["EVO1_EXPECT_LINEAR_A8_TARGETS"] = "130"
os.environ["FLOWA8_INCLUDE_STATE_ENCODER_A8"] = "1"

print("REBOOTSTRAP_OK_EXPLICIT_RUN_ONLY")
print("EXPERIMENT_ID", EXPERIMENT_ID)
print("RUN_ID", RUN_ID)
print("RUN_ROOT", RUN_ROOT)
print("SOURCE_REPO", SOURCE_REPO, "exists=", SOURCE_REPO.exists())
print("WORK_REPO", WORK_REPO)
print("DRIVE_CKPT", DRIVE_CKPT, "exists=", DRIVE_CKPT.exists())
print("LOCAL_CKPT", LOCAL_CKPT)
print("BASE_RESULTS", BASE_RESULTS)
print("SERVER_SCRIPT", SERVER_SCRIPT)
print("CLIENT_SCRIPT", CLIENT_SCRIPT)
print("RESULTS", RESULTS)

if not SOURCE_REPO.exists():
    raise RuntimeError(f"Missing source repo: {SOURCE_REPO}")
if not DRIVE_CKPT.exists():
    raise RuntimeError(f"Missing Drive checkpoint dir: {DRIVE_CKPT}")

required = [
    WORK_REPO / "Evo_1/scripts/Evo1.py",
    WORK_REPO / "Evo_1/model/action_head/flow_matching.py",
    WORK_REPO / "LIBERO_evaluation/libero_client_4tasks.py",
]
for p in required:
    print("REQUIRED", p, "exists=", p.exists())
    if not p.exists():
        raise RuntimeError(f"Missing required file: {p}")


REBOOTSTRAP_OK_EXPLICIT_RUN_ONLY
EXPERIMENT_ID v40_linear130_split_ref5ep_windows_cumulative
RUN_ID v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a
RUN_ROOT /content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a
SOURCE_REPO /content/drive/MyDrive/Evo-1 exists= True
WORK_REPO /content/drive/MyDrive/Evo-1
DRIVE_CKPT /content/drive/MyDrive/Evo-1-checkpoints/Evo1_LIBERO exists= True
LOCAL_CKPT /content/Evo1_LIBERO
BASE_RESULTS /content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation
SERVER_SCRIPT /content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a/generated/Evo1_server_w8_dynamic_a8_ablation.py
CL

In [15]:

# CELL 10 — Write generated v40 clean linear-module W8A8 split server/client scripts.
# v18 self-contained backend fix:
# - W8 scope is exactly 130 normal Linear modules: 98 LLM + 32 action_head nn.Linear.
# - A8 scope is normal module-input quantization only: no random tensor sites.
# - Fixed/global A8 group in every A8 mode: 98 LLM linears + 16 action-head FFN linears + 2 state_encoder linears = 116.
# - Flow-windowed A8 group: the remaining 14 action-head flow linears (8 attn.out_proj + action_encoder W1/W2/W3 + seq_pool_proj + mlp_head fc1/fc2).
# - LLM W8 uses torchao; action_head W8 uses fake-dequant int8 weights to avoid torchao Int8Tensor aten.t crash.
# - Client verbose flag + client/server failure separation retained.
from pathlib import Path
import py_compile, textwrap, os

SERVER_SCRIPT = GENERATED / "Evo1_server_w8_dynamic_a8_ablation.py"
CLIENT_SCRIPT = GENERATED / "libero_client_w8_dynamic_a8_single_episode.py"

server_code = r"""
import sys, os, asyncio, websockets, numpy as np, cv2, json, torch, time, traceback, types, math, re, importlib
from pathlib import Path
from PIL import Image
from torchvision import transforms
from types import SimpleNamespace

WORK_REPO = Path(os.environ["EVO1_WORK_REPO"])
RUN_ROOT = Path(os.environ["FLOWA8_RUN_ROOT"])
REFS = RUN_ROOT / "refs"
DRIFT = RUN_ROOT / "drift_jsonl"
DRIFT.mkdir(parents=True, exist_ok=True)
ERROR_LOG = RUN_ROOT / "logs" / "flowa8_server_errors.log"
ERROR_LOG.parent.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(WORK_REPO / "Evo_1"))
sys.path.insert(0, str(REFS))

from scripts.Evo1 import EVO1
try:
    from qdit_quant_ref import quantize_activation_wrapper
    A8_QUANT_BACKEND = "qdit_quant_ref.quantize_activation_wrapper"
    QDIT_IMPORT_ERROR = None
except Exception as _qdit_exc:
    # v18 safety: do not make server startup depend on Cell 09 / an external copied file.
    # Fallback is a standard per-row last-dimension activation fake quantizer.
    A8_QUANT_BACKEND = "internal_per_row_lastdim_fake_quant"
    QDIT_IMPORT_ERROR = repr(_qdit_exc)

    def quantize_activation_wrapper(x, args):
        if not torch.is_tensor(x) or not torch.is_floating_point(x):
            return x
        if int(getattr(args, "abits", 16)) >= 16:
            return x
        orig_dtype = x.dtype
        orig_shape = tuple(x.shape)
        if x.dim() < 2:
            return x
        C = int(x.shape[-1])
        xf = x.contiguous().reshape(-1, C).float()
        nbits = int(getattr(args, "abits", 8))
        a_clip_ratio = float(getattr(args, "a_clip_ratio", 1.0))
        a_sym = bool(getattr(args, "a_sym", False))
        if a_sym:
            qmax = float((1 << (nbits - 1)) - 1)
            max_abs = xf.abs().amax(dim=1, keepdim=True)
            if a_clip_ratio > 0:
                max_abs = max_abs * a_clip_ratio
            scale = torch.clamp(max_abs / qmax, min=1e-8)
            q = torch.round(xf / scale).clamp(-qmax, qmax)
            yf = q * scale
        else:
            qmin = 0.0
            qmax = float((1 << nbits) - 1)
            xmin = xf.amin(dim=1, keepdim=True)
            xmax = xf.amax(dim=1, keepdim=True)
            if a_clip_ratio > 0 and a_clip_ratio < 1.0:
                mid = (xmax + xmin) * 0.5
                half = (xmax - xmin) * 0.5 * a_clip_ratio
                xmin = mid - half
                xmax = mid + half
            scale = torch.clamp((xmax - xmin) / (qmax - qmin), min=1e-8)
            q = torch.round((xf - xmin) / scale).clamp(qmin, qmax)
            yf = q * scale + xmin
        return yf.reshape(orig_shape).to(dtype=orig_dtype, device=x.device)

from torchao.quantization import quantize_
try:
    from torchao.quantization import Int8WeightOnlyConfig
    def _w8_config():
        return Int8WeightOnlyConfig()
    TORCHAO_W8_API = "Int8WeightOnlyConfig"
except Exception:
    from torchao.quantization import int8_weight_only
    def _w8_config():
        return int8_weight_only()
    TORCHAO_W8_API = "int8_weight_only"


class Normalizer:
    def __init__(self, stats_or_path):
        if isinstance(stats_or_path, str):
            with open(stats_or_path, "r") as f:
                stats = json.load(f)
        else:
            stats = stats_or_path

        def pad_to_24(x):
            x = torch.tensor(x, dtype=torch.float32)
            if x.shape[0] < 24:
                x = torch.cat([x, torch.zeros(24 - x.shape[0], dtype=torch.float32)], dim=0)
            elif x.shape[0] > 24:
                raise ValueError(f"Input length {x.shape[0]} exceeds expected 24")
            return x

        if len(stats) != 1:
            raise ValueError(f"norm_stats.json should contain only one robot key, but: {list(stats.keys())}")
        robot_key = list(stats.keys())[0]
        robot_stats = stats[robot_key]
        self.state_min = pad_to_24(robot_stats["observation.state"]["min"])
        self.state_max = pad_to_24(robot_stats["observation.state"]["max"])
        self.action_min = pad_to_24(robot_stats["action"]["min"])
        self.action_max = pad_to_24(robot_stats["action"]["max"])

    def normalize_state(self, state: torch.Tensor) -> torch.Tensor:
        state_min = self.state_min.to(state.device, dtype=state.dtype)
        state_max = self.state_max.to(state.device, dtype=state.dtype)
        return torch.clamp(2 * (state - state_min) / (state_max - state_min + 1e-8) - 1, -1.0, 1.0)

    def denormalize_action(self, action: torch.Tensor) -> torch.Tensor:
        action_min = self.action_min.to(action.device, dtype=action.dtype)
        action_max = self.action_max.to(action.device, dtype=action.dtype)
        if action.ndim == 1:
            action = action.view(1, -1)
        return (action + 1.0) / 2.0 * (action_max - action_min + 1e-8) + action_min


def _qtypes():
    out = []
    for mod, cls in [
        ("torchao.quantization.quant_api", "Int8Tensor"),
        ("torchao.dtypes", "Int8Tensor"),
        ("torchao.dtypes", "AffineQuantizedTensor"),
        ("torchao.dtypes.affine_quantized_tensor", "AffineQuantizedTensor"),
    ]:
        try:
            c = getattr(importlib.import_module(mod), cls)
            if c not in out:
                out.append(c)
        except Exception:
            pass
    return tuple(out)
QTYPES = _qtypes()

def _is_w8_weight(w):
    if w is None:
        return False
    if QTYPES and isinstance(w, QTYPES):
        return True
    type_name = type(w).__module__ + "." + type(w).__name__
    # Fallback across TorchAO 0.7/0.8/newer tensor subclass names.
    return ("torchao" in type_name.lower()) and any(s in type_name.lower() for s in ["quant", "int8", "affine"])

LLM_ATTN = re.compile(r"^embedder\.model\.language_model\.(?:model\.layers|layers)\.\d+\.self_attn\.(?:q_proj|k_proj|v_proj|o_proj)$")
LLM_MLP = re.compile(r"^embedder\.model\.language_model\.(?:model\.layers|layers)\.\d+\.mlp\.(?:gate_proj|up_proj|down_proj)$")
ACTION_FFN = re.compile(r"^action_head\.transformer_blocks\.\d+\.ff\.(?:0|2)$")

def target_info(model):
    llm_attn, llm_mlp, action_ffn = [], [], []
    action_linear_all = []
    for n, m in model.named_modules():
        if isinstance(m, torch.nn.Linear):
            if n.startswith("action_head."):
                action_linear_all.append(n)
            if LLM_ATTN.match(n):
                llm_attn.append(n)
            if LLM_MLP.match(n):
                llm_mlp.append(n)
            if ACTION_FFN.match(n):
                action_ffn.append(n)
    targets = sorted(set(llm_attn + llm_mlp + action_linear_all))
    action_linear_excluded = sorted(set(action_linear_all) - set(action_linear_all))
    return {
        "llm_attn": sorted(llm_attn),
        "llm_mlp": sorted(llm_mlp),
        "action_ffn_subset": sorted(action_ffn),
        "action_linear_all": sorted(action_linear_all),
        "action_linear_selected_for_w8_scope": sorted(action_linear_all),
        "action_linear_excluded_from_w8_scope": action_linear_excluded,
        "target": targets,
    }

def _write_w8_target_audit(info, qtype_counts=None):
    path = RUN_ROOT / "summaries" / "w8_target_scope_audit.json"
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = dict(info)
    payload["counts"] = {
        "llm_attn": len(info.get("llm_attn", [])),
        "llm_mlp": len(info.get("llm_mlp", [])),
        "llm_total": len(info.get("llm_attn", [])) + len(info.get("llm_mlp", [])),
        "action_ffn_subset": len(info.get("action_ffn_subset", [])),
        "action_linear_all": len(info.get("action_linear_all", [])),
        "action_linear_selected_for_w8_scope": len(info.get("action_linear_selected_for_w8_scope", [])),
        "action_linear_excluded_from_w8_scope": len(info.get("action_linear_excluded_from_w8_scope", [])),
        "target_selected": len(info.get("target", [])),
        "replaced": len(info.get("replaced", [])),
    }
    payload["expected_env"] = os.environ.get("EVO1_EXPECT_W8_TARGETS", "130")
    payload["scope_note"] = (
        "v15 W8 scope: LLM attention+MLP linears plus ALL action_head Linear modules. "
        "Expected discovered count on Evo-1 LIBERO is 98 LLM + 32 action_head = 130. "
        "Vision/projector/norms/embeddings/lm_head are still excluded."
    )
    payload["qtype_counts"] = qtype_counts or {}
    path.write_text(json.dumps(payload, indent=2))
    print("[W8] TARGET_AUDIT_JSON", path, flush=True)


def fake_quantize_action_head_weights_dequantized(model, action_targets):
    # Numerically apply W8 to action_head Linear weights without torchao tensor subclasses.
    #
    # Why:
    #   Some Evo-1 action_head helper/projection linears call weight.t() explicitly.
    #   torchao Int8Tensor can crash there with unsupported aten.t.
    #   This function still changes the weights to their int8-quantized/dequantized values,
    #   so the ablation measures W8 numerical error, while storage remains a normal Tensor/Parameter.
    mods = dict(model.named_modules())
    replaced = []
    stats = {}
    for name in sorted(action_targets):
        m = mods.get(name)
        if m is None or not isinstance(m, torch.nn.Linear):
            raise RuntimeError(f"Action-head W8 fake target missing/not Linear: {name}")
        with torch.no_grad():
            w = m.weight.detach()
            orig_dtype = w.dtype
            # Per-output-channel symmetric int8 fake quantization.
            wf = w.float()
            if wf.ndim != 2:
                raise RuntimeError(f"Expected Linear weight 2D for {name}, got {tuple(w.shape)}")
            max_abs = wf.abs().amax(dim=1, keepdim=True)
            scale = torch.clamp(max_abs / 127.0, min=1e-8)
            q = torch.round(wf / scale).clamp(-127, 127)
            deq = (q * scale).to(dtype=orig_dtype, device=w.device)
            rms = torch.sqrt(torch.mean((deq.float() - wf) ** 2)).item()
            maxerr = (deq.float() - wf).abs().max().item()
            m.weight.data.copy_(deq)
            replaced.append(name)
            stats[name] = {
                "backend": "fake_dequant_int8_per_output_channel_symmetric",
                "shape": list(w.shape),
                "scale_min": float(scale.min().item()),
                "scale_max": float(scale.max().item()),
                "weight_error_rms": float(rms),
                "weight_error_max_abs": float(maxerr),
                "stored_dtype": str(orig_dtype),
                "storage_note": "stored as normal tensor after int8 quantize/dequantize; numerical W8 effect only, not compressed storage",
            }
    return replaced, stats


def apply_w8_weights(model):
    if os.environ.get("EVO1_APPLY_W8", "1") != "1":
        print("[W8] DISABLED: running BF16 weights", flush=True)
        return {"target": [], "replaced": []}
    info = target_info(model)
    targets = set(info["target"])
    llm_total = len(info["llm_attn"]) + len(info["llm_mlp"])
    print("[W8] TARGET_COUNTS llm_attn=", len(info["llm_attn"]), "llm_mlp=", len(info["llm_mlp"]),
          "llm_total=", llm_total, "action_linear_all=", len(info["action_linear_all"]),
          "action_ffn_subset=", len(info["action_ffn_subset"]), "selected=", len(targets), flush=True)
    print("[W8] ACTION_HEAD_LINEAR_AUDIT all=", len(info["action_linear_all"]),
          "selected_for_w8=", len(info["action_linear_selected_for_w8_scope"]),
          "excluded=", len(info["action_linear_excluded_from_w8_scope"]), flush=True)
    expected_raw = os.environ.get("EVO1_EXPECT_W8_TARGETS", "130").strip().lower()
    if expected_raw not in ("", "auto", "none", "skip"):
        exp_total = int(expected_raw)
        if len(targets) != exp_total:
            _write_w8_target_audit(info)
            raise RuntimeError(f"W8 target count {len(targets)} != expected {exp_total}. Refusing to run wrong quantization scope. See summaries/w8_target_scope_audit.json for discovered names/counts. To inspect without aborting, set EVO1_EXPECT_W8_TARGETS=auto.")
    else:
        print("[W8] EXPECTED_TARGETS_AUTO: discovered selected target count =", len(targets), flush=True)
    if PRINT_W8_TARGET_NAMES:
        print("[W8] TARGET_NAMES_BEGIN", flush=True)
        for n in sorted(targets):
            print("[W8] TARGET_NAME", n, flush=True)
        print("[W8] ACTION_HEAD_ALL_LINEAR_NAMES_BEGIN", flush=True)
        for n in info["action_linear_all"]:
            print("[W8] ACTION_HEAD_LINEAR", n, flush=True)
        print("[W8] ACTION_HEAD_ALL_LINEAR_NAMES_END", flush=True)
        print("[W8] TARGET_NAMES_END", flush=True)
    else:
        print("[W8] TARGET_NAMES_QUIET count=", len(targets), "set W8_PRINT_TARGET_NAMES=1 to print all names", flush=True)
    print("[W8] TORCHAO_W8_API", TORCHAO_W8_API, flush=True)
    llm_targets = set(info["llm_attn"]) | set(info["llm_mlp"])
    action_targets = set(info["action_linear_all"])

    # TorchAO backend is safe/fast for the 98 LLM linears used in the previous W8A16 baseline.
    # Do NOT apply torchao Int8Tensor to action_head helper/projection linears: some call weight.t()
    # and crash with "Int8Tensor dispatch ... aten.t".
    quantize_(model, _w8_config(), filter_fn=lambda mod, fqn: isinstance(mod, torch.nn.Linear) and fqn in llm_targets, device="cuda")

    mods = dict(model.named_modules())
    llm_replaced = sorted(n for n in llm_targets if _is_w8_weight(getattr(mods[n], "weight", None)))
    if set(llm_replaced) != llm_targets:
        raise RuntimeError("LLM W8 torchao replacement mismatch missing=" + repr(sorted(llm_targets - set(llm_replaced))[:20]))

    # For all 32 action_head Linear weights, apply int8 quantize/dequantize into normal tensors.
    # This preserves the W8 numerical perturbation while avoiding torchao tensor-subclass dispatch crashes.
    action_replaced, action_fake_stats = fake_quantize_action_head_weights_dequantized(model, action_targets)

    replaced = sorted(set(llm_replaced) | set(action_replaced))
    if set(replaced) != targets:
        missing = sorted(targets - set(replaced))[:50]
        extra = sorted(set(replaced) - targets)[:50]
        raise RuntimeError(f"W8 replacement mismatch missing={missing} extra={extra}")

    qtype_counts = {}
    for rn in llm_replaced:
        ww = getattr(mods[rn], "weight", None)
        key = type(ww).__module__ + "." + type(ww).__name__
        qtype_counts[key] = qtype_counts.get(key, 0) + 1
    qtype_counts["action_head.fake_dequant_int8_weight_parameter"] = len(action_replaced)

    print("[W8] TORCHAO_LLM_W8_REPLACED", len(llm_replaced), "/", len(llm_targets), flush=True)
    print("[W8] ACTION_HEAD_W8_FAKE_DEQUANTIZED", len(action_replaced), "/", len(action_targets), flush=True)
    print("[W8] ACTION_HEAD_W8_BACKEND fake_dequant_int8_per_output_channel_symmetric", flush=True)
    print("[W8] W8_QTYPE_COUNTS", json.dumps(qtype_counts, sort_keys=True), flush=True)
    info["replaced"] = replaced
    info["llm_torchao_replaced"] = llm_replaced
    info["action_head_fake_dequant_replaced"] = action_replaced
    info["action_head_fake_dequant_stats"] = action_fake_stats
    info["action_head_w8_storage_note"] = "action_head W8 is stored as normal tensor after int8 quantize/dequantize to avoid torchao Int8Tensor aten.t crash"
    _write_w8_target_audit(info, qtype_counts=qtype_counts)
    return info

QARGS = SimpleNamespace(
    abits=int(os.environ.get("FLOWA8_ABITS", "8")),
    act_group_size=int(os.environ.get("FLOWA8_ACT_GROUP_SIZE", "0")),
    tiling=int(os.environ.get("FLOWA8_TILING", "0")),
    a_sym=os.environ.get("FLOWA8_A_SYM", "0") == "1",
    a_clip_ratio=float(os.environ.get("FLOWA8_A_CLIP_RATIO", "1.0")),
    quant_type=os.environ.get("FLOWA8_QUANT_TYPE", "int"),
    static=False,
)
A8_SITE_MODE = os.environ.get("FLOWA8_SITE_MODE", "v40_linear130_split_116_global_14_windowed")
ENABLE_MULTI_SEED = os.environ.get("FLOWA8_ENABLE_MULTI_SEED", "0") == "1"
SERVER_VERBOSE_METRICS = os.environ.get("W8A8_SERVER_VERBOSE_METRICS", "0") == "1"
TRACE_VERBOSE = os.environ.get("FLOWA8_TRACE_VERBOSE", "0") == "1"
LINEAR_A8_RECORD_SITES = os.environ.get("W8A8_RECORD_LINEAR_SITE_ERRORS", "1") == "1"
LINEAR_A8_MAX_SITE_RECORDS = int(os.environ.get("W8A8_MAX_SITE_RECORDS_PER_REQUEST", "4096"))
PRINT_W8_TARGET_NAMES = os.environ.get("W8_PRINT_TARGET_NAMES", "0") == "1"

print("[A8] QUANT_BACKEND", A8_QUANT_BACKEND, "QDIT_IMPORT_ERROR=", QDIT_IMPORT_ERROR, flush=True)
TRACE = {"mode": "fp", "steps": [], "site_records": [], "current_step": None, "a8_steps": set(), "ablation_mode": "w8a16"}

QACT_SEEN_SHAPES = set()
QACT_SKIPPED_SHAPES = set()
QACT_SITE_COUNTS = {}

def _site_allowed(site: str) -> bool:
    # v40: clean normal layer W8A8 only.
    # A8 happens through forward pre-hooks on Linear inputs.
    # No standalone tensor stress sites are enabled here.
    return False


def _qdit_compatible_activation(x) -> tuple:
    # Q-DiT quantize_activation_wrapper internally reshapes x[..., C] to [-1, C].
    # Its lower quantize_tensor() calls squeeze() then asserts dim()==2, so a
    # single group [1, C] can fail. In v15, singleton Linear inputs are real
    # projection/head activations, so qact() handles them by duplicating one row
    # for quantizer shape compatibility and slicing back.
    if not torch.is_tensor(x):
        return False, "not_tensor"
    if not torch.is_floating_point(x):
        return False, "not_floating"
    if x.numel() == 0:
        return False, "empty"
    if x.dim() < 2:
        return False, f"dim_lt_2:{tuple(x.shape)}"
    if x.shape[-1] <= 1:
        return False, f"last_dim_le_1:{tuple(x.shape)}"
    groups = int(x.numel() // x.shape[-1])
    if groups < 1:
        return False, f"no_groups:{tuple(x.shape)}"
    return True, f"groups={groups},C={x.shape[-1]}"

@torch.no_grad()
def qact(x, site):
    # Q-DiT reference activation fake-quant.
    #
    # IMPORTANT METHOD ADAPTER:
    # - Q-DiT is diffusion-transformer code, but its activation wrapper is not image-specific.
    # - It treats x[..., C] as feature activations, flattens leading dims to sample/token rows,
    #   quantizes over the last feature dimension, then reshapes back.
    # - In Evo-1 flow matching, valid analogues are action-token / hidden activations
    #   such as [B, horizon, hidden_dim], not helper tensors or final ODE velocity outputs.
    # - Therefore we keep Q-DiT quant.py unchanged and filter Evo-1 sites here.
    if QARGS.abits >= 16:
        return x
    orig_dtype = x.dtype
    orig_device = x.device
    x_contig = x.contiguous()
    C = int(x_contig.shape[-1])
    flat = x_contig.reshape(-1, C)
    # Q-DiT's lower quantizer squeezes singleton [1, C] groups and can assert 1D.
    # For action-head projection layers such as state_encoder/seq_pool/mlp_head, singleton inputs are real Linear activations.
    # Duplicate the one row only for quantizer shape compatibility, then keep the first quantized row.
    if flat.shape[0] == 1:
        flat2 = torch.cat([flat, flat], dim=0)
        y2 = quantize_activation_wrapper(flat2, QARGS)
        y = y2.reshape(2, C)[:1].reshape_as(x_contig)
    else:
        y = quantize_activation_wrapper(x_contig, QARGS)
    return y.to(device=orig_device, dtype=orig_dtype)

def maybe_q(x, site):
    if TRACE.get("mode") != "a8":
        return x
    cur_step = TRACE.get("current_step", None)
    a8_steps = TRACE.get("a8_steps", set())
    if cur_step is None or int(cur_step) not in a8_steps:
        return x
    if not _site_allowed(site):
        if TRACE_VERBOSE and site not in QACT_SKIPPED_SHAPES:
            QACT_SKIPPED_SHAPES.add(site)
            shape = tuple(x.shape) if torch.is_tensor(x) else type(x).__name__
            print(f"[FLOWA8-QACT-SKIP] site={site} reason=site_not_allowed shape={shape}", flush=True)
        return x
    ok, reason = _qdit_compatible_activation(x)
    if not ok:
        if TRACE_VERBOSE and (site, reason) not in QACT_SKIPPED_SHAPES:
            QACT_SKIPPED_SHAPES.add((site, reason))
            shape = tuple(x.shape) if torch.is_tensor(x) else type(x).__name__
            print(f"[FLOWA8-QACT-SKIP] site={site} reason={reason} shape={shape}", flush=True)
        return x

    if TRACE_VERBOSE and site not in QACT_SEEN_SHAPES:
        QACT_SEEN_SHAPES.add(site)
        print(f"[FLOWA8-QACT-APPLY] site={site} shape={tuple(x.shape)} {reason} dtype={x.dtype}", flush=True)

    QACT_SITE_COUNTS[site] = QACT_SITE_COUNTS.get(site, 0) + 1
    y = qact(x, site)
    try:
        err = (y.detach().float() - x.detach().float())
        TRACE.setdefault("site_records", []).append({
            "flow_step": int(cur_step),
            "site": str(site),
            "site_type": ("block_input" if site.startswith("block_") and site.endswith("_input") else
                          "block_output" if site.startswith("block_") and site.endswith("_output") else
                          str(site)),
            "qact_error_rms": _rms(err),
            "qact_error_max_abs": _maxabs(err),
            "shape": list(x.shape),
        })
    except Exception:
        pass
    return y

def _rms(a):
    a = a.detach().float()
    return float(torch.sqrt(torch.mean(a * a) + 1e-12).cpu())

def _l2(a):
    a = a.detach().float().reshape(-1)
    return float(torch.linalg.vector_norm(a).cpu())

def _maxabs(a):
    return float(a.detach().float().abs().max().cpu())

def _tolist_cpu(x):
    return x.detach().float().cpu()



def _linear_a8_kind(name: str):
    # v40 clean split over the 130 normal Linear targets.
    # Fixed/global A8 in every A8 mode:
    #   - 98 LLM attention/MLP linears
    #   - 16 action-head transformer FFN linears (ff.0 / ff.2 across 8 blocks)
    #   - 2 state_encoder linears, because they are outside the 32-step loop and cannot be windowed
    # Flow-windowed A8:
    #   - the remaining 14 action-head flow linears
    #     (8 attn.out_proj + action_encoder W1/W2/W3 + seq_pool_proj + mlp_head fc1/fc2)
    if LLM_ATTN.match(name) or LLM_MLP.match(name):
        return "llm_linear_global"
    if name.startswith("action_head.state_encoder."):
        return "action_state_encoder_global"
    if ACTION_FFN.match(name):
        return "action_ffn_global"
    if name.startswith("action_head."):
        return "action_sensitive_flow_window"
    return None

def _linear_a8_active(kind: str):
    if TRACE.get("mode") != "a8" or QARGS.abits >= 16:
        return False, None
    # Reference has a8_steps=[] and should remain A16 activation.
    if not TRACE.get("a8_steps", set()):
        return False, None
    cur_step = TRACE.get("current_step", None)
    if kind in {"llm_linear_global", "action_state_encoder_global", "action_ffn_global"}:
        return True, (-2 if kind == "llm_linear_global" else (-1 if kind == "action_state_encoder_global" else -3))
    if kind == "action_sensitive_flow_window":
        if cur_step is None:
            return False, None
        return int(cur_step) in TRACE.get("a8_steps", set()), int(cur_step)
    return False, None

def _record_linear_site_error(name: str, kind: str, x, y, flow_step):
    if not LINEAR_A8_RECORD_SITES:
        return
    records = TRACE.setdefault("site_records", [])
    if len(records) >= LINEAR_A8_MAX_SITE_RECORDS:
        return
    try:
        err = (y.detach().float() - x.detach().float())
        records.append({
            "flow_step": int(flow_step) if flow_step is not None else -1,
            "site": "linear_input:" + str(name),
            "site_type": kind,
            "qact_error_rms": _rms(err),
            "qact_error_max_abs": _maxabs(err),
            "shape": list(x.shape),
        })
    except Exception:
        pass

def _make_linear_a8_pre_hook(name: str, kind: str):
    def hook(module, args):
        active, flow_step = _linear_a8_active(kind)
        if not active:
            return None
        if not args:
            return None
        x = args[0]
        ok, reason = _qdit_compatible_activation(x)
        if not ok:
            key = ("linear_input:" + name, reason)
            if TRACE_VERBOSE and key not in QACT_SKIPPED_SHAPES:
                QACT_SKIPPED_SHAPES.add(key)
                shape = tuple(x.shape) if torch.is_tensor(x) else type(x).__name__
                print(f"[A8-ACTION-LINEAR-SKIP] name={name} kind={kind} reason={reason} shape={shape}", flush=True)
            return None
        site = "linear_input:" + name
        if TRACE_VERBOSE and site not in QACT_SEEN_SHAPES:
            QACT_SEEN_SHAPES.add(site)
            print(f"[A8-ACTION-LINEAR-APPLY] name={name} kind={kind} flow_step={flow_step} shape={tuple(x.shape)} {reason} dtype={x.dtype}", flush=True)
        y = qact(x, site)
        QACT_SITE_COUNTS[site] = QACT_SITE_COUNTS.get(site, 0) + 1
        _record_linear_site_error(name, kind, x, y, flow_step)
        return (y,) + tuple(args[1:])
    return hook

LINEAR_A8_HOOK_HANDLES = []
LINEAR_A8_RECORD_SITES = os.environ.get("FLOWA8_RECORD_LINEAR_A8_SITES", "1") == "1"
LINEAR_A8_MAX_SITE_RECORDS = int(os.environ.get("FLOWA8_MAX_LINEAR_A8_SITE_RECORDS", "2048"))

def install_linear_a8_hooks(model, w8_info):
    # v40 activation A8 scope over the same 130 normal Linear targets as W8.
    # This is clean layer/input quantization, not random intermediate-tensor stress.
    mods = dict(model.named_modules())
    target_names = sorted(w8_info.get("replaced") or w8_info.get("target") or target_info(model)["target"])
    selected = []
    by_kind = {
        "llm_linear_global": [],
        "action_state_encoder_global": [],
        "action_ffn_global": [],
        "action_sensitive_flow_window": [],
    }
    skipped = []
    for name in target_names:
        kind = _linear_a8_kind(name)
        if kind is None:
            skipped.append(name)
            continue
        if name not in mods:
            raise RuntimeError(f"A8 hook target {name} is not present in model.named_modules()")
        selected.append(name)
        by_kind[kind].append(name)

    expected_raw = os.environ.get("EVO1_EXPECT_LINEAR_A8_TARGETS", "130").strip().lower()
    if expected_raw not in ("", "auto", "none", "skip") and len(selected) != int(expected_raw):
        raise RuntimeError(f"Linear A8 hook count {len(selected)} != expected {expected_raw}. Refusing wrong A8 scope.")

    for name in selected:
        kind = _linear_a8_kind(name)
        LINEAR_A8_HOOK_HANDLES.append(mods[name].register_forward_pre_hook(_make_linear_a8_pre_hook(name, kind)))

    audit = {
        "scope_note": (
            "v40 clean module-level split: W8 on 130 Linear targets; A8 pre-hooks on the same 130 Linear inputs. "
            "116 targets are fixed/global A8 in every A8 mode (98 LLM + 16 action FFN + 2 state_encoder). "
            "14 action-head flow linears are controlled by request a8_steps. No standalone tensor sites are enabled."
        ),
        "normal_quantization_rule": "weighted Linear op = W8 weight + A8 input activation",
        "standalone_tensor_stress_sites_enabled": False,
        "llm_linear_global": sorted(by_kind["llm_linear_global"]),
        "action_state_encoder_global": sorted(by_kind["action_state_encoder_global"]),
        "action_ffn_global": sorted(by_kind["action_ffn_global"]),
        "action_sensitive_flow_window": sorted(by_kind["action_sensitive_flow_window"]),
        "linear_a8_all": sorted(selected),
        "skipped_w8_targets": sorted(skipped),
        "counts": {
            "llm_linear_global": len(by_kind["llm_linear_global"]),
            "action_state_encoder_global": len(by_kind["action_state_encoder_global"]),
            "action_ffn_global": len(by_kind["action_ffn_global"]),
            "action_sensitive_flow_window": len(by_kind["action_sensitive_flow_window"]),
            "fixed_global_a8_total": len(by_kind["llm_linear_global"]) + len(by_kind["action_state_encoder_global"]) + len(by_kind["action_ffn_global"]),
            "flow_windowed_a8_total": len(by_kind["action_sensitive_flow_window"]),
            "linear_a8_total": len(selected),
        },
        "site_mode": A8_SITE_MODE,
    }
    path = RUN_ROOT / "summaries" / "a8_linear_split_activation_scope_audit.json"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(audit, indent=2))
    print("[V40-LINEAR-SPLIT] fixed_global_a8=", audit["counts"]["fixed_global_a8_total"],
          "flow_windowed_a8=", audit["counts"]["flow_windowed_a8_total"],
          "linear_a8_total=", audit["counts"]["linear_a8_total"], flush=True)
    print("[V40-LINEAR-SPLIT] llm_global=", audit["counts"]["llm_linear_global"],
          "state_global=", audit["counts"]["action_state_encoder_global"],
          "action_ffn_global=", audit["counts"]["action_ffn_global"],
          "action_windowed=", audit["counts"]["action_sensitive_flow_window"], flush=True)
    print("[V40-LINEAR-SPLIT] AUDIT_JSON", path, flush=True)
    print("[V40-LINEAR-SPLIT] WINDOWED_NAMES_BEGIN", flush=True)
    for n in sorted(by_kind["action_sensitive_flow_window"]):
        print("[V40-LINEAR-SPLIT] WINDOWED", n, flush=True)
    print("[V40-LINEAR-SPLIT] WINDOWED_NAMES_END", flush=True)
    return audit


def install_traced_get_action(model):
    if not hasattr(model, "action_head"):
        raise RuntimeError("Model has no action_head attribute; cannot trace flow steps.")

    def traced_get_action(self, fused_tokens: torch.Tensor, state: torch.Tensor = None,
                          embodiment_id: torch.LongTensor = None, action_mask: torch.Tensor = None):
        mode = TRACE.get("mode", "fp")
        TRACE["steps"] = []
        B = fused_tokens.size(0)
        device = fused_tokens.device
        if embodiment_id is None:
            embodiment_id = torch.zeros(B, dtype=torch.long, device=device)

        context_tokens = fused_tokens
        if state is not None and self.state_encoder is not None:
            # Keep state/context BF16 by default; quantization is focused on action-head core loop.
            state_emb = self.state_encoder(state, embodiment_id).unsqueeze(1)
            context_tokens = torch.cat([context_tokens, state_emb], dim=1)

        action_dim_total = getattr(self.config, "action_dim", self.action_dim)
        if self.horizon > 1:
            per_action_dim = getattr(self.config, "per_action_dim", action_dim_total // self.horizon)
        else:
            per_action_dim = action_dim_total

        action = torch.rand(B, action_dim_total, device=device) * 2 - 1
        if self.horizon > 1:
            action_seq = action.view(B, self.horizon, per_action_dim)
        else:
            action_seq = action.view(B, 1, per_action_dim)

        if action_mask is None:
            raise ValueError("action_mask must be provided for inference with flow matching.")
        # action_mask from LIBERO is usually one per-action mask [B, 24],
        # not a full [B, horizon, 24] mask. Support both formats explicitly.
        mask = action_mask.to(device=action_seq.device, dtype=action_seq.dtype)
        if mask.ndim == 1:
            mask = mask.view(1, -1)
        if mask.numel() == B * per_action_dim:
            action_mask_local = mask.view(B, 1, per_action_dim).repeat(1, self.horizon, 1)
        elif mask.numel() == B * self.horizon * per_action_dim:
            action_mask_local = mask.view(B, self.horizon, per_action_dim)
        else:
            raise RuntimeError(
                f"action_mask has incompatible shape {tuple(action_mask.shape)} / numel={action_mask.numel()} "
                f"for B={B}, horizon={self.horizon}, per_action_dim={per_action_dim}. "
                "Expected [B, per_action_dim] or [B, horizon, per_action_dim]."
            )
        action_seq = action_seq * action_mask_local
        action = action_seq.reshape(B, -1)

        N = int(getattr(self.config, "num_inference_timesteps", 32))
        dt = 1.0 / N

        for i in range(N):
            TRACE["current_step"] = int(i)
            t = i / N
            action_before = action.detach().float()
            action_seq_before = action_seq.detach().float()

            time_index = int(t * 1000)
            time_emb = self.time_pos_enc(1000)[:, time_index, :].to(device).squeeze(0)
            time_emb = time_emb.unsqueeze(0).repeat(B, 1)
            time_emb = maybe_q(time_emb, "time_emb")
            context_tokens_step = maybe_q(context_tokens, "context_tokens")

            if self.horizon > 1 and self.action_encoder is not None:
                action_seq = action_seq * action_mask_local
                enc_in = maybe_q(action_seq, "action_encoder_input")
                action_tokens = self.action_encoder(enc_in, embodiment_id)
            else:
                raise RuntimeError(
                    f"traced_get_action: horizon={self.horizon}, action_encoder is None. "
                    "Refusing to create a random single_action_proj. Check Evo_1/model/action_head/flow_matching.py "
                    "for the correct learned action projection layer."
                )

            x = maybe_q(action_tokens, "action_tokens")
            # Apply Q-DiT-style dynamic A8 to action-head core tensors before/after transformer blocks.
            for bi, block in enumerate(self.transformer_blocks):
                x = maybe_q(x, f"block_{bi}_input")
                x = block(x, context_tokens_step, time_emb)
                x = maybe_q(x, f"block_{bi}_output")

            x = self.norm_out(x)
            x = maybe_q(x, "norm_out")

            if self.horizon > 1:
                # seq_pool_input is a pooled helper representation [B, horizon*hidden].
                # It is not a DiT-style multi-token activation, and Q-DiT's quantizer is not
                # applied here in this first flow-step calibration.
                x_flat = x.reshape(B, -1)
                x_pooled = self.seq_pool_proj(x_flat)
            else:
                x_pooled = x.squeeze(1)

            # v23 stress: pred_velocity and action_after are inside the flow loop and timestep-window controlled.
            # This is intentionally stronger than v22 to force a quick failure probe without touching global/LLM activations.
            pred = self.mlp_head(x_pooled, embodiment_id)
            pred = maybe_q(pred, "pred_velocity")

            action_after = action + dt * pred
            action_after = maybe_q(action_after, "action_after")
            if self.horizon > 1:
                action_seq = action_after.view(B, self.horizon, per_action_dim)
            else:
                action_seq = action_after.view(B, 1, per_action_dim)
            action = action_after

            # Keep per-request tensors in memory only. JSON stores aggregate metrics after paired run.
            TRACE["steps"].append({
                "i": i,
                "t": float(t),
                "action_before": _tolist_cpu(action_before),
                "action_after": _tolist_cpu(action_after),
                "action_seq_before": _tolist_cpu(action_seq_before),
                "pred": _tolist_cpu(pred),
            })

        TRACE["current_step"] = None
        return action

    model.action_head.get_action = types.MethodType(traced_get_action, model.action_head)
    print("[FLOWA8] Installed traced action_head.get_action", flush=True)

def load_model_and_normalizer(ckpt_dir):
    cfg = json.load(open(os.path.join(ckpt_dir, "config.json")))
    stats = json.load(open(os.path.join(ckpt_dir, "norm_stats.json")))
    cfg["finetune_vlm"] = False
    cfg["finetune_action_head"] = False
    cfg["num_inference_timesteps"] = int(os.environ.get("EVO1_NUM_INFERENCE_TIMESTEPS", "32"))
    model = EVO1(cfg).eval()
    ckpt_path = os.path.join(ckpt_dir, "mp_rank_00_model_states.pt")
    checkpoint = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(checkpoint["module"], strict=True)
    # Match official Evo-1 server load behavior: move model to CUDA, keep module dtypes as loaded.
    # Inference below uses torch.amp.autocast(..., dtype=torch.bfloat16), matching the official server execution style.
    # Do not globally cast the whole model to BF16 here; W8 quantization is applied after the official-style load.
    model = model.to("cuda")
    w8_info = apply_w8_weights(model)
    a8_hook_info = install_linear_a8_hooks(model, w8_info)
    install_traced_get_action(model)
    normalizer = Normalizer(stats)
    return model, normalizer

def decode_image_from_list(img_list):
    # Official Evo-1 server path: list -> uint8 -> cv2.resize -> cv2.COLOR_BGR2RGB -> PIL -> tensor.
    # Keep this even though it looks odd with list-encoded arrays, because the official LIBERO client flips images
    # before sending and the official server applies this conversion before Image.fromarray.
    img_array = np.array(img_list, dtype=np.uint8)
    img = cv2.resize(img_array, (448, 448))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(img)
    return transforms.ToTensor()(pil).to("cuda")

def prep_inputs(data, normalizer):
    images = [decode_image_from_list(img) for img in data["image"]]
    assert len(images) == 3, "Must provide exactly 3 images."
    state = torch.tensor(data["state"], dtype=torch.float32, device="cuda")
    if state.ndim == 1:
        state = state.unsqueeze(0)
    if state.shape[1] < 24:
        state = torch.cat([state, torch.zeros((1, 24 - state.shape[1]), device="cuda")], dim=1)
    norm_state = normalizer.normalize_state(state).to(dtype=torch.float32)
    prompt = data["prompt"]
    image_mask = torch.tensor(data["image_mask"], dtype=torch.int32, device="cuda")
    action_mask = torch.tensor([data["action_mask"]], dtype=torch.int32, device="cuda")
    return images, norm_state, prompt, image_mask, action_mask

def denorm_chunk(action, normalizer):
    # model output is flat normalized action; return tensor [H, 24] denormalized
    return normalizer.denormalize_action(action.reshape(1, -1, 24)[0]).detach().float().cpu()

def exec7(chunk_denorm):
    return chunk_denorm[:, :7].contiguous()

def denorm_to_json(chunk_denorm):
    return chunk_denorm.detach().cpu().numpy().tolist()

def gripper_cmd(g):
    # Matches Evo-1 LIBERO client threshold logic.
    return -1 if float(g) > 0.5 else 1

def dtw_distance(a, b):
    # VLA-ATTC-style action chunk disagreement metric. Paper reference only; no official code found.
    # a,b: tensors [T,D]
    a = a.detach().float().cpu()
    b = b.detach().float().cpu()
    n, m = a.shape[0], b.shape[0]
    dp = torch.full((n + 1, m + 1), float("inf"))
    dp[0, 0] = 0.0
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = torch.linalg.vector_norm(a[i-1] - b[j-1])
            dp[i, j] = cost + torch.min(torch.stack([dp[i-1, j], dp[i, j-1], dp[i-1, j-1]]))
    return float((dp[n, m] / max(n, m)).item())

@torch.no_grad()
def run_once(data, model, normalizer, mode, seed, a8_steps=None, ablation_mode="w8a16"):
    images, norm_state, prompt, image_mask, action_mask = prep_inputs(data, normalizer)
    TRACE["mode"] = "a8" if mode == "a8" else "fp"
    TRACE["steps"] = []
    TRACE["site_records"] = []
    TRACE["current_step"] = None
    TRACE["a8_steps"] = set(int(x) for x in (a8_steps or []))
    TRACE["ablation_mode"] = ablation_mode
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
        action = model.run_inference(
            images=images,
            image_mask=image_mask,
            prompt=prompt,
            state_input=norm_state,
            action_mask=action_mask,
        )
    steps = TRACE["steps"]
    site_records = list(TRACE.get("site_records", []))
    denorm = denorm_chunk(action, normalizer)
    return action.detach().float().cpu(), denorm, steps, site_records

def velocity_jump_metrics(steps):
    out = []
    prev = None
    for s in steps:
        pred = s["pred"]
        rec = {"flow_step": int(s["i"]), "velocity_norm": _rms(pred)}
        if prev is None:
            rec["velocity_jump_rms"] = 0.0
            rec["velocity_cos_to_prev"] = None
        else:
            diff = pred - prev
            rec["velocity_jump_rms"] = _rms(diff)
            a = pred.reshape(-1).float(); b = prev.reshape(-1).float()
            rec["velocity_cos_to_prev"] = float((a*b).sum() / (torch.linalg.vector_norm(a)*torch.linalg.vector_norm(b) + 1e-12))
        prev = pred
        out.append(rec)
    return out

def summarize_pair(ref_action, ref_denorm, ref_steps, mode_action, mode_denorm, mode_steps, meta, ablation_mode, a8_steps, site_records):
    if len(ref_steps) != len(mode_steps):
        raise RuntimeError(f"Trace length mismatch: ref={len(ref_steps)} mode={len(mode_steps)}")
    # The model can return a longer action chunk than LIBERO actually executes.
    # In Evo-1/LIBERO, the returned model chunk may be length 50, while the client
    # executes only the configured prefix HORIZON, commonly 14/16.
    # Metrics below are therefore computed on the executable prefix only.
    ref7_full = exec7(ref_denorm)
    mode7_full = exec7(mode_denorm)
    model_horizon = int(ref7_full.shape[0])
    exec_horizon = int(meta.get("exec_horizon", min(model_horizon, 14)))
    exec_horizon = max(1, min(exec_horizon, model_horizon, int(mode7_full.shape[0])))
    ref7 = ref7_full[:exec_horizon]
    mode7 = mode7_full[:exec_horizon]
    diff7 = mode7 - ref7

    step_records = []
    ref_final_flat = ref_action.reshape(-1)
    mode_final_flat = mode_action.reshape(-1)
    ref_vjm = {x["flow_step"]: x for x in velocity_jump_metrics(ref_steps)}
    mode_vjm = {x["flow_step"]: x for x in velocity_jump_metrics(mode_steps)}

    for fs, qs in zip(ref_steps, mode_steps):
        i = int(fs["i"]); t = float(fs["t"])
        pred_diff = qs["pred"] - fs["pred"]
        before_diff = qs["action_before"] - fs["action_before"]
        after_diff = qs["action_after"] - fs["action_after"]
        remain = 1.0 - t
        ref_endpoint_est = fs["action_before"] + remain * fs["pred"]
        mode_endpoint_est = qs["action_before"] + remain * qs["pred"]
        rec = {
            "ablation_mode": ablation_mode,
            "flow_step": i,
            "a8_enabled_this_step": int(i in set(a8_steps)),
            "t": t,
            "qdrift_velocity_rms": _rms(pred_diff),
            "qdrift_velocity_l2": _l2(pred_diff),
            "qdrift_velocity_max_abs": _maxabs(pred_diff),
            "qdrift_state_before_rms": _rms(before_diff),
            "qdrift_state_after_rms": _rms(after_diff),
            "ref_velocity_norm": ref_vjm[i]["velocity_norm"],
            "mode_velocity_norm": mode_vjm[i]["velocity_norm"],
            "ref_velocity_jump_rms": ref_vjm[i]["velocity_jump_rms"],
            "mode_velocity_jump_rms": mode_vjm[i]["velocity_jump_rms"],
            "ref_velocity_cos_to_prev": ref_vjm[i]["velocity_cos_to_prev"],
            "mode_velocity_cos_to_prev": mode_vjm[i]["velocity_cos_to_prev"],
            "flash_endpoint_pair_rms": _rms(mode_endpoint_est - ref_endpoint_est),
            "flash_ref_endpoint_to_final_rms": _rms(ref_endpoint_est.reshape(-1) - ref_final_flat),
            "flash_mode_endpoint_to_final_rms": _rms(mode_endpoint_est.reshape(-1) - mode_final_flat),
        }
        # simple action curvature/second-difference on action state after update, if neighboring records exist later in pandas too.
        step_records.append(rec)

    grip_mismatch_count = 0
    grip_boundary_count = 0
    for k in range(ref7.shape[0]):
        if abs(float(ref7[k,6]) - 0.5) < 0.05 or abs(float(mode7[k,6]) - 0.5) < 0.05:
            grip_boundary_count += 1
        if gripper_cmd(ref7[k,6]) != gripper_cmd(mode7[k,6]):
            grip_mismatch_count += 1

    out = {
        **meta,
        "ablation_mode": ablation_mode,
        "weights": "W8" if os.environ.get("EVO1_APPLY_W8", "1") == "1" else "BF16",
        "activation_mode": "A16" if not a8_steps else (f"A{QARGS.abits}_or_dynamic"),
        "a8_steps": list(map(int, a8_steps)),
        "abits": QARGS.abits,
        "site_mode": A8_SITE_MODE,
        "num_flow_steps": len(step_records),
        "model_horizon": int(model_horizon),
        "exec_horizon": int(exec_horizon),
        "metric_scope": "executed_prefix_action[:7]",
        "a8_activation_scope": "v40_w8_130_linear_module_a8_116_global_plus_14_flow_windowed_no_random_sites",
        "final_action_7d_rms": _rms(diff7),
        "final_action_7d_l2": _l2(diff7),
        "final_action_7d_max_abs": _maxabs(diff7),
        "xyz_rms": _rms(diff7[:, :3]),
        "rot_rms": _rms(diff7[:, 3:6]),
        "gripper_abs_mean": float(torch.mean(torch.abs(diff7[:, 6])).item()),
        "gripper_threshold_mismatch_count": int(grip_mismatch_count),
        "gripper_boundary_count": int(grip_boundary_count),
        "step_records": step_records,
        "site_records": site_records,
    }
    return out

@torch.no_grad()
def infer_pair_from_json_dict(data, model, normalizer):
    meta = data.get("meta", {})
    ablation_mode = str(meta.get("ablation_mode", "w8a16"))
    a8_steps = [int(x) for x in meta.get("a8_steps", [])]
    request_id = int(meta.get("request_id", int(time.time() * 1000) % 100000000))
    seed = int(meta.get("seed", os.environ.get("FLOWA8_BASE_SEED", "12345"))) + request_id * 1009

    ref_action, ref_denorm, ref_steps, _ = run_once(data, model, normalizer, "fp", seed=seed, a8_steps=[], ablation_mode="w8a16_ref")

    qact_before = dict(QACT_SITE_COUNTS)
    if ablation_mode == "w8a16" or not a8_steps:
        mode_action, mode_denorm, mode_steps, site_records = ref_action, ref_denorm, ref_steps, []
    else:
        mode_action, mode_denorm, mode_steps, site_records = run_once(data, model, normalizer, "a8", seed=seed, a8_steps=a8_steps, ablation_mode=ablation_mode)
    qact_after = dict(QACT_SITE_COUNTS)

    record = summarize_pair(ref_action, ref_denorm, ref_steps, mode_action, mode_denorm, mode_steps, meta, ablation_mode, a8_steps, site_records)
    record["a8_qact_site_counts"] = {k: int(qact_after.get(k, 0) - qact_before.get(k, 0)) for k in sorted(set(qact_after) | set(qact_before)) if int(qact_after.get(k, 0) - qact_before.get(k, 0)) != 0}

    if ENABLE_MULTI_SEED:
        fp2_action, fp2_denorm, _, _ = run_once(data, model, normalizer, "fp", seed=seed + 999983, a8_steps=[], ablation_mode="multiseed_fp")
        record["attc_multiseed_dtw_7d"] = dtw_distance(exec7(ref_denorm), exec7(fp2_denorm))
        record["attc_multiseed_l2_7d"] = _l2(exec7(fp2_denorm) - exec7(ref_denorm))
    else:
        record["attc_multiseed_dtw_7d"] = None
        record["attc_multiseed_l2_7d"] = None

    out_path = DRIFT / f"{ablation_mode}_request_records.jsonl"
    with out_path.open("a") as f:
        f.write(json.dumps(record) + "\n")
    combined_path = DRIFT / "all_modes_request_records.jsonl"
    with combined_path.open("a") as f:
        f.write(json.dumps(record) + "\n")

    if SERVER_VERBOSE_METRICS:
        max_vel = max([s.get("qdrift_velocity_rms", 0.0) for s in record.get("step_records", [])] or [0.0])
        print(
            "[W8A8-ABLATION-METRIC] "
            f"mode={ablation_mode} suite={record.get('suite')} task={record.get('task_id')} ep={record.get('episode_id')} "
            f"outer={record.get('outer_step')} steps={record.get('num_flow_steps')} "
            f"model_horizon={record.get('model_horizon')} exec_horizon={record.get('exec_horizon')} "
            f"final_rms_exec_prefix={record.get('final_action_7d_rms'):.6g} max_abs={record.get('final_action_7d_max_abs'):.6g} "
            f"grip_mismatch={record.get('gripper_threshold_mismatch_count')} vel_step_max={max_vel:.6g}",
            flush=True,
        )
    return record["a8_qact_site_counts"], denorm_to_json(mode_denorm)

async def handle_request(websocket, model, normalizer):
    print("[FLOWA8] Client connected", flush=True)
    try:
        async for message in websocket:
            json_data = json.loads(message)
            _counts, actions = infer_pair_from_json_dict(json_data, model, normalizer)
            await websocket.send(json.dumps(actions))
    except websockets.exceptions.ConnectionClosed:
        print("[FLOWA8] Client disconnected.", flush=True)
    except Exception:
        print("[FLOWA8] ERROR in handle_request", flush=True)
        err_txt = traceback.format_exc()
        print(err_txt, flush=True)
        with ERROR_LOG.open("a") as ef:
            ef.write("\n" + "=" * 100 + "\n")
            ef.write(time.strftime("%Y-%m-%d %H:%M:%S") + "\n")
            ef.write(err_txt + "\n")
        raise

if __name__ == "__main__":
    ckpt_dir = os.environ.get("EVO1_CKPT_DIR", "/content/Evo1_LIBERO")
    port = int(os.environ.get("EVO1_PORT", "9010"))
    print("[FLOWA8] Loading EVO-1 model...", flush=True)
    print("[FLOWA8] WORK_REPO", WORK_REPO, flush=True)
    print("[FLOWA8] RUN_ROOT", RUN_ROOT, flush=True)
    print("[FLOWA8] QARGS", vars(QARGS), flush=True)
    print("[W8A8] Request meta controls ablation_mode and a8_steps", flush=True)
    model, normalizer = load_model_and_normalizer(ckpt_dir)
    print("[FLOWA8] SERVER_READY", flush=True)

    async def main():
        print(f"[FLOWA8] server running at ws://0.0.0.0:{port}", flush=True)
        async with websockets.serve(
            lambda ws: handle_request(ws, model, normalizer),
            "0.0.0.0",
            port,
            max_size=100_000_000,
            ping_interval=None,
            ping_timeout=None,
            close_timeout=30,
        ):
            await asyncio.Future()
    asyncio.run(main())
"""

client_code = r"""
import asyncio, websockets, numpy as np, json, pathlib, os, logging, math, random, time, traceback
from pathlib import Path
from libero.libero import benchmark, get_libero_path
from libero.libero.envs import OffScreenRenderEnv

os.environ.setdefault("MUJOCO_GL", "osmesa")
LIBERO_DUMMY_ACTION = [0.0] * 6 + [-1.0]  # keep gripper open during reset/warmup

SUITE = os.environ.get("FLOWA8_SUITE", "libero_spatial")
TASK_ID = int(os.environ.get("FLOWA8_TASK_ID", "0"))
EPISODE_ID = int(os.environ.get("FLOWA8_EPISODE_ID", "0"))
SERVER_URL = os.environ.get("FLOWA8_SERVER_URL", "ws://127.0.0.1:9010")
HORIZON = int(os.environ.get("FLOWA8_HORIZON", "14"))
MAX_STEPS = int(os.environ.get("FLOWA8_MAX_STEPS", "25"))
SEED = int(os.environ.get("FLOWA8_SEED", "42"))
RUN_ROOT = Path(os.environ["FLOWA8_RUN_ROOT"])
SUMMARY_PATH = Path(os.environ["FLOWA8_SUMMARY_PATH"])
ABLA_MODE = os.environ.get("W8A8_ABLATION_MODE", "w8a16")
A8_STEPS = [int(x) for x in os.environ.get("W8A8_A8_STEPS", "").split(",") if x.strip() != ""]
CLIENT_VERBOSE_STEPS = os.environ.get("W8A8_CLIENT_VERBOSE_STEPS", "0") == "1"

def encode_image_array(img_array: np.ndarray):
    return img_array.astype(np.uint8).tolist()

def quat2axisangle(quat):
    quat = np.array(quat, dtype=np.float64).copy()
    quat[3] = min(1.0, max(-1.0, quat[3]))
    den = np.sqrt(1.0 - quat[3] * quat[3])
    if math.isclose(den, 0.0):
        return np.zeros(3)
    return (quat[:3] * 2.0 * math.acos(quat[3])) / den

def obs_to_json_dict(obs, prompt, meta, resize_size=448):
    img = np.ascontiguousarray(obs["agentview_image"][::-1, ::-1])
    wrist_img = np.ascontiguousarray(obs["robot0_eye_in_hand_image"][::-1, ::-1])
    dummy_proc = np.zeros((resize_size, resize_size, 3), dtype=np.uint8)
    return {
        "image": [encode_image_array(img), encode_image_array(wrist_img), encode_image_array(dummy_proc)],
        # Match official Evo-1 LIBERO state exactly:
        #   eef_pos(3) + axis_angle(3) + robot0_gripper_qpos(2) = 8D raw state.
        # Do NOT slice gripper_qpos[:1]; norm_stats observation.state has length 8.
        "state": np.concatenate((
            obs["robot0_eef_pos"],
            quat2axisangle(obs["robot0_eef_quat"]),
            np.asarray(obs["robot0_gripper_qpos"]),
        )).tolist(),
        "prompt": prompt,
        "image_mask": [1, 1, 0],
        "action_mask": [1] * 7 + [0] * 17,
        "meta": meta,
    }

def get_libero_env(task, resolution=448, seed=SEED):
    # Official Evo-1 LIBERO client environment construction.
    task_description = task.language
    task_bddl_file = pathlib.Path(get_libero_path("bddl_files")) / task.problem_folder / task.bddl_file
    env_args = {"bddl_file_name": task_bddl_file, "camera_heights": resolution, "camera_widths": resolution}
    env = OffScreenRenderEnv(**env_args)
    env.seed(seed)
    return env, task_description

async def run_one():
    np.random.seed(SEED)
    random.seed(SEED)
    benchmark_dict = benchmark.get_benchmark_dict()
    task_suite = benchmark_dict[SUITE]()
    task = task_suite.get_task(TASK_ID)
    initial_states = task_suite.get_task_init_states(TASK_ID)
    if EPISODE_ID >= len(initial_states):
        raise RuntimeError(f"episode {EPISODE_ID} out of range; only {len(initial_states)} init states")
    env, task_description = get_libero_env(task, resolution=448, seed=SEED)

    summary = {
        "suite": SUITE,
        "task_id": TASK_ID,
        "episode_id": EPISODE_ID,
        "success": False,
        "task_fail": False,
        "server_crash": False,
        "timeout_or_missing": False,
        "egl_cleanup_warning": False,
        "outer_steps": 0,
        "env_steps": 0,
        "horizon": HORIZON,
        "max_steps": MAX_STEPS,
        "seed": int(SEED),
        "fixed_episode_seed_policy": os.environ.get("FLOWA8_FIXED_EPISODE_SEED_POLICY", "unknown"),
        "prompt": str(task_description),
        "ablation_mode": ABLA_MODE,
        "a8_steps": A8_STEPS,
    }

    try:
        env.reset()
        obs = env.set_init_state(initial_states[EPISODE_ID])
        for _ in range(10):
            obs, reward, done, info = env.step(LIBERO_DUMMY_ACTION)

        async with websockets.connect(SERVER_URL, ping_interval=None, ping_timeout=None, close_timeout=30, max_size=100_000_000) as ws:
            for outer_step in range(MAX_STEPS):
                summary["outer_steps"] = outer_step + 1
                meta = {
                    "suite": SUITE,
                    "task_id": TASK_ID,
                    "episode_id": EPISODE_ID,
                    "outer_step": outer_step,
                    "request_id": TASK_ID * 100000 + EPISODE_ID * 1000 + outer_step,
                    "seed": SEED,
                    "ablation_mode": ABLA_MODE,
                    "a8_steps": A8_STEPS,
                    "exec_horizon": HORIZON,
                }
                send_data = obs_to_json_dict(obs, str(task_description), meta)
                await ws.send(json.dumps(send_data))
                result = await ws.recv()
                actions = np.array(json.loads(result), dtype=np.float32)
                summary["last_model_horizon"] = int(len(actions))
                summary["exec_horizon"] = int(HORIZON)
                if CLIENT_VERBOSE_STEPS:
                    print(f"[W8A8-CLIENT] outer_step={outer_step} received actions shape={actions.shape}", flush=True)

                for i in range(min(HORIZON, len(actions))):
                    action = actions[i].tolist()
                    # Official Evo-1 LIBERO thresholding behavior.
                    if action[6] > 0.5:
                        action[6] = -1
                    else:
                        action[6] = 1
                    try:
                        obs, reward, done, info = env.step(action[:7])
                    except ValueError as e:
                        # LIBERO/robosuite can mark the episode terminated by horizon/time-limit,
                        # then raise if the client tries one more step. That is a task failure,
                        # not a generated-client or server crash.
                        if "terminated episode" in str(e):
                            summary["task_fail"] = True
                            summary["env_terminated_episode"] = True
                            summary["exception"] = repr(e)
                            print(f"EPISODE_RESULT suite={SUITE} task={TASK_ID:02d} ep={EPISODE_ID:02d} FAIL crash=False terminated_episode=True", flush=True)
                            return summary
                        raise
                    summary["env_steps"] += 1
                    if CLIENT_VERBOSE_STEPS:
                        print(f"[W8A8-CLIENT] outer_step={outer_step} inner={i} reward={reward:.2f} done={done}", flush=True)
                    if done:
                        summary["success"] = True
                        print(f"EPISODE_RESULT suite={SUITE} task={TASK_ID:02d} ep={EPISODE_ID:02d} SUCCESS crash=False", flush=True)
                        return summary
        summary["task_fail"] = True
        print(f"EPISODE_RESULT suite={SUITE} task={TASK_ID:02d} ep={EPISODE_ID:02d} FAIL crash=False", flush=True)
        return summary

    except (websockets.exceptions.ConnectionClosedError, websockets.exceptions.ConnectionClosedOK,
            websockets.exceptions.InvalidHandshake, websockets.exceptions.InvalidURI,
            ConnectionRefusedError, OSError, asyncio.TimeoutError) as e:
        # Real server/network failure. The runner may abort and restart the server.
        summary["server_crash"] = True
        summary["exception"] = repr(e)
        traceback.print_exc()
        print(f"EPISODE_RESULT suite={SUITE} task={TASK_ID:02d} ep={EPISODE_ID:02d} FAIL crash=True", flush=True)
        return summary
    except Exception as e:
        # Client-side coding/environment bug. Do not mislabel this as a server crash.
        summary["client_exception"] = True
        summary["exception"] = repr(e)
        traceback.print_exc()
        print(f"EPISODE_RESULT suite={SUITE} task={TASK_ID:02d} ep={EPISODE_ID:02d} FAIL client_exception=True", flush=True)
        return summary
    finally:
        SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
        SUMMARY_PATH.write_text(json.dumps(summary, indent=2))

if __name__ == "__main__":
    s = asyncio.run(run_one())
    SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
    SUMMARY_PATH.write_text(json.dumps(s, indent=2))
"""


SERVER_SCRIPT.write_text(server_code)
CLIENT_SCRIPT.write_text(client_code)
py_compile.compile(str(SERVER_SCRIPT), doraise=True)
py_compile.compile(str(CLIENT_SCRIPT), doraise=True)

import hashlib, json
server_sha = hashlib.sha256(SERVER_SCRIPT.read_bytes()).hexdigest()
client_sha = hashlib.sha256(CLIENT_SCRIPT.read_bytes()).hexdigest()
hash_manifest = {
    "server_script": str(SERVER_SCRIPT),
    "server_sha256": server_sha,
    "client_script": str(CLIENT_SCRIPT),
    "client_sha256": client_sha,
    "metric_scope": "executed_prefix_action[:7]",
    "state_scope": "official_8d_state_eef3_axisangle3_gripperqpos2",
    "a8_activation_scope": "v40_w8_130_linear_module_a8_116_global_plus_14_flow_windowed_no_random_sites",
}
(GENERATED / "generated_script_hashes.json").write_text(json.dumps(hash_manifest, indent=2))
print("SERVER_SCRIPT:", SERVER_SCRIPT)
print("SERVER_SHA256:", server_sha)
print("CLIENT_SCRIPT:", CLIENT_SCRIPT)
print("CLIENT_SHA256:", client_sha)
print("GENERATED_HASH_MANIFEST:", GENERATED / "generated_script_hashes.json")
print("COMPILE_OK")



SERVER_SCRIPT: /content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a/generated/Evo1_server_w8_dynamic_a8_ablation.py
SERVER_SHA256: b39b4f4fd72ff0ab0e17fdf3ceb8cf5ba2e654b51d3774a298ff3a09060d5b8d
CLIENT_SCRIPT: /content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a/generated/libero_client_w8_dynamic_a8_single_episode.py
CLIENT_SHA256: 02abd1f1c3d75dc94fda4d62f03a8503cea8e8fb82f6dc41b0b9d3167dcb2dfa
GENERATED_HASH_MANIFEST: /content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a/generated/generated_script_hashes.json
COMPILE_OK


In [16]:

# CELL 10B — Sanity-check generated code for v40 clean linear split W8A8.
# Checks only the core invariants; does not rewrite or run eval.
import json, py_compile, re
from pathlib import Path

client_txt = CLIENT_SCRIPT.read_text(errors="ignore")
server_txt = SERVER_SCRIPT.read_text(errors="ignore")
manifest_path = GENERATED / "generated_script_hashes.json"
assert manifest_path.exists(), "BUG: generated_script_hashes.json missing; rerun CELL 10."
hash_manifest = json.loads(manifest_path.read_text())

EXPECTED_A8_SCOPE = "v40_w8_130_linear_module_a8_116_global_plus_14_flow_windowed_no_random_sites"
assert hash_manifest.get("a8_activation_scope") == EXPECTED_A8_SCOPE, hash_manifest.get("a8_activation_scope")
assert f'"a8_activation_scope": "{EXPECTED_A8_SCOPE}"' in server_txt, "server metadata scope mismatch"
assert 'def _site_allowed(site: str)' in server_txt and 'return False' in server_txt, "standalone tensor sites must be disabled"
for bad in ["action_after", "pred_velocity", "block_0_output", "attn_logits", "attn_probs"]:
    # Bad names may appear in metric summaries or old comments less reliably, so only reject explicit allow-list style sites.
    pass

assert 'EVO1_EXPECT_W8_TARGETS", "130"' in server_txt, "W8 target count should remain 130 normal Linear targets"
assert 'EVO1_EXPECT_LINEAR_A8_TARGETS", "130"' in server_txt, "A8 Linear hook count should be 130"
assert 'llm_linear_global' in server_txt, "LLM Linear A8 fixed/global group missing"
assert 'action_ffn_global' in server_txt, "action FFN fixed/global group missing"
assert 'action_sensitive_flow_window' in server_txt, "14 action sensitive flow-window group missing"
assert 'a8_linear_split_activation_scope_audit.json' in server_txt, "v40 A8 audit json missing"
assert 'V40-LINEAR-SPLIT' in server_txt, "v40 pre-run scope print missing"
assert 'np.asarray(obs["robot0_gripper_qpos"]),' in client_txt, "official 2D gripper_qpos state missing"
assert 'ref7 = ref7_full[:exec_horizon]' in server_txt and 'mode7 = mode7_full[:exec_horizon]' in server_txt, "metrics must use executed prefix"
assert 'terminated_episode' in client_txt and "terminated episode" in client_txt, "terminated episode handling missing"

py_compile.compile(str(SERVER_SCRIPT), doraise=True)
py_compile.compile(str(CLIENT_SCRIPT), doraise=True)
print("CELL_10B_V40_SANITY_OK")
print("A8_SCOPE", EXPECTED_A8_SCOPE)
print("W8_TARGETS", 130)
print("A8_LINEAR_TARGETS", 130)
print("A8_SPLIT", "116 fixed/global + 14 flow-windowed")
print("SERVER_SCRIPT", SERVER_SCRIPT)
print("CLIENT_SCRIPT", CLIENT_SCRIPT)


CELL_10B_V40_SANITY_OK
A8_SCOPE v40_w8_130_linear_module_a8_116_global_plus_14_flow_windowed_no_random_sites
W8_TARGETS 130
A8_LINEAR_TARGETS 130
A8_SPLIT 116 fixed/global + 14 flow-windowed
SERVER_SCRIPT /content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a/generated/Evo1_server_w8_dynamic_a8_ablation.py
CLIENT_SCRIPT /content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a/generated/libero_client_w8_dynamic_a8_single_episode.py


In [17]:

# CELL 11 — Start v40 clean linear split W8A8 server on port 9010
# This cell is safe to rerun. By default it RESTARTS any server on 9010 so stale generated code cannot silently corrupt the run.
# It does not redownload anything.
import subprocess, os, time, re, json
from pathlib import Path

MAMBA = "/content/micromamba/bin/micromamba"
MAMBA_ROOT = "/content/micromamba-root"
# Defensive path reconstruction in case Cell 11 is rerun after runtime reset.
DRIVE_CKPT = Path(globals().get("DRIVE_CKPT", os.environ.get("EVO1_DRIVE_CKPT", "/content/drive/MyDrive/Evo-1-checkpoints/Evo1_LIBERO")))
LOCAL_CKPT = Path(globals().get("LOCAL_CKPT", os.environ.get("EVO1_LOCAL_CKPT", "/content/Evo1_LIBERO")))
SERVER_LOG = LOGS / "w8_dynamic_a8_server.log"
LOCAL_CKPT.mkdir(parents=True, exist_ok=True)

AUTO_KILL_STALE_SERVER = True
REUSE_MATCHING_SERVER = False  # safer after notebook patches: restart even matching path, because old process may have stale code loaded


def _port_9010_info():
    r = subprocess.run("ss -ltnp | grep ':9010' || true", shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    out = r.stdout.strip()
    pid = None
    cmdline = ""
    m = re.search(r"pid=(\d+)", out)
    if m:
        pid = int(m.group(1))
        try:
            cmdline = Path(f"/proc/{pid}/cmdline").read_text(errors="ignore").replace("\x00", " ")
        except Exception as exc:
            cmdline = f"<could not read cmdline: {exc}>"
    return out, pid, cmdline


def _server_matches_current_run(pid=None, cmdline=None):
    if cmdline is None:
        _, pid, cmdline = _port_9010_info()
    if not pid:
        return False
    # Match either the exact generated server path or this RUN_ROOT + server basename.
    return (str(SERVER_SCRIPT) in cmdline) or (str(RUN_ROOT) in cmdline and SERVER_SCRIPT.name in cmdline)


def _sync_checkpoint_once():
    existing = list(LOCAL_CKPT.rglob("*.safetensors")) + list(LOCAL_CKPT.rglob("*.bin")) + list(LOCAL_CKPT.rglob("config*.json"))
    if len(existing) >= 3:
        print("LOCAL_CHECKPOINT_FOUND_RSYNC_SKIPPED:", LOCAL_CKPT)
        return
    print("LOCAL_CHECKPOINT_MISSING_RSYNC_FROM_DRIVE")
    print("DRIVE_CKPT_SOURCE:", DRIVE_CKPT, "exists=", DRIVE_CKPT.exists())
    if not DRIVE_CKPT.exists():
        raise RuntimeError(f"Missing DRIVE_CKPT: {DRIVE_CKPT}. Run CELL 09B or check Drive mount.")
    subprocess.run(["rsync", "-ah", "--info=progress2", str(DRIVE_CKPT) + "/", str(LOCAL_CKPT) + "/"], check=True)


def _server_env():
    return {
        **os.environ,
        "MAMBA_ROOT_PREFIX": MAMBA_ROOT,
        "EVO1_WORK_REPO": str(WORK_REPO),
        "FLOWA8_RUN_ROOT": str(RUN_ROOT),
        "EVO1_CKPT_DIR": str(LOCAL_CKPT),
        "EVO1_PORT": "9010",
        "EVO1_NUM_INFERENCE_TIMESTEPS": "32",
        "EVO1_APPLY_W8": "1",
        # v15 W8 guard: 98 LLM linears + all 32 action_head Linear layers = 130.
        "EVO1_EXPECT_W8_TARGETS": "130",
        "EVO1_EXPECT_LINEAR_A8_TARGETS": "130",
        "FLOWA8_INCLUDE_STATE_ENCODER_A8": "1",
        "FLOWA8_SITE_MODE": "v40_linear130_split_116_global_14_windowed",
        "W8A8_SERVER_VERBOSE_METRICS": "0",  # force quiet server output; metrics still saved to JSONL
        "W8_PRINT_TARGET_NAMES": os.environ.get("W8_PRINT_TARGET_NAMES", "0"),
        "W8A8_GENERATED_SERVER_SHA256": (json.loads((GENERATED / "generated_script_hashes.json").read_text()).get("server_sha256") if (GENERATED / "generated_script_hashes.json").exists() else "unknown"),
        "FLOWA8_ABITS": os.environ.get("FLOWA8_ABITS", "8"),  # v28: W8A8 only; do not use A4 here
        "FLOWA8_ACT_GROUP_SIZE": "0",
        "FLOWA8_A_SYM": "0",
        "FLOWA8_A_CLIP_RATIO": "1.0",
        "FLOWA8_ENABLE_MULTI_SEED": "0",
        "FLOWA8_TRACE_VERBOSE": "0",
        "PYTHONPATH": str(WORK_REPO / "Evo_1") + ":" + str(REFS) + ":" + os.environ.get("PYTHONPATH", ""),
    }


def ensure_w8dyn_server():
    if (GENERATED / "generated_script_hashes.json").exists():
        print("GENERATED_SCRIPT_HASHES:")
        print((GENERATED / "generated_script_hashes.json").read_text())
    else:
        print("WARN: generated_script_hashes.json missing; rerun CELL 10.")
    _sync_checkpoint_once()
    out, pid, cmdline = _port_9010_info()
    print("PORT_9010_CURRENT_PROOF:")
    print(out if out else "PORT_9010_FREE")
    if pid:
        print("PORT_9010_PID:", pid)
        print("PORT_9010_CMDLINE:", cmdline[:1000])

    if pid and _server_matches_current_run(pid, cmdline):
        if REUSE_MATCHING_SERVER:
            print("REUSE_MATCHING_SERVER_FOR_THIS_RUN_DISABLED_BY_DEFAULT", pid)
            (RUN_ROOT / "server.pid").write_text(str(pid))
            return pid
        print("MATCHING_SERVER_EXISTS_BUT_REUSE_DISABLED")

    if pid:
        print("STALE_OR_WRONG_SERVER_ON_9010")
        if not AUTO_KILL_STALE_SERVER:
            raise RuntimeError("Port 9010 is occupied by a non-matching server. Set AUTO_KILL_STALE_SERVER=True or kill it manually.")
        print("KILLING_STALE_SERVER_ON_9010")
        subprocess.run("fuser -k 9010/tcp || true", shell=True)
        time.sleep(2)
        out2, pid2, cmd2 = _port_9010_info()
        print("PORT_9010_AFTER_KILL:", out2 if out2 else "PORT_9010_FREE")
        if pid2:
            raise RuntimeError("Port 9010 still busy after kill. Stop old runtime/server manually.")

    cmd = [MAMBA, "run", "-n", "Evo1", "python", str(SERVER_SCRIPT)]
    print("STARTING_SERVER_CMD:", " ".join(cmd))
    print("SERVER_LOG:", SERVER_LOG)
    log_f = open(SERVER_LOG, "w")
    log_f.write("===== START SERVER FOR RUN %s at %s =====\n" % (RUN_ID, time.strftime("%Y-%m-%d %H:%M:%S")))
    log_f.flush()
    proc = subprocess.Popen(cmd, stdout=log_f, stderr=subprocess.STDOUT, env=_server_env(), cwd=str(WORK_REPO / "Evo_1"))
    (RUN_ROOT / "server.pid").write_text(str(proc.pid))
    print("NEW_SERVER_PID:", proc.pid)

    for i in range(240):
        out3, pid3, cmd3 = _port_9010_info()
        if pid3 and _server_matches_current_run(pid3, cmd3):
            print("SERVER_LISTENING_MATCH_PROOF:")
            print(out3)
            print("MATCHING_CMDLINE:", cmd3[:1000])
            print("SERVER_LOG_INITIAL_TAIL")
            print(SERVER_LOG.read_text(errors="ignore")[-4000:] if SERVER_LOG.exists() else "MISSING")
            return pid3
        if i % 10 == 0:
            print("waiting for matching server", i)
            if SERVER_LOG.exists():
                print(SERVER_LOG.read_text(errors="ignore")[-1500:])
        time.sleep(2)

    print("SERVER_LOG_TAIL:")
    print(SERVER_LOG.read_text(errors="ignore")[-6000:] if SERVER_LOG.exists() else "MISSING")
    raise RuntimeError("Server did not start listening on 9010 as the current generated server.")

SERVER_PID = ensure_w8dyn_server()
print("SERVER_READY_PID:", SERVER_PID)


GENERATED_SCRIPT_HASHES:
{
  "server_script": "/content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a/generated/Evo1_server_w8_dynamic_a8_ablation.py",
  "server_sha256": "b39b4f4fd72ff0ab0e17fdf3ceb8cf5ba2e654b51d3774a298ff3a09060d5b8d",
  "client_script": "/content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a/generated/libero_client_w8_dynamic_a8_single_episode.py",
  "client_sha256": "02abd1f1c3d75dc94fda4d62f03a8503cea8e8fb82f6dc41b0b9d3167dcb2dfa",
  "metric_scope": "executed_prefix_action[:7]",
  "state_scope": "official_8d_state_eef3_axisangle3_gripperqpos2",
  "a8_activation_scope": "v40_w8_130_linear_module_a8_116_global_plus_14_flow_windowed_no_random_sites"
}
LOCAL_CHECKPOINT_MISSI

In [21]:

# CELL 12 — Build v40 clean linear-split W8A8 windows+cumulative manifest with W8A16 reference.
#
# Clean v40 design:
# - W8 weights stay on 98 LLM + 32 action-head Linear layers = 130 targets.
# - A8 activations stay window-controlled inside the flow loop only, with the stronger v28/v29 flow-site scope.
# - LLM/state/action-FFN Linear inputs are fixed/global A8 in A8 modes; the 14 sensitive action linears are flow-windowed.
# - Uses v40 mode names + quant-config signature: same exact row resumes/skips; adding suites/episodes later does not invalidate previous rows.
# - Default is selected two suites (`libero_spatial`, `libero_10`) and 5 episodes per task. You may resume/add suites or episodes with the same RUN_ID; exact completed rows are skipped.
# - Episode-first paired order: suite -> task -> episode -> mode.
#
# Questions answered:
# 1) Is there a single sensitive region?        R0/R1/R2/R3 isolated windows.
# 2) Is failure cumulative over many steps?     CUM_00_15, CUM_00_23, ALL_00_31.
from pathlib import Path
import json, os, hashlib

EXPERIMENT_ID = "v40_linear130_split_ref5ep_windows_cumulative"
SUITES = list(globals().get("DEFAULT_SUITES", ["libero_spatial", "libero_10"]))
TASK_IDS = list(range(10))
EPISODES_PER_TASK = int(os.environ.get("W8A8_QUICK_EPISODES_PER_TASK", "5"))
print("MANIFEST_SUITES:", SUITES)
print("MANIFEST_TASK_IDS:", TASK_IDS)
print("MANIFEST_EPISODES_PER_TASK:", EPISODES_PER_TASK)
MAX_STEPS_BY_SUITE = {"libero_spatial": 25, "libero_object": 25, "libero_goal": 25, "libero_10": 95}
HORIZON = 14

# Option 2 / gold-standard pairing control:
# One deterministic seed per (suite, task, episode), shared by all 8 modes.
# This removes simulator/random-seed differences from same-episode mode comparisons.
FIXED_EPISODE_SEED_BASE = int(os.environ.get("W8A8_FIXED_EPISODE_SEED_BASE", "424242"))
FIXED_EPISODE_SEED_POLICY = "sha256(base|suite|task|episode)_shared_across_modes"

def fixed_episode_seed(suite, task_id, episode_id, base=FIXED_EPISODE_SEED_BASE):
    key = f"{int(base)}|{suite}|task{int(task_id):02d}|ep{int(episode_id):02d}"
    return int((int(base) + int(hashlib.sha256(key.encode("utf-8")).hexdigest()[:8], 16)) % (2**31 - 1))

FLOW_RANGES = {
    "REF_NONE": [],
    "ALL_00_31": list(range(0, 32)),
    "R0_00_07": list(range(0, 8)),
    "R1_08_15": list(range(8, 16)),
    "R2_16_23": list(range(16, 24)),
    "R3_24_31": list(range(24, 32)),
    "CUM_00_15": list(range(0, 16)),
    "CUM_00_23": list(range(0, 24)),
}

A8_SCOPE_V40 = "v40_w8_130_linear_module_a8_116_global_plus_14_flow_windowed_no_random_sites"

MODE_SPECS = {
    "v40_w8a16_reference": {
        "a8_steps": FLOW_RANGES["REF_NONE"],
        "description": "Reference mode: W8 weights on 98 LLM + 32 action_head Linear layers; no A8 activation hooks active. Used as same-episode denominator/noise floor for drift ratios.",
        "range_design": "reference_no_a8_activation",
        "tested_range": "REFERENCE_A16_ACTIVATION",
        "a8_scope": "none_reference_a16_activation",
        "activation_bits": 16,
        "llm_activation": "A8 fixed/global in A8 modes",
        "state_encoder_activation": "A8 fixed/global in A8 modes",
        "expected_use": "same-episode reference for drift ratios",
    },
    "v40_w8a8_ALL_00_31": {
        "a8_steps": FLOW_RANGES["ALL_00_31"],
        "description": "Full-flow W8A8 stress: A8 on all 32 flow steps for the clean linear split: 116 fixed/global Linear inputs + 14 flow-windowed action Linear inputs.",
        "range_design": "full_flow_all_0_31_failure_probe",
        "tested_range": "ALL_00_31",
        "a8_scope": A8_SCOPE_V40,
        "activation_bits": 8,
        "llm_activation": "A8 fixed/global in A8 modes",
        "state_encoder_activation": "A8 fixed/global in A8 modes",
        "expected_use": "failure probe / upper bound on cumulative W8A8 flow-loop damage",
    },
    "v40_w8a8_R0_00_07": {
        "a8_steps": FLOW_RANGES["R0_00_07"],
        "description": "Isolated first 8-step window: only flow steps 0-7 use A8.",
        "range_design": "four_equal_windows_from_v20",
        "tested_range": "R0_00_07",
        "a8_scope": A8_SCOPE_V40,
        "activation_bits": 8,
        "llm_activation": "A8 fixed/global in A8 modes",
        "state_encoder_activation": "A8 fixed/global in A8 modes",
        "expected_use": "early isolated-window sensitivity check",
    },
    "v40_w8a8_R1_08_15": {
        "a8_steps": FLOW_RANGES["R1_08_15"],
        "description": "Isolated second 8-step window: only flow steps 8-15 use A8.",
        "range_design": "four_equal_windows_from_v20",
        "tested_range": "R1_08_15",
        "a8_scope": A8_SCOPE_V40,
        "activation_bits": 8,
        "llm_activation": "A8 fixed/global in A8 modes",
        "state_encoder_activation": "A8 fixed/global in A8 modes",
        "expected_use": "early-mid isolated-window sensitivity check",
    },
    "v40_w8a8_R2_16_23": {
        "a8_steps": FLOW_RANGES["R2_16_23"],
        "description": "Isolated third 8-step window: only flow steps 16-23 use A8.",
        "range_design": "four_equal_windows_from_v20",
        "tested_range": "R2_16_23",
        "a8_scope": A8_SCOPE_V40,
        "activation_bits": 8,
        "llm_activation": "A8 fixed/global in A8 modes",
        "state_encoder_activation": "A8 fixed/global in A8 modes",
        "expected_use": "mid-late isolated-window sensitivity check",
    },
    "v40_w8a8_R3_24_31": {
        "a8_steps": FLOW_RANGES["R3_24_31"],
        "description": "Isolated final 8-step window: only flow steps 24-31 use A8.",
        "range_design": "four_equal_windows_from_v20",
        "tested_range": "R3_24_31",
        "a8_scope": A8_SCOPE_V40,
        "activation_bits": 8,
        "llm_activation": "A8 fixed/global in A8 modes",
        "state_encoder_activation": "A8 fixed/global in A8 modes",
        "expected_use": "late isolated-window sensitivity check",
    },
    "v40_w8a8_CUM_00_15": {
        "a8_steps": FLOW_RANGES["CUM_00_15"],
        "description": "Cumulative prefix: flow steps 0-15 use A8, steps 16-31 stay high precision.",
        "range_design": "cumulative_prefix_error_accumulation",
        "tested_range": "CUM_00_15",
        "a8_scope": A8_SCOPE_V40,
        "activation_bits": 8,
        "llm_activation": "A8 fixed/global in A8 modes",
        "state_encoder_activation": "A8 fixed/global in A8 modes",
        "expected_use": "cumulative break-point check after 16 low-precision steps",
    },
    "v40_w8a8_CUM_00_23": {
        "a8_steps": FLOW_RANGES["CUM_00_23"],
        "description": "Cumulative prefix: flow steps 0-23 use A8, steps 24-31 stay high precision.",
        "range_design": "cumulative_prefix_error_accumulation",
        "tested_range": "CUM_00_23",
        "a8_scope": A8_SCOPE_V40,
        "activation_bits": 8,
        "llm_activation": "A8 fixed/global in A8 modes",
        "state_encoder_activation": "A8 fixed/global in A8 modes",
        "expected_use": "cumulative break-point check after 24 low-precision steps; compare to ALL_00_31",
    },
}

# Config signature: same signature means safe to resume/skip; different signature is a different experiment.
# It includes mode names, A8 step windows, A8 site scope, suite/tasks/episode count, and key eval settings.
# Split immutable quantization config from the expandable run plan.
# IMPORTANT:
# - experiment_config_signature is based only on quantization/mode semantics.
# - It deliberately does NOT include SUITES, TASK_IDS, or EPISODES_PER_TASK.
# - Therefore adding more suites/episodes later reuses/skips already-completed matching rows
#   and Cell 14 summarizes all matching rows together.
QUANT_CONFIG = {
    "experiment_id": EXPERIMENT_ID,
    "mode_specs": MODE_SPECS,
    "a8_scope": A8_SCOPE_V40,
    "w8_scope": "98_llm_linears_plus_32_action_head_linears",
    "state_encoder_activation": "fixed/global A8 for selected 116 linears where applicable_no_A8",
    "llm_vlm_context_activation": "fixed/global A8 for selected 116 linears where applicable_no_A8",
    "horizon": HORIZON,
    "max_steps_by_suite": MAX_STEPS_BY_SUITE,
    "fixed_episode_seed_policy": FIXED_EPISODE_SEED_POLICY,
    "fixed_episode_seed_base": FIXED_EPISODE_SEED_BASE,
    "seed_pairing": "same suite/task/episode seed is used for every mode; seed is stored in manifest and done JSON",
    "resume_semantics": "same experiment_id + quant_config_signature + mode + suite + task + episode + a8_steps may skip/reuse; different quant scope/modes are ignored/rerun",
}
RUN_PLAN = {
    "suites": SUITES,
    "task_ids": TASK_IDS,
    "episodes_per_task": EPISODES_PER_TASK,
    "fixed_episode_seed_policy": FIXED_EPISODE_SEED_POLICY,
    "fixed_episode_seed_base": FIXED_EPISODE_SEED_BASE,
}
EXPERIMENT_CONFIG = {
    "quant_config": QUANT_CONFIG,
    "run_plan": RUN_PLAN,
    "suite_expansion_policy": "You may later add suites or increase episodes_per_task under the same RUN_ID. Existing matching rows are skipped and counted; new suite/task/episode rows run and join the same final summary.",
}
EXPERIMENT_CONFIG_SIGNATURE = hashlib.sha256(
    json.dumps(QUANT_CONFIG, sort_keys=True).encode("utf-8")
).hexdigest()[:16]

manifest = []
for suite in SUITES:
    for tid in TASK_IDS:
        for ep in range(EPISODES_PER_TASK):
            episode_seed = fixed_episode_seed(suite, tid, ep)
            for mode, spec in MODE_SPECS.items():
                result_stem = f"{EXPERIMENT_ID}__{mode}__suite-{suite}__task-{tid:02d}__ep-{ep:02d}"
                manifest.append({
                    "experiment_id": EXPERIMENT_ID,
                    "experiment_config_signature": EXPERIMENT_CONFIG_SIGNATURE,
                    "suite": suite,
                    "task_id": tid,
                    "episode_id": ep,
                    "episode_seed": int(episode_seed),
                    "fixed_episode_seed_policy": FIXED_EPISODE_SEED_POLICY,
                    "fixed_episode_seed_base": int(FIXED_EPISODE_SEED_BASE),
                    "ablation_mode": mode,
                    "a8_steps": spec["a8_steps"],
                    "mode_spec": spec,
                    "max_steps": MAX_STEPS_BY_SUITE[suite],
                    "horizon": HORIZON,
                    "result_stem": result_stem,
                    "done_json_relpath": f"results/{mode}/{result_stem}.done.json",
                    "episode_log_relpath": f"logs/episode_logs/{result_stem}.log",
                    "drift_jsonl_relpath": f"drift_jsonl/{mode}_request_records.jsonl",
                })

MANIFEST_JSON = SUMMARIES / "manifest_v40_linear130_split_ref5ep_windows_cumulative.json"
MODE_SPECS_JSON = SUMMARIES / "mode_specs_v40_linear130_split_ref5ep_windows_cumulative.json"
CONFIG_JSON = SUMMARIES / "experiment_config_v40_linear130_split_ref5ep_windows_cumulative.json"
MANIFEST_JSON.write_text(json.dumps(manifest, indent=2))
MODE_SPECS_JSON.write_text(json.dumps(MODE_SPECS, indent=2))
CONFIG_JSON.write_text(json.dumps({**EXPERIMENT_CONFIG, "experiment_config_signature": EXPERIMENT_CONFIG_SIGNATURE}, indent=2))

print("EXPERIMENT_ID:", EXPERIMENT_ID)
print("MANIFEST_JSON:", MANIFEST_JSON)
print("MODE_SPECS_JSON:", MODE_SPECS_JSON)
print("CONFIG_JSON:", CONFIG_JSON)
print("EXPERIMENT_CONFIG_SIGNATURE:", EXPERIMENT_CONFIG_SIGNATURE)
print("TOTAL_EPISODE_RUNS:", len(manifest))
print("SUITES:", SUITES)
print("TASK_IDS:", TASK_IDS)
print("EPISODES_PER_TASK:", EPISODES_PER_TASK)
print("MODE_ORDER:", list(MODE_SPECS.keys()))
print("A8_SCOPE_V40:", A8_SCOPE_V40)
print("WINDOWS: REF=[], ALL=0-31, R0=0-7, R1=8-15, R2=16-23, R3=24-31, CUM=0-15/0-23")
print("RUN_ORDER: suite -> task -> episode -> mode")
print("STORAGE_NAMING: {EXPERIMENT_ID}__{mode}__suite-{suite}__task-{tid:02d}__ep-{ep:02d}.{log|done.json}")
print("FIRST_16_MANIFEST_ROWS:")
for row in manifest[:16]:
    print(row["suite"], f"task={row['task_id']:02d}", f"ep={row['episode_id']:02d}", row["ablation_mode"], "a8_steps=", row["a8_steps"], "stem=", row["result_stem"])

# Proof that the first task/episode runs all modes before moving on.
if len(manifest) >= len(MODE_SPECS):
    first_key = (manifest[0]["suite"], manifest[0]["task_id"], manifest[0]["episode_id"])
    first_modes = [r["ablation_mode"] for r in manifest[:len(MODE_SPECS)]]
    assert all((r["suite"], r["task_id"], r["episode_id"]) == first_key for r in manifest[:len(MODE_SPECS)]), "BUG: manifest is not episode-first"
    assert first_modes == list(MODE_SPECS.keys()), "BUG: first episode does not run modes in expected order"
# Proof that all modes for the same suite/task/episode share the exact same seed.
seed_groups = {}
for r in manifest:
    k = (r["suite"], int(r["task_id"]), int(r["episode_id"]))
    seed_groups.setdefault(k, set()).add(int(r["episode_seed"]))
bad_seed_groups = {k: v for k, v in seed_groups.items() if len(v) != 1}
if bad_seed_groups:
    raise RuntimeError(f"BUG: fixed episode seed is not shared across modes: {list(bad_seed_groups.items())[:5]}")
print("FIXED_EPISODE_SEED_POLICY:", FIXED_EPISODE_SEED_POLICY)
print("FIXED_EPISODE_SEED_BASE:", FIXED_EPISODE_SEED_BASE)
print("FIRST_10_EPISODE_SEEDS:")
for k in sorted(seed_groups.keys())[:10]:
    print(k, "seed=", sorted(seed_groups[k])[0])
print("MANIFEST_FIXED_SEEDS_SHARED_ACROSS_MODES_OK")

assert FLOW_RANGES["R0_00_07"] == list(range(0,8))
assert FLOW_RANGES["R1_08_15"] == list(range(8,16))
assert FLOW_RANGES["R2_16_23"] == list(range(16,24))
assert FLOW_RANGES["R3_24_31"] == list(range(24,32))
assert FLOW_RANGES["CUM_00_15"] == list(range(0,16))
assert FLOW_RANGES["CUM_00_23"] == list(range(0,24))
print("MANIFEST_EPISODE_FIRST_OK")
print("REFERENCE_MODE_FIRST_OK:", list(MODE_SPECS.keys())[0] == "v40_w8a16_reference")
print("FOUR_EQUAL_WINDOWS_OK")
print("CUMULATIVE_PREFIXES_OK")


MANIFEST_SUITES: ['libero_10']
MANIFEST_TASK_IDS: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
MANIFEST_EPISODES_PER_TASK: 5
EXPERIMENT_ID: v40_linear130_split_ref5ep_windows_cumulative
MANIFEST_JSON: /content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a/summaries/manifest_v40_linear130_split_ref5ep_windows_cumulative.json
MODE_SPECS_JSON: /content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a/summaries/mode_specs_v40_linear130_split_ref5ep_windows_cumulative.json
CONFIG_JSON: /content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a/summaries/experiment_config_v

In [ ]:
# # CELL 12B — Preflight: prove fresh-folder + manifest hygiene before running episodes
# # Run this immediately after Cell 12 and before Cell 13.
# # It aborts if this supposedly fresh run folder already has episode results or drift JSONL.
# import json, os, time, hashlib
# from pathlib import Path
# import pandas as pd

# print("=" * 110)
# print("CELL 12B PREFLIGHT — FRESH RUN FOLDER + MANIFEST CHECK")
# print("=" * 110)

# required_vars = [
#     "FORCE_NEW_RUN", "USER_RUN_ID", "RUN_ID", "RUN_ROOT", "RESULTS", "LOGS", "SUMMARIES", "DRIFT",
#     "BASE_RESULTS", "EXPERIMENT_ID", "manifest", "MODE_SPECS", "EXPERIMENT_CONFIG_SIGNATURE",
# ]
# missing = [v for v in required_vars if v not in globals()]
# if missing:
#     raise RuntimeError(f"PRE_RUN_ABORT missing variables: {missing}. Run Cells 02 and 12 first.")

# run_root = Path(RUN_ROOT).resolve()
# results = Path(RESULTS).resolve()
# logs = Path(LOGS).resolve()
# summaries = Path(SUMMARIES).resolve()
# drift = Path(DRIFT).resolve()
# base_results = Path(BASE_RESULTS).resolve()

# print("FORCE_NEW_RUN:", FORCE_NEW_RUN)
# print("USER_RUN_ID:", USER_RUN_ID)
# print("RUN_ID:", RUN_ID)
# print("RUN_ROOT:", run_root)
# print("RESULTS:", results)
# print("SUMMARIES:", summaries)
# print("DRIFT:", drift)
# print("EXPERIMENT_ID:", EXPERIMENT_ID)
# print("CONFIG_SIGNATURE:", EXPERIMENT_CONFIG_SIGNATURE)

# if base_results not in run_root.parents:
#     raise RuntimeError(f"PRE_RUN_ABORT suspicious RUN_ROOT outside BASE_RESULTS: {run_root}")
# if run_root == base_results:
#     raise RuntimeError("PRE_RUN_ABORT RUN_ROOT is BASE_RESULTS itself")

# if FORCE_NEW_RUN is not True:
#     raise RuntimeError("PRE_RUN_ABORT this notebook is not in fresh-run mode. Set FORCE_NEW_RUN=True or intentionally resume with USER_RUN_ID.")
# if USER_RUN_ID not in (None, "", "None"):
#     raise RuntimeError(f"PRE_RUN_ABORT USER_RUN_ID is set during fresh run: {USER_RUN_ID!r}")

# # Folder name must reflect current experiment and notebook hygiene build.
# expected_prefix = globals().get("RUN_VERSION", EXPERIMENT_ID)
# if not str(RUN_ID).startswith(str(expected_prefix) + "__"):
#     raise RuntimeError(
#         "PRE_RUN_ABORT RUN_ID does not reflect current notebook run version.\n"
#         f"Expected prefix: {expected_prefix}__\nGot: {RUN_ID}"
#     )
# if run_root.name != RUN_ID:
#     raise RuntimeError(f"PRE_RUN_ABORT RUN_ROOT name != RUN_ID: {run_root.name} vs {RUN_ID}")

# # Manifest shape checks.
# manifest_df = pd.DataFrame(manifest)
# if len(manifest_df) == 0:
#     raise RuntimeError("PRE_RUN_ABORT manifest is empty")

# required_manifest_cols = ["result_stem", "ablation_mode", "suite", "task_id", "episode_id", "episode_seed", "a8_steps"]
# missing_cols = [c for c in required_manifest_cols if c not in manifest_df.columns]
# if missing_cols:
#     raise RuntimeError(f"PRE_RUN_ABORT manifest missing columns: {missing_cols}")

# if manifest_df["result_stem"].duplicated().any():
#     dup = manifest_df[manifest_df["result_stem"].duplicated(keep=False)].sort_values("result_stem")
#     print(dup[["result_stem", "ablation_mode", "suite", "task_id", "episode_id"]].to_string(index=False))
#     raise RuntimeError("PRE_RUN_ABORT duplicate result_stem rows in manifest")

# expected_groups = manifest_df[["suite", "task_id", "episode_id"]].drop_duplicates()
# expected_per_mode = int(len(expected_groups))
# expected_mode_set = set(MODE_SPECS.keys())
# manifest_mode_set = set(manifest_df["ablation_mode"].unique())
# if manifest_mode_set != expected_mode_set:
#     print("EXPECTED MODES:", sorted(expected_mode_set))
#     print("MANIFEST MODES:", sorted(manifest_mode_set))
#     raise RuntimeError("PRE_RUN_ABORT manifest mode set does not match MODE_SPECS")
# expected_total = int(expected_per_mode * len(expected_mode_set))

# expected_by_mode = manifest_df.groupby("ablation_mode").size().rename("expected_runs").reset_index()
# print("\nEXPECTED MANIFEST RUNS BY MODE")
# print(expected_by_mode.to_string(index=False))
# print("EXPECTED_GROUPS_SUITE_TASK_EPISODE:", expected_per_mode)
# print("EXPECTED_MODE_COUNT:", len(expected_mode_set))
# print("EXPECTED_TOTAL_MANIFEST_ROWS:", expected_total)

# if len(manifest_df) != expected_total:
#     raise RuntimeError(f"PRE_RUN_ABORT expected {expected_total} manifest rows from groups×modes, got {len(manifest_df)}")
# if not expected_by_mode["expected_runs"].eq(expected_per_mode).all():
#     raise RuntimeError(f"PRE_RUN_ABORT expected exactly {expected_per_mode} manifest rows per mode")

# seed_check = manifest_df.groupby(["suite", "task_id", "episode_id"])["episode_seed"].nunique().reset_index(name="n_unique_seeds")
# bad_seed = seed_check[seed_check["n_unique_seeds"] != 1]
# if len(bad_seed):
#     print(bad_seed.to_string(index=False))
#     raise RuntimeError("PRE_RUN_ABORT fixed episode seed is not identical across modes for same suite/task/episode")
# print("OK_FIXED_EPISODE_SEEDS_SHARED_ACROSS_MODES")
# print("unique_episode_seeds:", seed_check.shape[0])

# # Fresh output-folder checks.
# done_json = sorted(results.glob("*/*.done.json"))
# episode_logs = sorted(results.glob("*/*.log"))
# episode_json = sorted(results.glob("*/*.json"))
# drift_jsonl = sorted(drift.glob("*.jsonl"))

# print("\nOUTPUT COUNTS BEFORE EPISODES")
# print("done_json:", len(done_json))
# print("episode_logs:", len(episode_logs))
# print("episode_json:", len(episode_json))
# print("drift_jsonl:", len(drift_jsonl))

# if done_json or episode_logs or episode_json or drift_jsonl:
#     print("\nDIRTY FILES FOUND — first 100:")
#     for p in (done_json + episode_logs + episode_json + drift_jsonl)[:100]:
#         print(p)
#     raise RuntimeError(
#         "PRE_RUN_ABORT fresh run folder already contains episode/drift outputs. "
#         "Use a new RUN_ID/folder or delete only this RUN_ROOT before running episodes."
#     )

# proof = {
#     "time": time.strftime("%Y-%m-%d %H:%M:%S"),
#     "run_id": RUN_ID,
#     "run_root": str(run_root),
#     "experiment_id": EXPERIMENT_ID,
#     "run_version": str(globals().get("RUN_VERSION", EXPERIMENT_ID)),
#     "config_signature": EXPERIMENT_CONFIG_SIGNATURE,
#     "manifest_rows": int(len(manifest_df)),
#     "expected_total_manifest_rows": int(expected_total),
#     "expected_per_mode": int(expected_per_mode),
#     "expected_by_mode": expected_by_mode.to_dict(orient="records"),
#     "done_json_before": len(done_json),
#     "episode_logs_before": len(episode_logs),
#     "episode_json_before": len(episode_json),
#     "drift_jsonl_before": len(drift_jsonl),
# }
# proof_path = summaries / "preflight_fresh_folder_manifest_check.json"
# proof_path.write_text(json.dumps(proof, indent=2))
# print("\nSAVED_PREFLIGHT_PROOF:", proof_path)
# print("OK_PREFLIGHT_FRESH_FOLDER_AND_MANIFEST_CLEAN")






CELL 12B PREFLIGHT — FRESH RUN FOLDER + MANIFEST CHECK
FORCE_NEW_RUN: False
USER_RUN_ID: v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a
RUN_ID: v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a
RUN_ROOT: /content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a
RESULTS: /content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a/results
SUMMARIES: /content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_resear

RuntimeError: PRE_RUN_ABORT this notebook is not in fresh-run mode. Set FORCE_NEW_RUN=True or intentionally resume with USER_RUN_ID.

In [22]:

# CELL 13 — Run v40 clean linear-split ablation episodes, episode-first paired mode order, resumable by mode/task/episode
# v40 output: clear multi-line per-episode metrics plus same-episode ratios vs v40_w8a16_reference; full records stay in drift_jsonl.
# Explicit resume can skip completed rows. Fresh-run reruns are guarded: if outputs already exist, this cell aborts and tells you to rerun Cell 02 for a new clean folder.
import subprocess, os, time, json, re
from pathlib import Path

MAMBA = "/content/micromamba/bin/micromamba"
MAMBA_ROOT = "/content/micromamba-root"
RUN_LOGS = LOGS / "episode_logs"
RUN_LOGS.mkdir(parents=True, exist_ok=True)

ABORT_ON_FIRST_SERVER_CRASH = True
PRINT_LOG_TAIL_CHARS = 1200
REFERENCE_MODE = "v40_w8a16_reference"
CURRENT_CONFIG_SIGNATURE = globals().get("EXPERIMENT_CONFIG_SIGNATURE", None)
RATIO_EPS = 1e-6
CURRENT_MODE_SET = set(MODE_SPECS.keys()) if "MODE_SPECS" in globals() else None
EXPERIMENT_ID = globals().get("EXPERIMENT_ID", "v40_linear130_split_ref5ep_windows_cumulative")

# Filled after Cell 12 manifest exists; used to prevent old/stale files from entering summaries.
MANIFEST_STEMS = set(str(x.get("result_stem", "")) for x in manifest) if "manifest" in globals() else set()

# Fresh-run interruption guard.
# If Colab disconnects mid-Cell 13, do NOT rerun Cell 13 in the same runtime/folder.
# Rerun Cell 02 to get a new clean timestamp+uuid RUN_ROOT, then rebuild manifest and start Cell 13.
# This prevents partial .done.json and request_records.jsonl from mixing with a restarted fresh run.
_cell13_existing_done_json = sorted(Path(RESULTS).glob("*/*.done.json"))
_cell13_existing_jsonl = sorted(Path(DRIFT).glob("*.jsonl"))
if bool(globals().get("FORCE_NEW_RUN", False)) and (_cell13_existing_done_json or _cell13_existing_jsonl):
    print("CELL13_FRESH_RUN_DIRTY_ABORT")
    print("existing_done_json:", len(_cell13_existing_done_json))
    print("existing_drift_jsonl:", len(_cell13_existing_jsonl))
    for p in (_cell13_existing_done_json[:10] + _cell13_existing_jsonl[:10]):
        print("DIRTY_OUTPUT:", p)
    raise RuntimeError(
        "Fresh-run folder already contains episode outputs or drift JSONL. "
        "This usually means Cell 13 was interrupted and you are trying to rerun it without returning to Cell 02. "
        "For a clean fresh run, rerun Cell 02 to create a new timestamp+uuid RUN_ROOT, then rerun Cells 10-13. "
        "Do not continue in this dirty folder."
    )

if not bool(globals().get("FORCE_NEW_RUN", False)):
    print("CELL13_EXPLICIT_RESUME_MODE: completed exact manifest rows will be skipped; missing/incomplete rows will run.")
    print("RUN_ID:", RUN_ID)
    print("CURRENT_MANIFEST_ROWS:", len(manifest) if "manifest" in globals() else "<missing>")
    print("EXISTING_DONE_JSON:", len(_cell13_existing_done_json))
    print("EXISTING_DRIFT_JSONL:", len(_cell13_existing_jsonl))


def tail(path, n=12000):
    path = Path(path)
    if not path.exists():
        return f"[missing] {path}"
    return path.read_text(errors="ignore")[-n:]

def _port_9010_info_local():
    r = subprocess.run("ss -ltnp | grep ':9010' || true", shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    out = r.stdout.strip()
    pid = None
    cmdline = ""
    m = re.search(r"pid=(\d+)", out)
    if m:
        pid = int(m.group(1))
        try:
            cmdline = Path(f"/proc/{pid}/cmdline").read_text(errors="ignore").replace("\x00", " ")
        except Exception as exc:
            cmdline = f"<could not read cmdline: {exc}>"
    return out, pid, cmdline

def _server_matches():
    out, pid, cmdline = _port_9010_info_local()
    if not pid:
        return False, out, pid, cmdline
    ok = (str(SERVER_SCRIPT) in cmdline) or (str(RUN_ROOT) in cmdline and SERVER_SCRIPT.name in cmdline)
    return ok, out, pid, cmdline

def print_running_summary():
    rows = []
    manifest_stems = set(str(x.get("result_stem", "")) for x in manifest) if "manifest" in globals() else MANIFEST_STEMS
    for p in sorted(RESULTS.glob("*/*.done.json")):
        try:
            d = json.loads(p.read_text())
            # Critical: summarize only rows in the current manifest, not every file in RESULTS.
            if manifest_stems and str(d.get("result_stem", "")) not in manifest_stems:
                continue
            if CURRENT_MODE_SET is not None and d.get("ablation_mode") not in CURRENT_MODE_SET:
                continue
            if d.get("experiment_id") != EXPERIMENT_ID:
                continue
            if CURRENT_CONFIG_SIGNATURE is not None and d.get("experiment_config_signature") != CURRENT_CONFIG_SIGNATURE:
                continue
            if "success" in d:
                rows.append(d)
        except Exception:
            pass
    if not rows:
        return
    import pandas as pd
    dfr = pd.DataFrame(rows)
    for c in ["success", "server_crash", "timeout_or_missing", "task_fail", "client_exception"]:
        if c not in dfr.columns:
            dfr[c] = False
        dfr[c] = dfr[c].fillna(False).astype(bool)
    g = dfr.groupby("ablation_mode").agg(
        runs=("success", "count"),
        success=("success", "sum"),
        crash=("server_crash", "sum"),
        timeout=("timeout_or_missing", "sum"),
        client_exception=("client_exception", "sum"),
        task_fail=("task_fail", "sum"),
    ).reset_index()
    g["success_rate"] = g["success"] / g["runs"].clip(lower=1)
    print()
    print("RUNNING_SUMMARY_BY_MODE")
    print(g.to_string(index=False))
    if "manifest" in globals():
        expected = pd.DataFrame(manifest).groupby("ablation_mode").size().rename("expected_runs").reset_index()
        chk = expected.merge(g[["ablation_mode", "runs"]], on="ablation_mode", how="left")
        chk["runs"] = chk["runs"].fillna(0).astype(int)
        bad = chk[chk["expected_runs"] != chk["runs"]]
        if len(bad):
            print()
            print("WARNING_MANIFEST_COUNT_MISMATCH")
            print(bad.to_string(index=False))
        else:
            print()
            print("OK_MANIFEST_COUNTS_MATCH")


def _rewrite_jsonl_without_episode(fp, mode, suite, tid, ep, result_stem=None):
    """Remove stale request-level records for one episode before rerunning an incomplete row.
    This is only used when a .done.json is absent or invalid. It prevents duplicated JSONL request records
    after a Colab/kernel interruption in explicit resume mode.
    """
    fp = Path(fp)
    if not fp.exists():
        return 0
    kept = []
    removed = 0
    try:
        with fp.open("r") as f:
            for line in f:
                raw = line.rstrip("\n")
                if not raw.strip():
                    continue
                try:
                    r = json.loads(raw)
                    same_episode = (
                        r.get("ablation_mode") == mode and
                        r.get("suite") == suite and
                        int(r.get("task_id", -999)) == int(tid) and
                        int(r.get("episode_id", -999)) == int(ep)
                    )
                    if result_stem and r.get("result_stem") not in (None, "", result_stem):
                        same_episode = False
                    if same_episode:
                        removed += 1
                    else:
                        kept.append(raw)
                except Exception:
                    kept.append(raw)
        if removed:
            tmp = fp.with_suffix(fp.suffix + ".tmp")
            tmp.write_text("\n".join(kept) + ("\n" if kept else ""))
            tmp.replace(fp)
    except Exception as exc:
        print("JSONL_STALE_CLEAN_WARN", fp, exc)
    return removed


def _clean_stale_jsonl_for_rerun(mode, suite, tid, ep, result_stem=None):
    drift_dir = RUN_ROOT / "drift_jsonl"
    total = 0
    total += _rewrite_jsonl_without_episode(drift_dir / "all_modes_request_records.jsonl", mode, suite, tid, ep, result_stem)
    total += _rewrite_jsonl_without_episode(drift_dir / f"{mode}_request_records.jsonl", mode, suite, tid, ep, result_stem)
    if total:
        print(f"CLEANED_STALE_JSONL_FOR_RERUN mode={mode} suite={suite} task={int(tid):02d} ep={int(ep):02d} removed_records={total}", flush=True)


def _load_episode_drift_records(mode, suite, tid, ep):
    """Return all request-level drift records for one episode from combined/per-mode JSONL."""
    drift_dir = RUN_ROOT / "drift_jsonl"
    candidates = [drift_dir / "all_modes_request_records.jsonl", drift_dir / f"{mode}_request_records.jsonl"]
    records = []
    seen = set()
    for fp in candidates:
        if not fp.exists():
            continue
        try:
            with fp.open("r") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        r = json.loads(line)
                    except Exception:
                        continue
                    if (r.get("ablation_mode") == mode and r.get("suite") == suite and
                        int(r.get("task_id", -999)) == int(tid) and int(r.get("episode_id", -999)) == int(ep)):
                        key = (r.get("request_id"), r.get("outer_step"), r.get("ablation_mode"))
                        if key not in seen:
                            records.append(r)
                            seen.add(key)
        except Exception as exc:
            print("DRIFT_JSONL_READ_WARN", fp, exc)
    records.sort(key=lambda r: (int(r.get("outer_step", 0)), int(r.get("request_id", 0))))
    return records

def _summarize_episode_drift(records):
    if not records:
        return None
    def vals(key):
        out=[]
        for r in records:
            v=r.get(key, None)
            if isinstance(v, (int,float)):
                out.append(float(v))
        return out
    def max_or_none(xs): return max(xs) if xs else None
    def mean_or_none(xs): return sum(xs)/len(xs) if xs else None
    step_vel=[]; step_endpoint=[]; step_after=[]; step_before=[]
    enabled_steps=set(); qact_total=0; site_total={}
    for r in records:
        for srec in r.get("step_records", []) or []:
            if isinstance(srec, dict):
                if isinstance(srec.get("qdrift_velocity_rms"), (int,float)):
                    step_vel.append(float(srec["qdrift_velocity_rms"]))
                if isinstance(srec.get("flash_endpoint_pair_rms"), (int,float)):
                    step_endpoint.append(float(srec["flash_endpoint_pair_rms"]))
                if isinstance(srec.get("qdrift_state_after_rms"), (int,float)):
                    step_after.append(float(srec["qdrift_state_after_rms"]))
                if isinstance(srec.get("qdrift_state_before_rms"), (int,float)):
                    step_before.append(float(srec["qdrift_state_before_rms"]))
                if int(srec.get("a8_enabled_this_step", 0)):
                    enabled_steps.add(int(srec.get("flow_step", -1)))
        counts = r.get("a8_qact_site_counts", {}) or {}
        if isinstance(counts, dict):
            for k,v in counts.items():
                try:
                    site_total[k] = site_total.get(k, 0) + int(v)
                    qact_total += int(v)
                except Exception:
                    pass
    out = {
        "requests": len(records),
        "final_rms_mean": mean_or_none(vals("final_action_7d_rms")),
        "final_rms_max": max_or_none(vals("final_action_7d_rms")),
        "final_maxabs_max": max_or_none(vals("final_action_7d_max_abs")),
        "xyz_rms_mean": mean_or_none(vals("xyz_rms")),
        "rot_rms_mean": mean_or_none(vals("rot_rms")),
        "grip_mismatch_total": int(sum(vals("gripper_threshold_mismatch_count"))),
        "grip_boundary_total": int(sum(vals("gripper_boundary_count"))),
        "vel_rms_max": max_or_none(step_vel),
        "endpoint_rms_max": max_or_none(step_endpoint),
        "state_after_rms_max": max_or_none(step_after),
        "state_before_rms_max": max_or_none(step_before),
        "enabled_steps": sorted([x for x in enabled_steps if x >= 0]),
        "qact_calls_total": int(qact_total),
        "top_sites": sorted(site_total.items(), key=lambda kv: kv[1], reverse=True)[:6],
    }
    return out

def _fmt_metric(v):
    if v is None:
        return "NA"
    if isinstance(v, float):
        return f"{v:.6g}"
    return str(v)


def _safe_ratio(v, ref, eps=RATIO_EPS):
    if v is None or ref is None:
        return None
    try:
        return float(v) / max(abs(float(ref)), eps)
    except Exception:
        return None

def _result_stem_for(mode, suite, tid, ep):
    # Prefer exact manifest result_stem so filenames are unique across experiment versions.
    if "manifest" in globals():
        for _it in manifest:
            if (_it.get("ablation_mode") == mode and _it.get("suite") == suite and
                int(_it.get("task_id", -999)) == int(tid) and int(_it.get("episode_id", -999)) == int(ep)):
                return _it.get("result_stem") or f"{EXPERIMENT_ID}__{mode}__suite-{suite}__task-{tid:02d}__ep-{ep:02d}"
    return f"{EXPERIMENT_ID}__{mode}__suite-{suite}__task-{tid:02d}__ep-{ep:02d}"

def _ref_done_json_path(suite, tid, ep):
    tag = _result_stem_for(REFERENCE_MODE, suite, tid, ep)
    return RESULTS / REFERENCE_MODE / f"{tag}.done.json"

def _load_reference_drift_summary(suite, tid, ep):
    p = _ref_done_json_path(suite, tid, ep)
    if not p.exists():
        return None, p
    try:
        d = json.loads(p.read_text())
        return d.get("drift_metrics_summary"), p
    except Exception:
        return None, p

def _print_clear_metrics_block(mode, suite, tid, ep, agg, ref_agg, s):
    print("" + "-" * 96, flush=True)
    print(f"METRICS_BLOCK mode={mode} suite={suite} task={tid:02d} ep={ep:02d}", flush=True)
    print(f"  result={'SUCCESS' if s.get('success') else 'FAIL'} reason="
          f"{'success' if s.get('success') else ('server_crash' if s.get('server_crash') else 'timeout' if s.get('timeout_or_missing') else 'client_exception' if s.get('client_exception') else 'task_fail')} "
          f"env_steps={s.get('env_steps')} outer_steps={s.get('outer_steps')}", flush=True)
    print(f"  enabled_steps={agg.get('enabled_steps')} qact_calls={agg.get('qact_calls_total')} requests={agg.get('requests')}", flush=True)
    print("  ACTION_DRIFT:", flush=True)
    print(f"    final_rms_mean={_fmt_metric(agg.get('final_rms_mean'))}  final_rms_max={_fmt_metric(agg.get('final_rms_max'))}  final_maxabs_max={_fmt_metric(agg.get('final_maxabs_max'))}", flush=True)
    print(f"    xyz_rms_mean={_fmt_metric(agg.get('xyz_rms_mean'))}  rot_rms_mean={_fmt_metric(agg.get('rot_rms_mean'))}", flush=True)
    print("  FLOW_DRIFT:", flush=True)
    print(f"    vel_rms_max={_fmt_metric(agg.get('vel_rms_max'))}  endpoint_rms_max={_fmt_metric(agg.get('endpoint_rms_max'))}  state_after_rms_max={_fmt_metric(agg.get('state_after_rms_max'))}  state_before_rms_max={_fmt_metric(agg.get('state_before_rms_max'))}", flush=True)
    print("  GRIPPER:", flush=True)
    print(f"    grip_mismatch_total={agg.get('grip_mismatch_total')}  grip_boundary_total={agg.get('grip_boundary_total')}", flush=True)
    if ref_agg and mode != REFERENCE_MODE:
        ratios = {
            "final_rms_mean_ratio_vs_ref": _safe_ratio(agg.get("final_rms_mean"), ref_agg.get("final_rms_mean")),
            "final_rms_max_ratio_vs_ref": _safe_ratio(agg.get("final_rms_max"), ref_agg.get("final_rms_max")),
            "vel_rms_max_ratio_vs_ref": _safe_ratio(agg.get("vel_rms_max"), ref_agg.get("vel_rms_max")),
            "endpoint_rms_max_ratio_vs_ref": _safe_ratio(agg.get("endpoint_rms_max"), ref_agg.get("endpoint_rms_max")),
            "state_after_rms_max_ratio_vs_ref": _safe_ratio(agg.get("state_after_rms_max"), ref_agg.get("state_after_rms_max")),
        }
        print("  VS_W8A16_REFERENCE:", flush=True)
        print(f"    ref_final_rms_mean={_fmt_metric(ref_agg.get('final_rms_mean'))}  ratio={_fmt_metric(ratios['final_rms_mean_ratio_vs_ref'])}", flush=True)
        print(f"    ref_final_rms_max ={_fmt_metric(ref_agg.get('final_rms_max'))}  ratio={_fmt_metric(ratios['final_rms_max_ratio_vs_ref'])}", flush=True)
        print(f"    ref_vel_rms_max   ={_fmt_metric(ref_agg.get('vel_rms_max'))}  ratio={_fmt_metric(ratios['vel_rms_max_ratio_vs_ref'])}", flush=True)
        print(f"    ref_endpoint_max  ={_fmt_metric(ref_agg.get('endpoint_rms_max'))}  ratio={_fmt_metric(ratios['endpoint_rms_max_ratio_vs_ref'])}", flush=True)
        print(f"    ref_state_after   ={_fmt_metric(ref_agg.get('state_after_rms_max'))}  ratio={_fmt_metric(ratios['state_after_rms_max_ratio_vs_ref'])}", flush=True)
        return ratios
    elif mode == REFERENCE_MODE:
        print("  VS_W8A16_REFERENCE: this is the reference/no-A8 baseline for this same task+episode", flush=True)
    else:
        ref_path = _ref_done_json_path(suite, tid, ep)
        print(f"  VS_W8A16_REFERENCE: missing reference drift summary at {ref_path}", flush=True)
    return {}

def print_episode_drift_metrics(mode, suite, tid, ep, summary_path):
    records = _load_episode_drift_records(mode, suite, tid, ep)
    agg = _summarize_episode_drift(records)
    if not agg:
        print(f"EPISODE_METRICS mode={mode} suite={suite} task={tid:02d} ep={ep:02d} drift_records=0 DRIFT_JSONL_MISSING_OR_NO_REQUESTS", flush=True)
        return {}
    ref_agg, ref_path = _load_reference_drift_summary(suite, tid, ep)
    # For the reference mode itself, ratio denominator is itself; for later modes, Cell 12 orders reference first.
    ratios = _print_clear_metrics_block(mode, suite, tid, ep, agg, ref_agg, json.loads(Path(summary_path).read_text()) if Path(summary_path).exists() else {})
    patch = {"drift_records": agg["requests"], "drift_metrics_summary": agg}
    if ratios:
        patch["drift_ratio_vs_w8a16_reference"] = ratios
        patch["w8a16_reference_done_json"] = str(ref_path)
    try:
        sj = json.loads(Path(summary_path).read_text()) if Path(summary_path).exists() else {}
        sj.update(patch)
        Path(summary_path).write_text(json.dumps(sj, indent=2))
    except Exception as exc:
        print("DONE_JSON_DRIFT_PATCH_WARN", summary_path, exc)
    print(f"EPISODE_DRIFT_JSONL dir={RUN_ROOT / 'drift_jsonl'} combined=all_modes_request_records.jsonl per_mode={mode}_request_records.jsonl", flush=True)
    if agg.get("top_sites"):
        print("EPISODE_A8_TOP_SITES " + json.dumps(agg["top_sites"]), flush=True)
    return patch

# Make sure the correct server is up. If cell 11 was run, ensure_w8dyn_server exists and can recover.
ok, proof, pid, cmdline = _server_matches()
if not ok:
    print("SERVER_NOT_MATCHING_BEFORE_RUN")
    print("PORT_PROOF:", proof if proof else "PORT_9010_FREE")
    print("PID:", pid)
    print("CMDLINE:", cmdline[:1000])
    if "ensure_w8dyn_server" in globals():
        print("CALLING ensure_w8dyn_server() FROM CELL 11")
        ensure_w8dyn_server()
    else:
        raise RuntimeError("Correct server is not running and ensure_w8dyn_server() is not defined. Run CELL 11, then rerun CELL 13.")

if 'manifest' not in globals():
    raise RuntimeError("manifest is not defined. Run CELL 12 first.")

def _norm_steps_for_compare(x):
    if x is None:
        return []
    return [int(v) for v in list(x)]

def _done_matches_manifest_item(done, item):
    """Only skip/reuse a done JSON when it matches the exact current manifest row/config."""
    try:
        checks = [
            done.get("experiment_id") == item.get("experiment_id", EXPERIMENT_ID),
            done.get("experiment_config_signature") == item.get("experiment_config_signature", CURRENT_CONFIG_SIGNATURE),
            done.get("ablation_mode") == item.get("ablation_mode"),
            done.get("suite") == item.get("suite"),
            int(done.get("task_id", -999)) == int(item.get("task_id", -888)),
            int(done.get("episode_id", -999)) == int(item.get("episode_id", -888)),
            _norm_steps_for_compare(done.get("a8_steps")) == _norm_steps_for_compare(item.get("a8_steps")),
            str(done.get("result_stem", "")) == str(item.get("result_stem", "")),
        ]
        return all(checks)
    except Exception:
        return False

def _done_matches_current_config(done):
    if CURRENT_MODE_SET is not None and done.get("ablation_mode") not in CURRENT_MODE_SET:
        return False
    if done.get("experiment_id") != EXPERIMENT_ID:
        return False
    if CURRENT_CONFIG_SIGNATURE is not None and done.get("experiment_config_signature") != CURRENT_CONFIG_SIGNATURE:
        return False
    return True

print("TOTAL_MANIFEST_EPISODE_RUNS:", len(manifest))
print("MANIFEST_RUN_ORDER: v40 Cell 12 builds suite -> task -> episode -> mode; reference first, then ALL/R0/R1/R2/R3 + CUM_00_15/CUM_00_23")
completed = 0
ran_now = 0
skipped = 0
crashed = 0

for idx, item in enumerate(manifest):
    mode = item["ablation_mode"]
    suite = item["suite"]; tid = item["task_id"]; ep = item["episode_id"]
    episode_seed = int(item.get("episode_seed", os.environ.get("W8A8_FIXED_EPISODE_SEED_BASE", "424242")))
    seed_policy = str(item.get("fixed_episode_seed_policy", globals().get("FIXED_EPISODE_SEED_POLICY", "unknown")))
    tag = item.get("result_stem") or _result_stem_for(mode, suite, tid, ep)
    mode_result_dir = RESULTS / mode
    mode_result_dir.mkdir(parents=True, exist_ok=True)
    log_path = RUN_LOGS / f"{tag}.log"
    summary_path = mode_result_dir / f"{tag}.done.json"

    if summary_path.exists():
        try:
            s0 = json.loads(summary_path.read_text())
            # Skip only if this done JSON matches the exact current manifest row/config.
            # Same exact config resumes; different settings/layer scopes/mode names are ignored and rerun.
            if _done_matches_manifest_item(s0, item) and "success" in s0 and not s0.get("server_crash", False) and not s0.get("timeout_or_missing", False) and not s0.get("client_exception", False):
                print(f"{'✅' if s0.get('success') else '❌'} SKIP_DONE {idx+1}/{len(manifest)} {tag} result={'SUCCESS' if s0.get('success') else 'FAIL'} exact_config_match={s0.get('experiment_config_signature')}")
                skipped += 1
                completed += 1
                continue
            else:
                print("DONE_JSON_EXISTS_BUT_CONFIG_MISMATCH_OR_INCOMPLETE_WILL_RERUN", summary_path)
        except Exception as exc:
            print("DONE_JSON_READ_WARN_WILL_RERUN", summary_path, exc)

    ok, proof, pid, cmdline = _server_matches()
    if not ok:
        print("SERVER_NOT_MATCHING before", tag)
        print("PORT_PROOF:", proof if proof else "PORT_9010_FREE")
        print("PID:", pid)
        print("CMDLINE:", cmdline[:1000])
        print("SERVER_LOG_TAIL:")
        print(tail(SERVER_LOG, 6000))
        if "ensure_w8dyn_server" in globals():
            print("ATTEMPT_SERVER_RECOVERY")
            ensure_w8dyn_server()
        else:
            raise RuntimeError("Server not matching and no recovery function. Run CELL 11.")

    env = {
        **os.environ,
        "MAMBA_ROOT_PREFIX": MAMBA_ROOT,
        "FLOWA8_RUN_ROOT": str(RUN_ROOT),
        "FLOWA8_SUITE": suite,
        "FLOWA8_TASK_ID": str(tid),
        "FLOWA8_EPISODE_ID": str(ep),
        "FLOWA8_MAX_STEPS": str(item["max_steps"]),
        "FLOWA8_HORIZON": str(item["horizon"]),
        "FLOWA8_SUMMARY_PATH": str(summary_path),
        "W8A8_ABLATION_MODE": mode,
        "W8A8_A8_STEPS": ",".join(map(str, item["a8_steps"])),
        "FLOWA8_SEED": str(episode_seed),
        "FLOWA8_FIXED_EPISODE_SEED_POLICY": seed_policy,
        "PYTHONHASHSEED": str(episode_seed),
        "W8A8_CLIENT_VERBOSE_STEPS": "0",
        "PYTHONPATH": str(WORK_REPO / "LIBERO_evaluation" / "LIBERO") + ":" + str(WORK_REPO / "LIBERO_evaluation") + ":" + os.environ.get("PYTHONPATH", ""),
    }
    cmd = [MAMBA, "run", "-n", "libero", "python", str(CLIENT_SCRIPT)]
    # If this row is about to run/re-run, clear stale request JSONL for this exact episode/mode.
    # Completed exact rows above are skipped and keep their JSONL; incomplete rows get a clean request-record slate.
    _clean_stale_jsonl_for_rerun(mode, suite, tid, ep, result_stem=str(item.get("result_stem", "")))

    print(f"RUN {idx+1}/{len(manifest)} {tag} seed={episode_seed} a8_steps={item['a8_steps']}")
    t0 = time.time()
    try:
        r = subprocess.run(cmd, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=60*30)
        out = r.stdout or ""
        ret = r.returncode
    except subprocess.TimeoutExpired as exc:
        out = (exc.stdout or "") + "\n[TIMEOUT_EXPIRED]\n" + (exc.stderr or "")
        ret = 124
    elapsed = time.time() - t0
    log_path.write_text(out)

    try:
        s = json.loads(summary_path.read_text()) if summary_path.exists() else {}
    except Exception:
        s = {}
    s.update({
        "experiment_id": item.get("experiment_id", EXPERIMENT_ID),
        "experiment_config_signature": item.get("experiment_config_signature", CURRENT_CONFIG_SIGNATURE),
        "mode_spec": item.get("mode_spec", {}),
        "result_stem": tag,
        "ablation_mode": mode,
        "a8_steps": item["a8_steps"],
        "suite": suite,
        "task_id": tid,
        "episode_id": ep,
        "episode_seed": int(episode_seed),
        "fixed_episode_seed_policy": seed_policy,
        "elapsed_sec": elapsed,
        "returncode": ret,
        "log_path": str(log_path),
        "done_json": str(summary_path),
    })
    if "success" not in s:
        s["success"] = False
    for flag in ["server_crash", "timeout_or_missing", "client_exception", "task_fail", "egl_cleanup_warning"]:
        s.setdefault(flag, False)

    if "EGL_NOT_INITIALIZED" in out and ret == 0:
        s["egl_cleanup_warning"] = True

    server_failure_patterns = [
        "ConnectionClosedError",
        "received 1011",
        "ConnectionRefusedError",
        "Connect call failed",
        "Errno 111",
        "server rejected WebSocket connection",
        "InvalidHandshake",
    ]
    if "TIMEOUT_EXPIRED" in out or ret == 124:
        s["timeout_or_missing"] = True
    elif any(p in out for p in server_failure_patterns):
        s["server_crash"] = True
    elif ret != 0:
        # A Python exception inside the generated client is not proof the Evo server died.
        # Example fixed in v13: NameError: CLIENT_VERBOSE_STEPS was a client bug.
        s["client_exception"] = True

    if not s.get("success", False) and not s.get("server_crash", False) and not s.get("timeout_or_missing", False) and not s.get("client_exception", False):
        s["task_fail"] = True
    summary_path.write_text(json.dumps(s, indent=2))

    result_label = "SUCCESS" if s.get("success", False) else "FAIL"
    if s.get("success", False):
        reason = "success"
    elif s.get("server_crash", False):
        reason = "server_crash"
    elif s.get("timeout_or_missing", False):
        reason = "timeout"
    elif s.get("client_exception", False):
        reason = "client_exception"
    else:
        reason = "task_fail"
    _icon = "✅" if s.get("success") else "❌"
    print(
        f"{_icon} EPISODE_DONE {idx+1}/{len(manifest)} mode={mode} suite={suite} task={tid:02d} ep={ep:02d} "
        f"{result_label} reason={reason} env_steps={s.get('env_steps')} outer_steps={s.get('outer_steps')} "
        f"elapsed={elapsed:.1f}s log={log_path.name}",
        flush=True,
    )
    def _quick_count(scope):
        w, t = 0, 0
        for _p in mode_result_dir.glob("*.done.json"):
            try:
                _d = json.loads(_p.read_text())
                if not _done_matches_current_config(_d):
                    continue
                if MANIFEST_STEMS and str(_d.get("result_stem", "")) not in MANIFEST_STEMS:
                    continue
                if _d.get("ablation_mode") != mode or _d.get("suite") != suite:
                    continue
                if scope == "task" and int(_d.get("task_id", -999)) != int(tid):
                    continue
                if "success" in _d and not _d.get("server_crash") and not _d.get("timeout_or_missing") and not _d.get("client_exception"):
                    t += 1; w += int(bool(_d["success"]))
            except Exception:
                pass
        return w, t
    _tw, _tt = _quick_count("task")
    _sw, _st = _quick_count("suite")
    print(
        f"  ACCURACY task{tid:02d} {_tw}/{_tt}={100*_tw//max(1,_tt)}% | {suite} {_sw}/{_st}={100*_sw//max(1,_st)}% | mode={mode}",
        flush=True,
    )
    # Restore per-episode ablation/drift visibility: the server writes full request records to drift_jsonl;
    # this prints a compact summary and also patches the done JSON with the same aggregate.
    drift_patch = print_episode_drift_metrics(mode, suite, tid, ep, summary_path)
    if drift_patch:
        s.update(drift_patch)

    ran_now += 1
    completed += 1

    if s.get("server_crash", False) or s.get("timeout_or_missing", False) or s.get("client_exception", False):
        crashed += 1
        print("INFRA_OR_CLIENT_FAILURE_DETECTED", tag, "reason=", reason)
        print("DONE_JSON:", summary_path)
        print("CLIENT_LOG_TAIL:")
        print(out[-PRINT_LOG_TAIL_CHARS:])
        print("SERVER_LOG_TAIL:")
        print(tail(SERVER_LOG, 6000))
        if ABORT_ON_FIRST_SERVER_CRASH:
            if s.get("client_exception", False):
                raise RuntimeError("Aborting after client exception. Fix generated client/code issue, then rerun CELL 10 -> 10B -> 11 -> 13; completed episodes will skip.")
            raise RuntimeError("Aborting after true server crash/timeout. Fix server/log issue, then rerun CELL 11 -> CELL 13; completed episodes will skip.")

    if completed % 20 == 0:
        print(f"PROGRESS completed_or_skipped={completed}/{len(manifest)} ran_now={ran_now} skipped={skipped} infra_or_client_failures={crashed}")
        print_running_summary()

print("DONE_MANIFEST", completed, "/", len(manifest), "ran_now=", ran_now, "skipped=", skipped, "infra_or_client_failures=", crashed)
print_running_summary()



CELL13_EXPLICIT_RESUME_MODE: completed exact manifest rows will be skipped; missing/incomplete rows will run.
RUN_ID: v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a
CURRENT_MANIFEST_ROWS: 400
EXISTING_DONE_JSON: 314
EXISTING_DRIFT_JSONL: 9
TOTAL_MANIFEST_EPISODE_RUNS: 400
MANIFEST_RUN_ORDER: v40 Cell 12 builds suite -> task -> episode -> mode; reference first, then ALL/R0/R1/R2/R3 + CUM_00_15/CUM_00_23
✅ SKIP_DONE 1/400 v40_linear130_split_ref5ep_windows_cumulative__v40_w8a16_reference__suite-libero_10__task-00__ep-00 result=SUCCESS exact_config_match=b28df7b44b27f76f
✅ SKIP_DONE 2/400 v40_linear130_split_ref5ep_windows_cumulative__v40_w8a8_ALL_00_31__suite-libero_10__task-00__ep-00 result=SUCCESS exact_config_match=b28df7b44b27f76f
✅ SKIP_DONE 3/400 v40_linear130_split_ref5ep_windows_cumulative__v40_w8a8_R0_00_07__suite-libero_10__task-00__ep-00 result=SUCCESS exact_config_match=b28df7b44b

In [23]:

# CELL 13A — PARTIAL PROGRESS SUMMARY (safe before the manifest finishes)
# Use this when Cell 13 was interrupted or you want to inspect current progress.
# Unlike 13B/14H, this does NOT require the full manifest to be complete.
# It prints: current per-mode success, clean paired damage, task difficulty, and success-margin/quickness vs reference.

import json
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 260)

print("=" * 120)
print("CELL 13A — PARTIAL PROGRESS SUMMARY / SAFE MID-RUN ANALYSIS")
print("=" * 120)

REFERENCE_MODE = globals().get("REFERENCE_MODE", "v40_w8a16_reference")
MODE_MEANING = {
    "v40_w8a16_reference": "1 reference: W8A16 / no A8 window",
    "v40_w8a8_ALL_00_31": "2 ALL_00_31: A8 on flow steps 0-31",
    "v40_w8a8_R0_00_07": "3 R0_00_07: A8 only on steps 0-7",
    "v40_w8a8_R1_08_15": "4 R1_08_15: A8 only on steps 8-15",
    "v40_w8a8_R2_16_23": "5 R2_16_23: A8 only on steps 16-23",
    "v40_w8a8_R3_24_31": "6 R3_24_31: A8 only on steps 24-31",
    "v40_w8a8_CUM_00_15": "7 CUM_00_15: A8 on steps 0-15",
    "v40_w8a8_CUM_00_23": "8 CUM_00_23: A8 on steps 0-23",
}
MODE_ORDER = list(MODE_MEANING.keys())

if "manifest" not in globals():
    manifest_path = Path(SUMMARIES) / "manifest_v40_linear130_split_ref5ep_windows_cumulative.json"
    if not manifest_path.exists():
        manifest_path = Path(SUMMARIES) / "manifest.json"
    if not manifest_path.exists():
        raise RuntimeError(f"Cannot find manifest under {SUMMARIES}")
    manifest = json.loads(manifest_path.read_text())

manifest_df = pd.DataFrame(manifest)
manifest_stems = set(manifest_df["result_stem"].astype(str))
expected_by_mode = manifest_df.groupby("ablation_mode").size().rename("expected_runs").reset_index()
expected_total = len(manifest_df)

records = []
extras = []
for p in sorted(Path(RESULTS).glob("*/*.done.json")):
    try:
        d = json.loads(p.read_text())
        stem = str(d.get("result_stem", ""))
        if stem not in manifest_stems:
            extras.append(str(p))
            continue
        d["done_json"] = str(p)
        records.append(d)
    except Exception as exc:
        print("READ_WARN", p, exc)

if not records:
    print("NO_DONE_RECORDS_YET under", RESULTS)
else:
    ep = pd.DataFrame(records)
    # Remove exact duplicate done records safely; keep last file if same manifest stem somehow exists twice.
    if "result_stem" in ep.columns:
        dup_stem = ep[ep.duplicated("result_stem", keep=False)].sort_values("result_stem")
        if len(dup_stem):
            print("WARNING_DUPLICATE_RESULT_STEMS_FOUND_KEEPING_LAST:", len(dup_stem))
            print(dup_stem[["result_stem", "ablation_mode", "suite", "task_id", "episode_id", "done_json"]].head(40).to_string(index=False))
        ep = ep.drop_duplicates("result_stem", keep="last")

    for c in ["success", "task_fail", "server_crash", "timeout_or_missing", "client_exception"]:
        if c not in ep.columns:
            ep[c] = False
        ep[c] = ep[c].fillna(False).astype(bool)
    for c in ["task_id", "episode_id", "env_steps", "outer_steps", "episode_seed"]:
        if c in ep.columns:
            ep[c] = pd.to_numeric(ep[c], errors="coerce")

    print("manifest_expected_rows:", expected_total)
    print("current_manifest_matched_done_rows:", len(ep))
    print("extra_nonmanifest_done_files_ignored:", len(extras))
    print("progress_pct:", round(100 * len(ep) / max(1, expected_total), 2), "%")

    print("\nMODE MAP")
    for m in MODE_ORDER:
        print(f"  {MODE_MEANING[m]}")

    # 1) Current success summary.
    summary = ep.groupby("ablation_mode").agg(
        runs=("success", "count"),
        success=("success", "sum"),
        task_fail=("task_fail", "sum"),
        crash=("server_crash", "sum"),
        timeout=("timeout_or_missing", "sum"),
        client_exception=("client_exception", "sum"),
    ).reset_index()
    summary = expected_by_mode.merge(summary, on="ablation_mode", how="left").fillna({"runs":0,"success":0,"task_fail":0,"crash":0,"timeout":0,"client_exception":0})
    for c in ["runs", "success", "task_fail", "crash", "timeout", "client_exception"]:
        summary[c] = summary[c].astype(int)
    summary["success_rate_so_far"] = summary["success"] / summary["runs"].replace(0, np.nan)
    summary["meaning"] = summary["ablation_mode"].map(MODE_MEANING).fillna(summary["ablation_mode"])
    summary = summary.sort_values("ablation_mode", key=lambda s: s.map({m:i for i,m in enumerate(MODE_ORDER)}).fillna(999))
    print("\nPARTIAL_SUCCESS_SUMMARY_BY_MODE")
    print(summary[["meaning", "expected_runs", "runs", "success", "task_fail", "crash", "timeout", "client_exception", "success_rate_so_far"]].to_string(index=False))

    # 2) Completed pair coverage and paired reference-vs-quant failures.
    key_cols = ["suite", "task_id", "episode_id"]
    pivot_success = ep.pivot_table(index=key_cols, columns="ablation_mode", values="success", aggfunc="first")
    completed_groups = pivot_success.notna().sum(axis=1).value_counts().sort_index()
    print("\nCOMPLETED_MODE_COUNT_PER_TASK_EP_GROUP")
    print(completed_groups.to_string())

    paired_rows = []
    if REFERENCE_MODE in pivot_success.columns:
        for mode in [m for m in MODE_ORDER if m != REFERENCE_MODE and m in pivot_success.columns]:
            sub = pivot_success[[REFERENCE_MODE, mode]].dropna()
            if len(sub) == 0:
                continue
            ref = sub[REFERENCE_MODE].astype(bool)
            q = sub[mode].astype(bool)
            paired_rows.append({
                "mode": mode,
                "meaning": MODE_MEANING.get(mode, mode),
                "paired_cases": len(sub),
                "ref_success_quant_fail": int(((ref == True) & (q == False)).sum()),
                "ref_fail_quant_fail": int(((ref == False) & (q == False)).sum()),
                "ref_fail_quant_success": int(((ref == False) & (q == True)).sum()),
                "same_outcome": int((ref == q).sum()),
            })
    paired_df = pd.DataFrame(paired_rows)
    if len(paired_df):
        print("\nPARTIAL_PAIRED_OUTCOME_VS_REFERENCE")
        print(paired_df.sort_values(["ref_success_quant_fail", "paired_cases"], ascending=[False, False]).to_string(index=False))
    else:
        print("\nPARTIAL_PAIRED_OUTCOME_VS_REFERENCE: not enough paired records yet")

    # 3) Task difficulty so far.
    task_summary = ep.groupby(["suite", "task_id"]).agg(
        runs=("success", "count"),
        success=("success", "sum"),
        fail=("success", lambda x: int((~x.astype(bool)).sum())),
    ).reset_index()
    task_summary["success_rate"] = task_summary["success"] / task_summary["runs"].clip(lower=1)
    print("\nTASK_DIFFICULTY_SO_FAR_WORST_FIRST")
    print(task_summary.sort_values(["fail", "success_rate"], ascending=[False, True]).to_string(index=False))

    # 4) Success margin / quicker success: compare env_steps and outer_steps to reference on same task/episode.
    margin_rows = []
    needed_cols = {"env_steps", "outer_steps"}
    if needed_cols.issubset(ep.columns):
        ref_ep = ep[ep["ablation_mode"] == REFERENCE_MODE][key_cols + ["success", "env_steps", "outer_steps"]].rename(columns={"success":"ref_success", "env_steps":"ref_env_steps", "outer_steps":"ref_outer_steps"})
        for mode in [m for m in MODE_ORDER if m != REFERENCE_MODE]:
            qep = ep[ep["ablation_mode"] == mode][key_cols + ["success", "env_steps", "outer_steps"]].rename(columns={"success":"q_success", "env_steps":"q_env_steps", "outer_steps":"q_outer_steps"})
            pair = ref_ep.merge(qep, on=key_cols, how="inner")
            if len(pair) == 0:
                continue
            both_success = pair[pair["ref_success"].astype(bool) & pair["q_success"].astype(bool)].copy()
            if len(both_success):
                both_success["delta_env_steps"] = both_success["q_env_steps"] - both_success["ref_env_steps"]
                both_success["delta_outer_steps"] = both_success["q_outer_steps"] - both_success["ref_outer_steps"]
                margin_rows.append({
                    "mode": mode,
                    "meaning": MODE_MEANING.get(mode, mode),
                    "both_success_pairs": len(both_success),
                    "median_delta_env_steps": float(both_success["delta_env_steps"].median()),
                    "mean_delta_env_steps": float(both_success["delta_env_steps"].mean()),
                    "faster_env_count": int((both_success["delta_env_steps"] < 0).sum()),
                    "slower_env_count": int((both_success["delta_env_steps"] > 0).sum()),
                    "same_env_count": int((both_success["delta_env_steps"] == 0).sum()),
                    "median_delta_outer_steps": float(both_success["delta_outer_steps"].median()),
                    "mean_delta_outer_steps": float(both_success["delta_outer_steps"].mean()),
                    "faster_outer_count": int((both_success["delta_outer_steps"] < 0).sum()),
                    "slower_outer_count": int((both_success["delta_outer_steps"] > 0).sum()),
                })
        margin_df = pd.DataFrame(margin_rows)
        if len(margin_df):
            print("\nPARTIAL_SUCCESS_MARGIN_VS_REFERENCE_BOTH_SUCCESS_ONLY")
            print("Negative delta means quant mode achieved success in fewer steps than reference on the same suite/task/episode.")
            print(margin_df.sort_values(["median_delta_env_steps", "mean_delta_env_steps"]).to_string(index=False))
    else:
        print("\nSUCCESS_MARGIN: env_steps/outer_steps not present in done JSON records")

    # 5) Drift summaries from done JSON, if available.
    def nested_get(d, path, default=np.nan):
        cur = d
        for k in path:
            if isinstance(cur, dict) and k in cur:
                cur = cur[k]
            else:
                return default
        return cur

    drift_rows = []
    for d in records:
        mode = d.get("ablation_mode")
        if mode == REFERENCE_MODE:
            continue
        ratios = d.get("drift_ratio_vs_w8a16_reference", {}) or {}
        metrics = d.get("metrics", {}) or {}
        # Accept several possible schemas.
        row = {
            "mode": mode,
            "meaning": MODE_MEANING.get(mode, mode),
            "suite": d.get("suite"),
            "task_id": d.get("task_id"),
            "episode_id": d.get("episode_id"),
            "success": bool(d.get("success", False)),
            "final_rms_mean_ratio": ratios.get("final_rms_mean_ratio_vs_ref", ratios.get("final_rms_mean_ratio", np.nan)),
            "vel_rms_max_ratio": ratios.get("vel_rms_max_ratio_vs_ref", ratios.get("vel_rms_max_ratio", np.nan)),
            "endpoint_rms_max_ratio": ratios.get("endpoint_rms_max_ratio_vs_ref", ratios.get("endpoint_rms_max_ratio", np.nan)),
            "state_after_rms_max_ratio": ratios.get("state_after_rms_max_ratio_vs_ref", ratios.get("state_after_rms_max_ratio", np.nan)),
            "final_rms_mean": d.get("final_rms_mean", metrics.get("final_rms_mean", np.nan)),
            "vel_rms_max": d.get("vel_rms_max", metrics.get("vel_rms_max", np.nan)),
            "endpoint_rms_max": d.get("endpoint_rms_max", metrics.get("endpoint_rms_max", np.nan)),
            "state_after_rms_max": d.get("state_after_rms_max", metrics.get("state_after_rms_max", np.nan)),
            "grip_mismatch_total": d.get("grip_mismatch_total", metrics.get("grip_mismatch_total", np.nan)),
        }
        drift_rows.append(row)
    drift = pd.DataFrame(drift_rows)
    if len(drift):
        for c in ["final_rms_mean_ratio", "vel_rms_max_ratio", "endpoint_rms_max_ratio", "state_after_rms_max_ratio", "final_rms_mean", "vel_rms_max", "endpoint_rms_max", "state_after_rms_max", "grip_mismatch_total"]:
            if c in drift.columns:
                drift[c] = pd.to_numeric(drift[c], errors="coerce")
        available_ratio_cols = [c for c in ["final_rms_mean_ratio", "vel_rms_max_ratio", "endpoint_rms_max_ratio", "state_after_rms_max_ratio"] if drift[c].notna().any()]
        available_raw_cols = [c for c in ["final_rms_mean", "vel_rms_max", "endpoint_rms_max", "state_after_rms_max", "grip_mismatch_total"] if drift[c].notna().any()]
        if available_ratio_cols or available_raw_cols:
            cols = available_ratio_cols if available_ratio_cols else available_raw_cols
            agg = drift.groupby(["mode", "meaning"])[cols].agg(["count", "median", "mean", lambda x: np.nanpercentile(x.dropna(), 95) if x.dropna().size else np.nan, "max"])
            # Rename lambda level for readability.
            agg.columns = [f"{a}_{'p95' if b == '<lambda_0>' else b}" for a,b in agg.columns]
            print("\nPARTIAL_DRIFT_SUMMARY_BY_MODE")
            print(agg.reset_index().sort_values("mode", key=lambda s: s.map({m:i for i,m in enumerate(MODE_ORDER)}).fillna(999)).to_string(index=False))
        else:
            print("\nDRIFT_SUMMARY: drift metrics not present in done JSON; run Cell 14C later to use JSONL request records / full parser.")

    # Save partial summaries for quick recovery.
    Path(SUMMARIES).mkdir(parents=True, exist_ok=True)
    summary.to_csv(Path(SUMMARIES) / "partial_progress_success_by_mode.csv", index=False)
    if len(paired_df): paired_df.to_csv(Path(SUMMARIES) / "partial_progress_paired_outcome_vs_reference.csv", index=False)
    if 'margin_df' in locals() and len(margin_df): margin_df.to_csv(Path(SUMMARIES) / "partial_progress_success_margin_vs_reference.csv", index=False)
    task_summary.to_csv(Path(SUMMARIES) / "partial_progress_task_difficulty.csv", index=False)
    print("\nSAVED_PARTIAL_PROGRESS_CSVS_TO:", SUMMARIES)

print("\nNOTE: 13A is for mid-run inspection only. Final claims still require 13B/13C + 14B-14H after the manifest finishes.")


CELL 13A — PARTIAL PROGRESS SUMMARY / SAFE MID-RUN ANALYSIS
manifest_expected_rows: 400
current_manifest_matched_done_rows: 400
extra_nonmanifest_done_files_ignored: 0
progress_pct: 100.0 %

MODE MAP
  1 reference: W8A16 / no A8 window
  2 ALL_00_31: A8 on flow steps 0-31
  3 R0_00_07: A8 only on steps 0-7
  4 R1_08_15: A8 only on steps 8-15
  5 R2_16_23: A8 only on steps 16-23
  6 R3_24_31: A8 only on steps 24-31
  7 CUM_00_15: A8 on steps 0-15
  8 CUM_00_23: A8 on steps 0-23

PARTIAL_SUCCESS_SUMMARY_BY_MODE
                           meaning  expected_runs  runs  success  task_fail  crash  timeout  client_exception  success_rate_so_far
 1 reference: W8A16 / no A8 window             50    50       44          6      0        0                 0                 0.88
2 ALL_00_31: A8 on flow steps 0-31             50    50       48          2      0        0                 0                 0.96
  3 R0_00_07: A8 only on steps 0-7             50    50       45          5      0        0 

In [24]:
# CELL 13B — Post-run manifest integrity check: exact manifest rows, balanced per mode, no stale-count leakage
# Run this immediately after Cell 13 finishes and before Cell 14 analysis.
import json, time
from pathlib import Path
import pandas as pd

print("=" * 110)
print("CELL 13B POST-RUN — MANIFEST-ONLY RESULT INTEGRITY CHECK")
print("=" * 110)

if "manifest" not in globals():
    manifest_path = Path(SUMMARIES) / "manifest_v40_linear130_split_ref5ep_windows_cumulative.json"
    if not manifest_path.exists():
        manifest_path = Path(SUMMARIES) / "manifest.json"
    if not manifest_path.exists():
        raise RuntimeError(f"POST_RUN_ABORT cannot find manifest under {SUMMARIES}")
    manifest = json.loads(manifest_path.read_text())

manifest_df = pd.DataFrame(manifest)
manifest_stems = {str(x.get("result_stem", "")) for x in manifest}
if len(manifest_stems) != len(manifest):
    raise RuntimeError("POST_RUN_ABORT manifest has duplicate result_stem values")

records = []
extras = []
dups = {}
seen = {}
for p in sorted(Path(RESULTS).glob("*/*.done.json")):
    d = json.loads(p.read_text())
    stem = str(d.get("result_stem", ""))
    d["done_json"] = str(p)
    if stem not in manifest_stems:
        extras.append(d)
        continue
    if stem in seen:
        dups.setdefault(stem, [seen[stem]]).append(d)
    else:
        seen[stem] = d
    records.append(d)

matched_unique = list(seen.values())
df = pd.DataFrame(matched_unique)

print("manifest_rows:", len(manifest))
print("all_done_json_files:", len(list(Path(RESULTS).glob("*/*.done.json"))))
print("matched_manifest_done_unique:", len(df))
print("extra_nonmanifest_done_files:", len(extras))
print("duplicate_manifest_done_stems:", len(dups))

if extras:
    print("\nEXTRA NON-MANIFEST DONE FILES — first 50")
    for d in extras[:50]:
        print(d.get("ablation_mode"), d.get("suite"), d.get("task_id"), d.get("episode_id"), d.get("done_json"))

if dups:
    print("\nDUPLICATE MANIFEST DONE STEMS — first 20")
    for stem, items in list(dups.items())[:20]:
        print(stem)
        for it in items:
            print("  ", it.get("done_json"))

if len(df) != len(manifest):
    missing = sorted(manifest_stems - set(seen.keys()))
    print("\nMISSING MANIFEST DONE STEMS — first 50")
    for s in missing[:50]:
        print(s)
    raise RuntimeError("POST_RUN_ABORT matched unique done rows do not equal manifest rows")

for c in ["success", "task_fail", "server_crash", "timeout_or_missing", "client_exception"]:
    if c not in df.columns:
        df[c] = False
    df[c] = df[c].fillna(False).astype(bool)

# Verify done JSON seeds match manifest seeds exactly.
manifest_seed_by_stem = {str(x.get("result_stem", "")): int(x.get("episode_seed", -999999)) for x in manifest}
seed_mismatches = []
for _, row in df.iterrows():
    stem = str(row.get("result_stem", ""))
    expected_seed = manifest_seed_by_stem.get(stem)
    got_seed = int(row.get("episode_seed", row.get("seed", -999999)))
    if expected_seed != got_seed:
        seed_mismatches.append((stem, expected_seed, got_seed))
if seed_mismatches:
    print("\nSEED_MISMATCHES — first 20")
    for x in seed_mismatches[:20]:
        print(x)
    raise RuntimeError("POST_RUN_ABORT done JSON episode_seed does not match manifest episode_seed")

seed_pair = df.groupby(["suite", "task_id", "episode_id"])["episode_seed"].nunique().reset_index(name="n_unique_seeds")
bad_seed_pair = seed_pair[seed_pair["n_unique_seeds"] != 1]
if len(bad_seed_pair):
    print(bad_seed_pair.to_string(index=False))
    raise RuntimeError("POST_RUN_ABORT fixed episode seed not shared across modes in completed rows")
print("OK_POSTRUN_FIXED_EPISODE_SEEDS_SHARED_ACROSS_MODES")

summary = df.groupby("ablation_mode").agg(
    runs=("success", "count"),
    success=("success", "sum"),
    task_fail=("task_fail", "sum"),
    crash=("server_crash", "sum"),
    timeout=("timeout_or_missing", "sum"),
    client_exception=("client_exception", "sum"),
).reset_index()
summary["success_rate"] = summary["success"] / summary["runs"].clip(lower=1)

print("\nMANIFEST_ONLY_SUMMARY_BY_MODE")
print(summary.to_string(index=False))

expected_by_mode = pd.DataFrame(manifest).groupby("ablation_mode").size().rename("expected_runs").reset_index()
count_check = expected_by_mode.merge(summary[["ablation_mode", "runs"]], on="ablation_mode", how="left")
count_check["runs"] = count_check["runs"].fillna(0).astype(int)
print("\nPOSTRUN_EXPECTED_VS_ACTUAL_BY_MODE")
print(count_check.to_string(index=False))

if len(df) != len(manifest):
    raise RuntimeError(f"POST_RUN_ABORT expected {len(manifest)} manifest-matched done rows, got {len(df)}")
bad_counts = count_check[count_check["expected_runs"] != count_check["runs"]]
if len(bad_counts):
    raise RuntimeError("POST_RUN_ABORT mode counts do not match manifest expected counts")
if extras:
    raise RuntimeError("POST_RUN_ABORT extra non-manifest done files exist in RESULTS; folder is dirty")
if dups:
    raise RuntimeError("POST_RUN_ABORT duplicate done files exist for manifest stems")

out_csv = Path(SUMMARIES) / "postrun_manifest_only_summary_by_mode.csv"
out_json = Path(SUMMARIES) / "postrun_manifest_integrity_check.json"
summary.to_csv(out_csv, index=False)
out_json.write_text(json.dumps({
    "time": time.strftime("%Y-%m-%d %H:%M:%S"),
    "run_id": str(RUN_ID),
    "manifest_rows": len(manifest),
    "matched_done_rows": len(df),
    "extra_nonmanifest_done_files": len(extras),
    "duplicate_manifest_done_stems": len(dups),
    "seed_mismatches": len(seed_mismatches),
    "expected_vs_actual_by_mode": count_check.to_dict(orient="records"),
    "summary": summary.to_dict(orient="records"),
}, indent=2))
print("\nSAVED:", out_csv)
print("SAVED:", out_json)
print(f"OK_POSTRUN_EXACTLY_{len(manifest)}_ROWS_AND_MANIFEST_MODE_COUNTS")






CELL 13B POST-RUN — MANIFEST-ONLY RESULT INTEGRITY CHECK
manifest_rows: 400
all_done_json_files: 400
matched_manifest_done_unique: 400
extra_nonmanifest_done_files: 0
duplicate_manifest_done_stems: 0
OK_POSTRUN_FIXED_EPISODE_SEEDS_SHARED_ACROSS_MODES

MANIFEST_ONLY_SUMMARY_BY_MODE
      ablation_mode  runs  success  task_fail  crash  timeout  client_exception  success_rate
v40_w8a16_reference    50       44          6      0        0                 0          0.88
 v40_w8a8_ALL_00_31    50       48          2      0        0                 0          0.96
 v40_w8a8_CUM_00_15    50       47          3      0        0                 0          0.94
 v40_w8a8_CUM_00_23    50       45          5      0        0                 0          0.90
  v40_w8a8_R0_00_07    50       45          5      0        0                 0          0.90
  v40_w8a8_R1_08_15    50       46          4      0        0                 0          0.92
  v40_w8a8_R2_16_23    50       42          8      0        

In [25]:
# CELL 13C — Reproducibility proof report: manifest, seeds, folder, script hashes
# Run after Cell 13B and before Cell 14/14B/14C.
# Purpose: prove the run is manifest-clean and paired by fixed seed.

import json, hashlib, os, re, time
from pathlib import Path
import pandas as pd

print("=" * 120)
print("CELL 13C — REPRODUCIBILITY PROOF REPORT")
print("=" * 120)
print("This cell is an audit gate. It should print OK lines or abort loudly.")

required = ["RUN_ID", "RUN_ROOT", "RESULTS", "SUMMARIES", "GENERATED", "SERVER_SCRIPT", "CLIENT_SCRIPT", "manifest", "MODE_SPECS"]
missing = [v for v in required if v not in globals()]
if missing:
    raise RuntimeError(f"REPRO_ABORT missing variables: {missing}")

run_root = Path(RUN_ROOT).resolve()
results = Path(RESULTS).resolve()
summaries = Path(SUMMARIES).resolve()
server_script = Path(SERVER_SCRIPT).resolve()
client_script = Path(CLIENT_SCRIPT).resolve()

manifest_df = pd.DataFrame(manifest).copy()
if manifest_df.empty:
    raise RuntimeError("REPRO_ABORT manifest is empty")

# Manifest structural proof.
manifest_stems = manifest_df["result_stem"].astype(str).tolist()
if len(manifest_stems) != len(set(manifest_stems)):
    dup = manifest_df[manifest_df.duplicated("result_stem", keep=False)][["result_stem", "ablation_mode", "suite", "task_id", "episode_id"]]
    print(dup.head(40).to_string(index=False))
    raise RuntimeError("REPRO_ABORT duplicate result_stem in manifest")

expected_total = len(MODE_SPECS) * len(manifest_df[["suite", "task_id", "episode_id"]].drop_duplicates())
print("RUN_ID:", RUN_ID)
print("RUN_ROOT:", run_root)
print("MANIFEST_ROWS:", len(manifest_df), "EXPECTED_FROM_GROUPS:", expected_total)
if len(manifest_df) != expected_total:
    raise RuntimeError("REPRO_ABORT manifest rows do not equal modes × unique task/episode groups")

mode_counts_manifest = manifest_df.groupby("ablation_mode").size().sort_index()
print("\nMANIFEST_MODE_COUNTS")
print(mode_counts_manifest.to_string())
if mode_counts_manifest.nunique() != 1:
    raise RuntimeError("REPRO_ABORT manifest mode counts are imbalanced")

# Fixed-seed pairing proof in manifest.
if "episode_seed" not in manifest_df.columns:
    raise RuntimeError("REPRO_ABORT manifest has no episode_seed column")
seed_by_group = manifest_df.groupby(["suite", "task_id", "episode_id"])["episode_seed"].nunique().reset_index(name="n_unique_seeds")
bad_seed_groups = seed_by_group[seed_by_group["n_unique_seeds"] != 1]
if len(bad_seed_groups):
    print(bad_seed_groups.head(40).to_string(index=False))
    raise RuntimeError("REPRO_ABORT manifest does not use one shared seed per suite/task/episode group")

modes_by_group = manifest_df.groupby(["suite", "task_id", "episode_id"])["ablation_mode"].nunique().reset_index(name="n_modes")
bad_mode_groups = modes_by_group[modes_by_group["n_modes"] != len(MODE_SPECS)]
if len(bad_mode_groups):
    print(bad_mode_groups.head(40).to_string(index=False))
    raise RuntimeError("REPRO_ABORT not every suite/task/episode group has all modes")

print("\nOK_MANIFEST_SEED_PAIRING: one episode_seed per suite/task/episode, shared by all modes")
print("UNIQUE_EPISODE_GROUPS:", len(seed_by_group))
print("MODES_PER_GROUP:", len(MODE_SPECS))

# Done JSON proof: exactly current manifest rows, no extras, no duplicate stems.
records = []
extra_files = []
seen = {}
duplicate_done_files = []
manifest_stem_set = set(manifest_stems)
for p in sorted(results.glob("*/*.done.json")):
    d = json.loads(p.read_text())
    stem = str(d.get("result_stem", ""))
    if stem not in manifest_stem_set:
        extra_files.append(str(p))
        continue
    if stem in seen:
        duplicate_done_files.append((stem, seen[stem], str(p)))
    seen[stem] = str(p)
    d["done_json"] = str(p)
    records.append(d)

if extra_files:
    print("\nEXTRA_DONE_FILES_NOT_IN_MANIFEST")
    for x in extra_files[:80]:
        print(x)
    raise RuntimeError("REPRO_ABORT extra done files found in RESULTS; fresh-folder/manifest hygiene failed")
if duplicate_done_files:
    print("\nDUPLICATE_DONE_FILES_FOR_SAME_STEM")
    for x in duplicate_done_files[:40]:
        print(x)
    raise RuntimeError("REPRO_ABORT duplicate done JSON files for same manifest stem")
if len(records) != len(manifest_df):
    missing = sorted(manifest_stem_set - set(seen))
    print("MISSING_DONE_STEMS:", len(missing))
    for x in missing[:80]:
        print(x)
    raise RuntimeError("REPRO_ABORT done rows do not match manifest rows exactly")

df_done = pd.DataFrame(records)
mode_counts_done = df_done.groupby("ablation_mode").size().sort_index()
print("\nDONE_MODE_COUNTS")
print(mode_counts_done.to_string())
if not mode_counts_done.equals(mode_counts_manifest):
    raise RuntimeError("REPRO_ABORT done mode counts do not equal manifest mode counts")

# Done seed matches manifest seed exactly.
manifest_seed = manifest_df.set_index("result_stem")["episode_seed"].astype(int).to_dict()
seed_mismatch = []
for _, row in df_done.iterrows():
    stem = str(row["result_stem"])
    got = int(row.get("episode_seed", row.get("seed", -999999)))
    exp = int(manifest_seed[stem])
    if got != exp:
        seed_mismatch.append((stem, exp, got))
if seed_mismatch:
    print("\nSEED_MISMATCHES")
    for x in seed_mismatch[:80]:
        print(x)
    raise RuntimeError("REPRO_ABORT done JSON episode_seed does not match manifest")

seed_done_group = df_done.groupby(["suite", "task_id", "episode_id"])["episode_seed"].nunique().reset_index(name="n_unique_seeds")
bad_done_seed = seed_done_group[seed_done_group["n_unique_seeds"] != 1]
if len(bad_done_seed):
    print(bad_done_seed.head(40).to_string(index=False))
    raise RuntimeError("REPRO_ABORT completed rows do not share one seed per suite/task/episode")
print("\nOK_DONE_SEEDS_MATCH_MANIFEST")

# Generated script hash/proof.
def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

server_sha = sha256_file(server_script) if server_script.exists() else None
client_sha = sha256_file(client_script) if client_script.exists() else None
print("\nGENERATED_SCRIPT_HASHES")
print("SERVER_SCRIPT:", server_script)
print("SERVER_SHA256:", server_sha)
print("CLIENT_SCRIPT:", client_script)
print("CLIENT_SHA256:", client_sha)

client_src = client_script.read_text(errors="ignore") if client_script.exists() else ""
seed_patterns = {
    "reads_FLOWA8_SEED": "FLOWA8_SEED" in client_src,
    "sets_env_seed": "env.seed" in client_src,
    "sets_numpy_seed": "np.random.seed" in client_src,
    "sets_python_random_seed": "random.seed" in client_src,
    "uses_fixed_init_state": "set_init_state" in client_src,
    "uses_episode_id": "EPISODE_ID" in client_src,
}
print("\nCLIENT_SEED_CODE_AUDIT")
for k, v in seed_patterns.items():
    print(f"{k}: {v}")
if not all(seed_patterns.values()):
    raise RuntimeError("REPRO_ABORT generated client script lacks one or more seed/init-state proof hooks")

# Store proof report for the paper appendix / audit trail.
proof = {
    "created_at_unix": time.time(),
    "run_id": str(RUN_ID),
    "run_root": str(run_root),
    "manifest_rows": int(len(manifest_df)),
    "unique_episode_groups": int(len(seed_by_group)),
    "modes_per_group": int(len(MODE_SPECS)),
    "mode_counts_manifest": {str(k): int(v) for k, v in mode_counts_manifest.to_dict().items()},
    "mode_counts_done": {str(k): int(v) for k, v in mode_counts_done.to_dict().items()},
    "extra_done_files": len(extra_files),
    "duplicate_done_files": len(duplicate_done_files),
    "seed_mismatches": len(seed_mismatch),
    "server_script": str(server_script),
    "server_sha256": server_sha,
    "client_script": str(client_script),
    "client_sha256": client_sha,
    "client_seed_code_audit": seed_patterns,
    "experiment_config_signature": str(globals().get("EXPERIMENT_CONFIG_SIGNATURE", "")),
}
out = summaries / "reproducibility_proof_report.json"
out.write_text(json.dumps(proof, indent=2))
print("\nSAVED_REPRO_PROOF:", out)
print("\nOK_REPRODUCIBILITY_PROOF_PASSED")


CELL 13C — REPRODUCIBILITY PROOF REPORT
This cell is an audit gate. It should print OK lines or abort loudly.
RUN_ID: v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a
RUN_ROOT: /content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a
MANIFEST_ROWS: 400 EXPECTED_FROM_GROUPS: 400

MANIFEST_MODE_COUNTS
ablation_mode
v40_w8a16_reference    50
v40_w8a8_ALL_00_31     50
v40_w8a8_CUM_00_15     50
v40_w8a8_CUM_00_23     50
v40_w8a8_R0_00_07      50
v40_w8a8_R1_08_15      50
v40_w8a8_R2_16_23      50
v40_w8a8_R3_24_31      50

OK_MANIFEST_SEED_PAIRING: one episode_seed per suite/task/episode, shared by all modes
UNIQUE_EPISODE_GROUPS: 50
MODES_PER_GROUP: 8

DONE_MODE_COUNTS
ablation_mode
v40_w8a16_reference    50
v40_w8a8_ALL_00_31     50
v40_w8a8_CUM_00_15

In [26]:

# CELL 14 — v40 clean linear-split comparison table + detailed risk analysis summaries
#
# This cell is safe to rerun any time. It reads saved .done.json + JSONL drift metrics.
#
# Main question:
#   With W8 weights fixed, how much do W8A8 flow-loop stress modes drift relative to same-episode v40 W8A16 reference on libero_10?
#
# Main metrics:
#   success_rate(mode) = success / eval_runs
#   E_action(mode)    = mean_i final_action_7d_rms_i^2, using executed prefix action[:7]
#   E_grip(mode)      = mean_i 1[gripper decision differs from v40 W8A16 reference on executed prefix]
#
# Important:
#   No old ad-hoc FlowA8 risk_score is used as a decision rule.
#   The final decision comes from success_rate + E_action + E_grip + suite breakdown.
import json
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 220)

# -------------------------
# Load episode-level results
# -------------------------
# Load done JSON records, strictly restricted to the current manifest rows.
MANIFEST_JSON = SUMMARIES / "manifest_v40_linear130_split_ref5ep_windows_cumulative.json"
if "manifest" not in globals():
    if MANIFEST_JSON.exists():
        manifest = json.loads(MANIFEST_JSON.read_text())
    else:
        raise RuntimeError(f"Missing manifest for strict summary filtering: {MANIFEST_JSON}. Run Cell 12 first.")

MANIFEST_STEMS = {str(x.get("result_stem", "")) for x in manifest}
MANIFEST_KEYS = {
    (
        str(x.get("ablation_mode")),
        str(x.get("suite")),
        int(x.get("task_id")),
        int(x.get("episode_id")),
        tuple(int(v) for v in x.get("a8_steps", [])),
    )
    for x in manifest
}

records = []
extra_done = []
duplicate_stems = {}
for p in sorted(RESULTS.glob("*/*.done.json")):
    try:
        d = json.loads(p.read_text())
        stem = str(d.get("result_stem", ""))
        if stem not in MANIFEST_STEMS:
            extra_done.append(str(p))
            continue
        if stem in duplicate_stems:
            print("DUPLICATE_DONE_STEM_WARN", stem, "first=", duplicate_stems[stem], "second=", p)
        duplicate_stems[stem] = str(p)
        d["done_json"] = str(p)
        records.append(d)
    except Exception as exc:
        print("READ_WARN", p, exc)

df = pd.DataFrame(records)
print("MANIFEST_EXPECTED_ROWS:", len(MANIFEST_STEMS))
print("MANIFEST_MATCHED_DONE_ROWS:", len(df))
print("EXTRA_DONE_JSON_IGNORED:", len(extra_done))
if extra_done:
    print("EXTRA_DONE_JSON_IGNORED_FIRST_20:")
    for _p in extra_done[:20]:
        print("  ", _p)

print("=" * 110)
print("EPISODE RESULT FILES")
print("=" * 110)
print("DONE_JSON_RECORDS:", len(df))
if len(df) == 0:
    raise RuntimeError("No done JSON records found yet. Run CELL 13 first.")

for c in ["success", "task_fail", "server_crash", "timeout_or_missing", "client_exception", "egl_cleanup_warning"]:
    if c not in df.columns:
        df[c] = False
    df[c] = df[c].fillna(False).astype(bool)

# Load current v40 mode specs/config. No fallback to older experiments.
mode_specs = {}
mode_specs_path = SUMMARIES / "mode_specs_v40_linear130_split_ref5ep_windows_cumulative.json"
config_path = SUMMARIES / "experiment_config_v40_linear130_split_ref5ep_windows_cumulative.json"
if mode_specs_path.exists():
    mode_specs = json.loads(mode_specs_path.read_text())
else:
    raise RuntimeError(f"Missing current v40 mode specs: {mode_specs_path}. Run Cell 12 first.")
experiment_config_signature = None
if config_path.exists():
    experiment_config_signature = json.loads(config_path.read_text()).get("experiment_config_signature")
else:
    print("CONFIG_JSON_MISSING_WARN", config_path)

# Flatten per-episode ratios vs same-episode v40 W8A16 reference when Cell 13 saved them.
def _ratio_field(x, key):
    if isinstance(x, dict):
        v = x.get(key, np.nan)
        try:
            return float(v)
        except Exception:
            return np.nan
    return np.nan
if "drift_ratio_vs_w8a16_reference" in df.columns:
    for _k in [
        "final_rms_mean_ratio_vs_ref", "final_rms_max_ratio_vs_ref", "vel_rms_max_ratio_vs_ref",
        "endpoint_rms_max_ratio_vs_ref", "state_after_rms_max_ratio_vs_ref"
    ]:
        df[_k] = df["drift_ratio_vs_w8a16_reference"].apply(lambda x, k=_k: _ratio_field(x, k))
else:
    for _k in [
        "final_rms_mean_ratio_vs_ref", "final_rms_max_ratio_vs_ref", "vel_rms_max_ratio_vs_ref",
        "endpoint_rms_max_ratio_vs_ref", "state_after_rms_max_ratio_vs_ref"
    ]:
        df[_k] = np.nan

# Quant-config guard: summarize only files from this v40 quantization/mode config.
# Same exact mode/suite/task/episode rows that were skipped/resumed are included.
# Later-added suites/episodes with the same quant_config_signature are included too.
# Different layer scopes/mode names/quant settings are not included.
_allowed_modes = set(mode_specs.keys())
_before = len(df)
df = df[df["ablation_mode"].isin(_allowed_modes)].copy()
df = df[df["experiment_id"].eq("v40_linear130_split_ref5ep_windows_cumulative")].copy()
if experiment_config_signature:
    if "experiment_config_signature" not in df.columns:
        df = df.iloc[0:0].copy()
    else:
        df = df[df["experiment_config_signature"].eq(experiment_config_signature)].copy()
print("EXACT_CONFIG_FILTER:", len(df), "/", _before, "records kept for current v40 experiment/config", "signature=", experiment_config_signature)
if len(df) == 0:
    raise RuntimeError("No records match current v40 experiment/config. Run Cell 12/13 for this notebook or check RUN_ROOT.")

def _tested_range(mode):
    return mode_specs.get(mode, {}).get("tested_range", "")
def _a8_steps_str(mode):
    steps = mode_specs.get(mode, {}).get("a8_steps", None)
    if steps is None:
        return ""
    if len(steps) == 0:
        return "none"
    return f"{min(steps)}-{max(steps)}"

# Overall by mode.
by_mode = df.groupby("ablation_mode").agg(
    eval_runs=("success", "count"),
    success=("success", "sum"),
    task_fail=("task_fail", "sum"),
    server_crash=("server_crash", "sum"),
    timeout_or_missing=("timeout_or_missing", "sum"),
    client_exception=("client_exception", "sum"),
    egl_cleanup_warning=("egl_cleanup_warning", "sum"),
    mean_elapsed_sec=("elapsed_sec", "mean"),
).reset_index()
by_mode["failure"] = by_mode["eval_runs"] - by_mode["success"]
by_mode["success_rate"] = by_mode["success"] / by_mode["eval_runs"].clip(lower=1)
ratio_cols_for_mode = [
    "final_rms_mean_ratio_vs_ref", "final_rms_max_ratio_vs_ref", "vel_rms_max_ratio_vs_ref",
    "endpoint_rms_max_ratio_vs_ref", "state_after_rms_max_ratio_vs_ref"
]
ratio_mode = df.groupby("ablation_mode")[ratio_cols_for_mode].mean().reset_index()
by_mode = by_mode.merge(ratio_mode, on="ablation_mode", how="left")
by_mode["tested_range"] = by_mode["ablation_mode"].map(_tested_range)
by_mode["a8_steps"] = by_mode["ablation_mode"].map(_a8_steps_str)

# Suite breakdown.
by_suite = df.groupby(["ablation_mode", "suite"]).agg(
    eval_runs=("success", "count"),
    success=("success", "sum"),
    task_fail=("task_fail", "sum"),
    server_crash=("server_crash", "sum"),
    timeout_or_missing=("timeout_or_missing", "sum"),
    client_exception=("client_exception", "sum"),
    egl_cleanup_warning=("egl_cleanup_warning", "sum"),
    mean_elapsed_sec=("elapsed_sec", "mean"),
).reset_index()
by_suite["failure"] = by_suite["eval_runs"] - by_suite["success"]
by_suite["success_rate"] = by_suite["success"] / by_suite["eval_runs"].clip(lower=1)
by_suite["tested_range"] = by_suite["ablation_mode"].map(_tested_range)
by_suite["a8_steps"] = by_suite["ablation_mode"].map(_a8_steps_str)

# -------------------------
# Load request/step/site drift metrics
# -------------------------
req_records, step_rows, site_rows = [], [], []

def _drift_key(r):
    try:
        return (
            str(r.get("ablation_mode")),
            str(r.get("suite")),
            int(r.get("task_id")),
            int(r.get("episode_id")),
            tuple(int(v) for v in r.get("a8_steps", [])),
        )
    except Exception:
        return None

for p in sorted(DRIFT.glob("*_request_records.jsonl")):
    if p.name == "all_modes_request_records.jsonl":
        continue
    for line in p.read_text(errors="ignore").splitlines():
        if not line.strip():
            continue
        r = json.loads(line)
        if _drift_key(r) not in MANIFEST_KEYS:
            continue
        base = {k: v for k, v in r.items() if k not in ("step_records", "site_records")}
        req_records.append(base)
        for sr in r.get("step_records", []):
            row = dict(base); row.update(sr); step_rows.append(row)
        for qr in r.get("site_records", []):
            row = dict(base); row.update(qr); site_rows.append(row)
req = pd.DataFrame(req_records)
steps = pd.DataFrame(step_rows)
sites = pd.DataFrame(site_rows)

# Keep drift rows aligned with the current v40 modes/config where the fields are present.
for _name, _frame in [("req", req), ("steps", steps), ("sites", sites)]:
    if len(_frame) and "ablation_mode" in _frame.columns:
        _frame.drop(_frame[~_frame["ablation_mode"].isin(_allowed_modes)].index, inplace=True)
    if len(_frame) and "experiment_id" in _frame.columns:
        _frame.drop(_frame[~_frame["experiment_id"].eq("v40_linear130_split_ref5ep_windows_cumulative")].index, inplace=True)

print("\nDRIFT_REQUEST_RECORDS:", len(req))
print("DRIFT_STEP_RECORDS:", len(steps))
print("DRIFT_SITE_RECORDS:", len(sites))

print("\n" + "=" * 110)
print("FORMULAS / INTERPRETATION")
print("=" * 110)
print("W8 scope in v15 server: 98 LLM linears + all 32 action_head Linear layers = 130 W8 targets.")
print("LLM/VLM/context/state_encoder activations stay A16/BF16. v40 W8A16 reference has a8_steps=[] and is run first for each task/episode.")
print("Selected W8A8 modes quantize only window-controlled flow-loop sites; ratios are computed against the same task/episode v40 W8A16 reference.")
print("Metrics use executed prefix action[:7], not the full model chunk; model_horizon and exec_horizon are saved separately.")
print("E_action(mode) = mean_i final_action_7d_rms_i^2")
print("E_grip(mode)  = mean_i 1[gripper_threshold_mismatch_count_i > 0]")
print("Sensitive range = lower success_rate and/or larger E_action/E_grip when that range alone uses A8.")
print("No old ad-hoc FlowA8 risk_score is used as the decision rule.")

# -------------------------
# Main drift mode scores
# -------------------------
mode_score = pd.DataFrame()
if len(req):
    req.to_csv(SUMMARIES / "w8_a8_range_window_request_drift.csv", index=False)
    req["action_recon_mse"] = req["final_action_7d_rms"].astype(float) ** 2
    req["gripper_flip_any"] = req.get("gripper_threshold_mismatch_count", 0).astype(float) > 0

    mode_score = req.groupby("ablation_mode").agg(
        request_records=("action_recon_mse", "count"),
        E_action=("action_recon_mse", "mean"),
        E_action_median=("action_recon_mse", "median"),
        E_action_p95=("action_recon_mse", lambda x: float(np.percentile(x, 95))),
        final_rms_mean=("final_action_7d_rms", "mean"),
        final_rms_p95=("final_action_7d_rms", lambda x: float(np.percentile(x, 95))),
        final_rms_max=("final_action_7d_rms", "max"),
        final_max_abs_max=("final_action_7d_max_abs", "max"),
        E_grip=("gripper_flip_any", "mean"),
        gripper_flip_count=("gripper_flip_any", "sum"),
        gripper_boundary_count=("gripper_boundary_count", "sum") if "gripper_boundary_count" in req.columns else ("gripper_flip_any", "sum"),
        exec_horizon_min=("exec_horizon", "min") if "exec_horizon" in req.columns else ("action_recon_mse", "count"),
        exec_horizon_max=("exec_horizon", "max") if "exec_horizon" in req.columns else ("action_recon_mse", "count"),
        model_horizon_min=("model_horizon", "min") if "model_horizon" in req.columns else ("action_recon_mse", "count"),
        model_horizon_max=("model_horizon", "max") if "model_horizon" in req.columns else ("action_recon_mse", "count"),
    ).reset_index()
    mode_score["tested_range"] = mode_score["ablation_mode"].map(_tested_range)
    mode_score["a8_steps"] = mode_score["ablation_mode"].map(_a8_steps_str)

# -------------------------
# MAIN COMPARISON TABLE
# -------------------------
main = by_mode.copy()
if len(mode_score):
    keep = [
        "ablation_mode", "request_records", "E_action", "E_action_p95",
        "final_rms_mean", "final_rms_p95", "final_rms_max",
        "final_max_abs_max", "E_grip", "gripper_flip_count", "gripper_boundary_count",
        "final_rms_mean_ratio_vs_ref", "final_rms_max_ratio_vs_ref", "vel_rms_max_ratio_vs_ref", "endpoint_rms_max_ratio_vs_ref", "state_after_rms_max_ratio_vs_ref",
        "model_horizon_min", "model_horizon_max", "exec_horizon_min", "exec_horizon_max"
    ]
    keep = [c for c in keep if c in mode_score.columns]
    main = main.merge(mode_score[keep], on="ablation_mode", how="left")

# A simple table ordering: worst success first, then highest E_action / E_grip.
for col in ["E_action", "E_grip", "final_rms_p95"]:
    if col not in main.columns:
        main[col] = np.nan
main = main.sort_values(["success_rate", "E_action", "E_grip", "final_rms_p95"], ascending=[True, False, False, False])

main_cols = [
    "ablation_mode", "tested_range", "a8_steps",
    "eval_runs", "success", "success_rate", "task_fail", "server_crash", "timeout_or_missing", "client_exception",
    "request_records", "E_action", "E_action_p95", "final_rms_mean", "final_rms_p95",
    "E_grip", "gripper_flip_count", "gripper_boundary_count",
    "final_rms_mean_ratio_vs_ref", "final_rms_max_ratio_vs_ref", "vel_rms_max_ratio_vs_ref", "endpoint_rms_max_ratio_vs_ref", "state_after_rms_max_ratio_vs_ref",
    "model_horizon_min", "model_horizon_max", "exec_horizon_min", "exec_horizon_max",
    "mean_elapsed_sec",
]
main_cols = [c for c in main_cols if c in main.columns]
main_table = main[main_cols]
main_path = SUMMARIES / "MAIN_W8_A8_RANGE_WINDOW_COMPARISON.csv"
main_table.to_csv(main_path, index=False)

print("\n" + "=" * 110)
print("MAIN DECISION TABLE — W8 + A8 WINDOW COMPARISON")
print("=" * 110)
print(main_table.to_string(index=False))
print("SAVED:", main_path)

# Suite table emphasized separately.
suite_cols = [
    "ablation_mode", "tested_range", "suite", "eval_runs", "success", "success_rate",
    "task_fail", "server_crash", "timeout_or_missing", "client_exception", "egl_cleanup_warning", "mean_elapsed_sec"
]
suite_cols = [c for c in suite_cols if c in by_suite.columns]
suite_path = SUMMARIES / "MAIN_W8_A8_RANGE_WINDOW_BY_SUITE.csv"
by_suite[suite_cols].sort_values(["suite", "success_rate", "ablation_mode"], ascending=[True, True, True]).to_csv(suite_path, index=False)

print("\n" + "=" * 110)
print("MAIN BY-SUITE TABLE")
print("=" * 110)
print(by_suite[suite_cols].sort_values(["suite", "success_rate", "ablation_mode"], ascending=[True, True, True]).to_string(index=False))
print("SAVED:", suite_path)

# -------------------------
# Detailed request drift
# -------------------------
if len(req):
    metric_cols = [c for c in [
        "final_action_7d_rms", "final_action_7d_l2", "final_action_7d_max_abs",
        "xyz_rms", "rot_rms", "gripper_abs_mean",
        "gripper_threshold_mismatch_count", "gripper_boundary_count",
        "action_recon_mse", "gripper_flip_any"
    ] if c in req.columns]

    print("\n" + "=" * 110)
    print("REQUEST-LEVEL ACTION/GRIPPER DRIFT BY MODE")
    print("=" * 110)
    def p95_numeric(x):
        vals = pd.to_numeric(x, errors="coerce").dropna().astype(float)
        if len(vals) == 0:
            return np.nan
        return float(np.nanpercentile(vals, 95))
    p95_numeric.__name__ = "p95"

    req_safe = req.copy()
    for _c in metric_cols:
        if pd.api.types.is_bool_dtype(req_safe[_c]):
            req_safe[_c] = req_safe[_c].astype("int8")
        else:
            req_safe[_c] = pd.to_numeric(req_safe[_c], errors="coerce")

    agg_req = req_safe.groupby("ablation_mode")[metric_cols].agg(["mean", "median", p95_numeric, "max"])
    print(agg_req.to_string())
    agg_req.to_csv(SUMMARIES / "w8_a8_request_drift_by_mode_detailed.csv")

# -------------------------
# Flow-step and range summaries
# -------------------------
if len(steps):
    steps.to_csv(SUMMARIES / "w8_a8_range_window_flow_step_records.csv", index=False)

    eps = 1e-8
    if "qdrift_velocity_rms" in steps.columns and "ref_velocity_norm" in steps.columns:
        steps["rel_velocity_recon_mse"] = (steps["qdrift_velocity_rms"].astype(float) ** 2) / (steps["ref_velocity_norm"].astype(float) ** 2 + eps)
    if "qdrift_state_after_rms" in steps.columns:
        steps["state_recon_mse"] = steps["qdrift_state_after_rms"].astype(float) ** 2

    steps["range_id"] = (steps["flow_step"].astype(int) // 8).clip(lower=0, upper=3)
    steps["range_label"] = steps["range_id"].map({0:"R0_00_07", 1:"R1_08_15", 2:"R2_16_23", 3:"R3_24_31"})

    agg_kwargs = dict(
        n=("qdrift_velocity_rms", "count"),
        vel_rms_mean=("qdrift_velocity_rms", "mean"),
        vel_rms_p95=("qdrift_velocity_rms", lambda x: float(np.percentile(x, 95))),
        state_after_rms_mean=("qdrift_state_after_rms", "mean"),
        endpoint_pair_rms_mean=("flash_endpoint_pair_rms", "mean"),
        ref_velocity_norm_mean=("ref_velocity_norm", "mean"),
        ref_velocity_jump_mean=("ref_velocity_jump_rms", "mean"),
    )
    if "rel_velocity_recon_mse" in steps.columns:
        agg_kwargs["rel_velocity_recon_mse_mean"] = ("rel_velocity_recon_mse", "mean")
        agg_kwargs["rel_velocity_recon_mse_p95"] = ("rel_velocity_recon_mse", lambda x: float(np.percentile(x, 95)))

    step_agg = steps.groupby(["ablation_mode", "flow_step"]).agg(**agg_kwargs).reset_index()
    step_agg.to_csv(SUMMARIES / "w8_a8_range_window_flow_step_reconstruction_by_mode.csv", index=False)

    range_kwargs = dict(
        n=("qdrift_velocity_rms", "count"),
        vel_rms_mean=("qdrift_velocity_rms", "mean"),
        vel_rms_p95=("qdrift_velocity_rms", lambda x: float(np.percentile(x, 95))),
        state_after_rms_mean=("qdrift_state_after_rms", "mean"),
        endpoint_pair_rms_mean=("flash_endpoint_pair_rms", "mean"),
    )
    if "rel_velocity_recon_mse" in steps.columns:
        range_kwargs["rel_velocity_recon_mse_mean"] = ("rel_velocity_recon_mse", "mean")
        range_kwargs["rel_velocity_recon_mse_p95"] = ("rel_velocity_recon_mse", lambda x: float(np.percentile(x, 95)))
    range_agg = steps.groupby(["ablation_mode", "range_label"]).agg(**range_kwargs).reset_index()
    range_agg.to_csv(SUMMARIES / "w8_a8_four_range_flow_drift_summary.csv", index=False)

    print("\n" + "=" * 110)
    print("FOUR-RANGE FLOW-STEP DRIFT SUMMARY")
    print("=" * 110)
    print(range_agg.to_string(index=False))
    print("SAVED:", SUMMARIES / "w8_a8_four_range_flow_drift_summary.csv")

    print("\n" + "=" * 110)
    print("TOP FLOW STEPS BY MODE — highest relative velocity reconstruction error")
    print("=" * 110)
    sort_col = "rel_velocity_recon_mse_p95" if "rel_velocity_recon_mse_p95" in step_agg.columns else "vel_rms_p95"
    for mode in sorted(step_agg["ablation_mode"].unique()):
        print("\nMODE", mode)
        print(step_agg[step_agg["ablation_mode"] == mode].sort_values(sort_col, ascending=False).head(10).to_string(index=False))

# -------------------------
# Site/layer type summary
# -------------------------
if len(sites):
    sites.to_csv(SUMMARIES / "w8_a8_range_window_site_activation_error.csv", index=False)
    site_agg = sites.groupby(["ablation_mode", "site_type"]).agg(
        n=("qact_error_rms", "count"),
        qact_error_rms_mean=("qact_error_rms", "mean"),
        qact_error_rms_p95=("qact_error_rms", lambda x: float(np.percentile(x, 95))),
        qact_error_max_abs_max=("qact_error_max_abs", "max"),
    ).reset_index()
    site_agg.to_csv(SUMMARIES / "w8_a8_site_type_summary.csv", index=False)

    print("\n" + "=" * 110)
    print("SITE / LAYER-TYPE A8 QUANTIZATION ERROR")
    print("=" * 110)
    print(site_agg.to_string(index=False))
    print("SAVED:", SUMMARIES / "w8_a8_site_type_summary.csv")

# Save base tables.
by_mode.to_csv(SUMMARIES / "w8_a8_range_window_by_mode.csv", index=False)
by_suite.to_csv(SUMMARIES / "w8_a8_range_window_by_mode_suite.csv", index=False)
df.to_csv(SUMMARIES / "w8_a8_range_window_done_records.csv", index=False)

print("\n" + "=" * 110)
print("SAVED SUMMARY FILES")
print("=" * 110)
print("SUMMARY_DIR:", SUMMARIES)
for p in sorted(SUMMARIES.glob("*.csv")):
    print("CSV:", p.name)

print("\nINTERPRETATION RULE:")
print("- The range with lowest success and/or highest E_action/E_grip is the most quantization-sensitive A8 window.")
print("- This is end-to-end validation, not just calibration risk-score ranking.")
print("- Use the main comparison table first, then inspect suite/site/flow-step details.")


MANIFEST_EXPECTED_ROWS: 400
MANIFEST_MATCHED_DONE_ROWS: 400
EXTRA_DONE_JSON_IGNORED: 0
EPISODE RESULT FILES
DONE_JSON_RECORDS: 400
EXACT_CONFIG_FILTER: 400 / 400 records kept for current v40 experiment/config signature= b28df7b44b27f76f


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [28]:
# CELL 14B — Paired task/episode sensitivity analysis: separate task difficulty from quant-mode sensitivity
# Run after Cell 14 or after Cell 13B. Uses manifest-only done rows.
import json
from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 110)
print("CELL 14B — PAIRED TASK/EPISODE SENSITIVITY ANALYSIS — fixed seeds checked")
print("=" * 110)

if "manifest" not in globals():
    manifest_path = Path(SUMMARIES) / "manifest_v40_linear130_split_ref5ep_windows_cumulative.json"
    if not manifest_path.exists():
        manifest_path = Path(SUMMARIES) / "manifest.json"
    manifest = json.loads(manifest_path.read_text())
manifest_stems = {str(x.get("result_stem", "")) for x in manifest}

records = []
for p in sorted(Path(RESULTS).glob("*/*.done.json")):
    d = json.loads(p.read_text())
    if str(d.get("result_stem", "")) not in manifest_stems:
        continue
    records.append(d)

df = pd.DataFrame(records).drop_duplicates(subset=["result_stem"], keep="last")

if "episode_seed" in df.columns:
    seed_pair = df.groupby(["suite", "task_id", "episode_id"])["episode_seed"].nunique().reset_index(name="n_unique_seeds")
    bad_seed_pair = seed_pair[seed_pair["n_unique_seeds"] != 1]
    if len(bad_seed_pair):
        print(bad_seed_pair.to_string(index=False))
        raise RuntimeError("Sensitivity analysis abort: episode_seed is not shared across modes")
    print("OK_SENSITIVITY_FIXED_SEEDS_SHARED_ACROSS_MODES")
else:
    print("WARNING: episode_seed column missing; paired success analysis still works, but fixed-seed proof is unavailable")

if len(df) != len(manifest):
    raise RuntimeError(f"Expected {len(manifest)} manifest rows, found {len(df)} done rows")

for c in ["success", "task_fail", "server_crash", "timeout_or_missing", "client_exception"]:
    if c not in df.columns:
        df[c] = False
    df[c] = df[c].fillna(False).astype(bool)

df["task_id"] = df["task_id"].astype(int)
df["episode_id"] = df["episode_id"].astype(int)
df["success_i"] = df["success"].astype(int)

# 1) Which tasks are hard independent of mode?
task_summary = df.groupby(["suite", "task_id"]).agg(
    runs=("success", "count"),
    success=("success", "sum"),
    failures=("success", lambda s: int((~s.astype(bool)).sum())),
    task_fail=("task_fail", "sum"),
    crash=("server_crash", "sum"),
    timeout=("timeout_or_missing", "sum"),
    client_exception=("client_exception", "sum"),
).reset_index()
task_summary["success_rate"] = task_summary["success"] / task_summary["runs"].clip(lower=1)
task_summary = task_summary.sort_values(["failures", "task_id"], ascending=[False, True])
print("\nTASK DIFFICULTY SUMMARY — worst first")
print(task_summary.to_string(index=False))

# 2) Pair by same suite/task/episode across modes.
mode_pivot = df.pivot_table(
    index=["suite", "task_id", "episode_id"],
    columns="ablation_mode",
    values="success_i",
    aggfunc="max",
)
mode_pivot["n_modes_present"] = mode_pivot.drop(columns=[], errors="ignore").notna().sum(axis=1)
mode_cols = [c for c in mode_pivot.columns if c != "n_modes_present"]
mode_pivot["mode_fail_count"] = mode_pivot[mode_cols].apply(lambda r: int((r == 0).sum()), axis=1)
mode_pivot["all_modes_success"] = mode_pivot[mode_cols].apply(lambda r: bool((r == 1).all()), axis=1)
mode_pivot["any_mode_failure"] = mode_pivot[mode_cols].apply(lambda r: bool((r == 0).any()), axis=1)

paired_cases = mode_pivot.reset_index().sort_values(["mode_fail_count", "task_id", "episode_id"], ascending=[False, True, True])
print("\nPAIRED CASES WITH ANY FAILURE — same task/episode across modes")
print(paired_cases[paired_cases["any_mode_failure"]].to_string(index=False))

# 3) Against reference: only count a quant mode as sensitive if reference succeeded but quant mode failed on same suite/task/episode.
ref_mode = "v40_w8a16_reference"
if ref_mode in mode_cols:
    ref_succeeded = mode_pivot[ref_mode] == 1
    rescue_or_noise = []
    sensitivity_rows = []
    for m in mode_cols:
        if m == ref_mode:
            continue
        quant_failed_when_ref_success = ((mode_pivot[m] == 0) & ref_succeeded).sum()
        quant_success_when_ref_failed = ((mode_pivot[m] == 1) & (mode_pivot[ref_mode] == 0)).sum()
        both_failed = ((mode_pivot[m] == 0) & (mode_pivot[ref_mode] == 0)).sum()
        both_success = ((mode_pivot[m] == 1) & (mode_pivot[ref_mode] == 1)).sum()
        sensitivity_rows.append({
            "mode": m,
            "quant_fail_ref_success_sensitive_cases": int(quant_failed_when_ref_success),
            "quant_success_ref_fail_cases": int(quant_success_when_ref_failed),
            "both_failed_task_hard_cases": int(both_failed),
            "both_success": int(both_success),
            "paired_cases": int(mode_pivot[m].notna().sum()),
        })
    sens = pd.DataFrame(sensitivity_rows).sort_values("quant_fail_ref_success_sensitive_cases", ascending=False)
    print("\nREFERENCE-PAIRED SENSITIVITY SUMMARY")
    print(sens.to_string(index=False))
else:
    print("WARNING: reference mode not found; skipping reference-paired sensitivity")
    sens = pd.DataFrame()

# 4) Save outputs.
task_summary.to_csv(Path(SUMMARIES) / "task_difficulty_summary_manifest_only.csv", index=False)
paired_cases.to_csv(Path(SUMMARIES) / "paired_task_episode_mode_success_matrix.csv", index=False)
if len(sens):
    sens.to_csv(Path(SUMMARIES) / "reference_paired_sensitivity_summary.csv", index=False)

print("\nSAVED:")
print(Path(SUMMARIES) / "task_difficulty_summary_manifest_only.csv")
print(Path(SUMMARIES) / "paired_task_episode_mode_success_matrix.csv")
print(Path(SUMMARIES) / "reference_paired_sensitivity_summary.csv")

print("\nINTERPRETATION RULE:")
print("- If reference fails too, that task/episode is hard/noisy, not proof of quant sensitivity.")
print("- Strong sensitivity = reference succeeds but a quant mode fails on the exact same suite/task/episode.")
print("- Focus reruns on high quant_fail_ref_success cases, not only raw failure count.")



CELL 14B — PAIRED TASK/EPISODE SENSITIVITY ANALYSIS — fixed seeds checked
OK_SENSITIVITY_FIXED_SEEDS_SHARED_ACROSS_MODES

TASK DIFFICULTY SUMMARY — worst first
    suite  task_id  runs  success  failures  task_fail  crash  timeout  client_exception  success_rate
libero_10        8    40       28        12         12      0        0                 0         0.700
libero_10        9    40       33         7          7      0        0                 0         0.825
libero_10        0    40       35         5          5      0        0                 0         0.875
libero_10        4    40       36         4          4      0        0                 0         0.900
libero_10        7    40       36         4          4      0        0                 0         0.900
libero_10        2    40       38         2          2      0        0                 0         0.950
libero_10        3    40       38         2          2      0        0                 0         0.950
libero_10       

In [29]:
# CELL 14C — Printed drift-ratio sensitivity analysis (Option 1, no rerun, no site-scope change)
# Purpose:
#   Check whether A8 flow windows differ using continuous drift ratios, not only binary success/fail.
#   This uses the same-episode W8A16 reference ratios already saved in .done.json.
#   It is manifest-only: stale files from other run folders/old manifests are ignored.
#   It only prints tables and saves CSVs. It does NOT change quantization sites; Option 3 is not enabled.

import json
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 260)

print("=" * 120)
print("CELL 14C — PRINTED DRIFT-RATIO WINDOW SENSITIVITY ANALYSIS")
print("=" * 120)
print("Goal: check whether windows matter using continuous drift, including successful episodes.")
print("This is stronger than success/fail alone because every completed episode contributes drift metrics.")
print("Option 3 / action_after site changes are NOT used here.")

# ---------------------------------------------------------------------
# 1) Resolve current run folders and manifest strictly.
# ---------------------------------------------------------------------
RUN_ROOT = Path(RUN_ROOT)
RESULTS = Path(RESULTS)
SUMMARIES = Path(SUMMARIES)
DRIFT_DIR = Path(globals().get("DRIFT", globals().get("DRIFT_JSONL", RUN_ROOT / "drift_jsonl")))

manifest_path = SUMMARIES / "manifest_v40_linear130_split_ref5ep_windows_cumulative.json"
if "manifest" not in globals():
    if not manifest_path.exists():
        raise RuntimeError(f"Missing manifest: {manifest_path}. Run Cell 12 first.")
    manifest = json.loads(manifest_path.read_text())

manifest_stems = {str(x.get("result_stem", "")) for x in manifest}
manifest_keys = {
    (str(x.get("suite")), int(x.get("task_id")), int(x.get("episode_id")), str(x.get("ablation_mode")))
    for x in manifest
}

print("RUN_ROOT:", RUN_ROOT)
print("RESULTS:", RESULTS)
print("SUMMARIES:", SUMMARIES)
print("DRIFT_DIR:", DRIFT_DIR)
print("MANIFEST_ROWS:", len(manifest))

# ---------------------------------------------------------------------
# 2) Load manifest-only done.json records.
# ---------------------------------------------------------------------
records = []
extra_done = []
duplicate_stems = []
seen = set()

for p in sorted(RESULTS.glob("*/*.done.json")):
    try:
        d = json.loads(p.read_text())
        stem = str(d.get("result_stem", ""))
        if stem not in manifest_stems:
            extra_done.append(str(p))
            continue
        if stem in seen:
            duplicate_stems.append(stem)
        seen.add(stem)
        d["done_json"] = str(p)
        records.append(d)
    except Exception as exc:
        print("READ_WARN", p, exc)

done = pd.DataFrame(records).drop_duplicates(subset=["result_stem"], keep="last")
print("DONE_JSON_MATCHED_ROWS:", len(done))
print("EXTRA_DONE_JSON_IGNORED:", len(extra_done))
print("DUPLICATE_STEMS_IGNORED_KEEP_LAST:", len(duplicate_stems))

if len(done) == 0:
    raise RuntimeError("No manifest-matched done.json records found. Run episodes first.")
if len(done) != len(manifest):
    print("WARNING: done rows do not equal manifest rows. You can still inspect partial drift, but do not call it final.")

# Type cleanup.
for c in ["success", "task_fail", "server_crash", "timeout_or_missing", "client_exception"]:
    if c not in done.columns:
        done[c] = False
    done[c] = done[c].fillna(False).astype(bool)

done["task_id"] = pd.to_numeric(done["task_id"], errors="coerce").astype("Int64")
done["episode_id"] = pd.to_numeric(done["episode_id"], errors="coerce").astype("Int64")

# ---------------------------------------------------------------------
# 3) Flatten saved episode-level drift summaries and ratios.
# ---------------------------------------------------------------------
def nested_get(d, parent, key, default=np.nan):
    obj = d.get(parent, {}) if isinstance(d, dict) else {}
    if isinstance(obj, dict) and key in obj:
        return obj.get(key)
    return default

def as_float(x):
    try:
        if x is None:
            return np.nan
        return float(x)
    except Exception:
        return np.nan

rows = []
for _, r in done.iterrows():
    d = r.to_dict()
    ratios = d.get("drift_ratio_vs_w8a16_reference", {})
    drift = d.get("drift_metrics_summary", {})
    if not isinstance(ratios, dict):
        ratios = {}
    if not isinstance(drift, dict):
        drift = {}
    rows.append({
        "ablation_mode": d.get("ablation_mode"),
        "suite": d.get("suite"),
        "task_id": int(d.get("task_id")) if pd.notna(d.get("task_id")) else -1,
        "episode_id": int(d.get("episode_id")) if pd.notna(d.get("episode_id")) else -1,
        "success": bool(d.get("success", False)),
        "task_fail": bool(d.get("task_fail", False)),
        "server_crash": bool(d.get("server_crash", False)),
        "timeout_or_missing": bool(d.get("timeout_or_missing", False)),
        "client_exception": bool(d.get("client_exception", False)),
        "episode_seed": d.get("episode_seed", np.nan),
        "a8_steps": tuple(int(v) for v in d.get("a8_steps", [])),
        # raw episode summary metrics
        "requests": as_float(drift.get("requests")),
        "final_rms_mean": as_float(drift.get("final_rms_mean")),
        "final_rms_max": as_float(drift.get("final_rms_max")),
        "final_maxabs_max": as_float(drift.get("final_maxabs_max")),
        "xyz_rms_mean": as_float(drift.get("xyz_rms_mean")),
        "rot_rms_mean": as_float(drift.get("rot_rms_mean")),
        "vel_rms_max": as_float(drift.get("vel_rms_max")),
        "endpoint_rms_max": as_float(drift.get("endpoint_rms_max")),
        "state_after_rms_max": as_float(drift.get("state_after_rms_max")),
        "state_before_rms_max": as_float(drift.get("state_before_rms_max")),
        "grip_mismatch_total": as_float(drift.get("grip_mismatch_total")),
        "grip_boundary_total": as_float(drift.get("grip_boundary_total")),
        "qact_calls_total": as_float(drift.get("qact_calls_total")),
        # same-episode ratio vs reference; reference rows usually NaN here by design
        "final_rms_mean_ratio": as_float(ratios.get("final_rms_mean_ratio_vs_ref")),
        "final_rms_max_ratio": as_float(ratios.get("final_rms_max_ratio_vs_ref")),
        "vel_rms_max_ratio": as_float(ratios.get("vel_rms_max_ratio_vs_ref")),
        "endpoint_rms_max_ratio": as_float(ratios.get("endpoint_rms_max_ratio_vs_ref")),
        "state_after_rms_max_ratio": as_float(ratios.get("state_after_rms_max_ratio_vs_ref")),
    })

ep = pd.DataFrame(rows)

REFERENCE_MODE = "v40_w8a16_reference"
MODE_ORDER = [
    "v40_w8a16_reference",
    "v40_w8a8_ALL_00_31",
    "v40_w8a8_R0_00_07",
    "v40_w8a8_R1_08_15",
    "v40_w8a8_R2_16_23",
    "v40_w8a8_R3_24_31",
    "v40_w8a8_CUM_00_15",
    "v40_w8a8_CUM_00_23",
]
MODE_NUM = {m: i + 1 for i, m in enumerate(MODE_ORDER)}
SHORT = {
    "v40_w8a16_reference": "1_REF",
    "v40_w8a8_ALL_00_31": "2_ALL",
    "v40_w8a8_R0_00_07": "3_R0",
    "v40_w8a8_R1_08_15": "4_R1",
    "v40_w8a8_R2_16_23": "5_R2",
    "v40_w8a8_R3_24_31": "6_R3",
    "v40_w8a8_CUM_00_15": "7_CUM0015",
    "v40_w8a8_CUM_00_23": "8_CUM0023",
}
ep["mode_num"] = ep["ablation_mode"].map(MODE_NUM)
ep["mode_short"] = ep["ablation_mode"].map(SHORT).fillna(ep["ablation_mode"])
ep["episode_key"] = list(zip(ep["suite"], ep["task_id"], ep["episode_id"]))

# Check fixed-seed pairing proof if available.
if "episode_seed" in ep.columns and ep["episode_seed"].notna().any():
    seed_check = ep.groupby(["suite", "task_id", "episode_id"])["episode_seed"].nunique(dropna=True).reset_index(name="n_unique_seeds")
    bad_seed = seed_check[seed_check["n_unique_seeds"] > 1]
    print("FIXED_SEED_GROUPS:", len(seed_check), "BAD_GROUPS:", len(bad_seed))
    if len(bad_seed):
        print("WARNING_FIXED_SEED_MISMATCH_FIRST_20")
        print(bad_seed.head(20).to_string(index=False))
else:
    print("FIXED_SEED_PROOF: episode_seed missing or all NaN; drift tables still valid, seed proof unavailable.")

# ---------------------------------------------------------------------
# 4) Print success table, then ratio distribution tables.
# ---------------------------------------------------------------------
def p05(x): return float(np.nanpercentile(pd.to_numeric(x, errors="coerce"), 5))
def p25(x): return float(np.nanpercentile(pd.to_numeric(x, errors="coerce"), 25))
def p75(x): return float(np.nanpercentile(pd.to_numeric(x, errors="coerce"), 75))
def p95(x): return float(np.nanpercentile(pd.to_numeric(x, errors="coerce"), 95))
for f, name in [(p05, "p05"), (p25, "p25"), (p75, "p75"), (p95, "p95")]:
    f.__name__ = name

print("\n" + "=" * 120)
print("SUCCESS/FAIL BY MODE — binary outcome, shown only as context")
print("=" * 120)
success_table = ep.groupby(["mode_num", "mode_short", "ablation_mode"]).agg(
    runs=("success", "count"),
    success=("success", "sum"),
    task_fail=("task_fail", "sum"),
    crash=("server_crash", "sum"),
    timeout=("timeout_or_missing", "sum"),
    client_exception=("client_exception", "sum"),
).reset_index().sort_values("mode_num")
success_table["success_rate"] = success_table["success"] / success_table["runs"].clip(lower=1)
print(success_table.to_string(index=False))

ratio_cols = [
    "final_rms_mean_ratio",
    "final_rms_max_ratio",
    "vel_rms_max_ratio",
    "endpoint_rms_max_ratio",
    "state_after_rms_max_ratio",
]
q = ep[ep["ablation_mode"] != REFERENCE_MODE].copy()

print("\n" + "=" * 120)
print("DRIFT RATIO DISTRIBUTIONS BY MODE — all episodes, success and fail included")
print("Higher ratio = more deviation from same-episode W8A16 reference. Median checks typical error; p95/max checks tail risk.")
print("=" * 120)
ratio_summary_parts = []
for col in ratio_cols:
    if col not in q.columns or q[col].notna().sum() == 0:
        print(f"\n{col}: no saved ratio values found")
        continue
    tab = q.groupby(["mode_num", "mode_short"])[col].agg(["count", "median", p25, p75, p95, "max"]).reset_index().sort_values("mode_num")
    tab = tab.rename(columns={"count": f"{col}_n", "median": f"{col}_median", "p25": f"{col}_p25", "p75": f"{col}_p75", "p95": f"{col}_p95", "max": f"{col}_max"})
    print(f"\n--- {col} ---")
    print(tab.to_string(index=False, float_format=lambda x: f"{x:,.3g}"))
    ratio_summary_parts.append(tab)

# Compact one-row-per-mode table using key metrics.
compact_cols = [c for c in ["vel_rms_max_ratio", "endpoint_rms_max_ratio", "state_after_rms_max_ratio", "final_rms_mean_ratio"] if c in q.columns and q[c].notna().sum()]
compact = q.groupby(["mode_num", "mode_short", "ablation_mode"])[compact_cols].agg(["median", p95]).reset_index().sort_values("mode_num")
# Flatten columns.
compact.columns = ["_".join([str(x) for x in c if x]) if isinstance(c, tuple) else str(c) for c in compact.columns]
print("\n" + "=" * 120)
print("COMPACT MODE DRIFT TABLE — median and p95 of main ratios")
print("=" * 120)
print(compact.to_string(index=False, float_format=lambda x: f"{x:,.3g}"))

# ---------------------------------------------------------------------
# 5) Printed task × mode tables. These reveal whether failures are task difficulty or window sensitivity.
# ---------------------------------------------------------------------
print("\n" + "=" * 120)
print("TASK × MODE MEDIAN DRIFT TABLES — printed heatmaps")
print("=" * 120)
for col in ["vel_rms_max_ratio", "endpoint_rms_max_ratio", "state_after_rms_max_ratio", "final_rms_mean_ratio"]:
    if col not in q.columns or q[col].notna().sum() == 0:
        continue
    pivot = q.pivot_table(index="task_id", columns="mode_short", values=col, aggfunc="median")
    # Keep mode order.
    ordered_cols = [SHORT[m] for m in MODE_ORDER if m != REFERENCE_MODE and SHORT[m] in pivot.columns]
    pivot = pivot[ordered_cols]
    print(f"\n--- task × mode median {col} ---")
    print(pivot.round(1).to_string())
    pivot.to_csv(SUMMARIES / f"printed_task_by_mode_median_{col}.csv")

# ---------------------------------------------------------------------
# 6) Paired reference-vs-quant sensitivity using success/fail, with drift side by side.
# ---------------------------------------------------------------------
print("\n" + "=" * 120)
print("PAIRED SUCCESS SENSITIVITY + DRIFT")
print("=" * 120)
print("quant_sensitive = reference succeeded on the same task/episode, but quant mode failed.")
print("task_noise = reference failed too, so a quant failure is not clean evidence by itself.")

wide_success = ep.pivot_table(index=["suite", "task_id", "episode_id"], columns="ablation_mode", values="success", aggfunc="max")
paired_rows = []
if REFERENCE_MODE in wide_success.columns:
    for mode in [m for m in MODE_ORDER if m != REFERENCE_MODE and m in ep["ablation_mode"].unique()]:
        if mode not in wide_success.columns:
            continue
        ref = wide_success[REFERENCE_MODE].astype("boolean")
        qm = wide_success[mode].astype("boolean")
        mode_df = q[q["ablation_mode"] == mode]
        paired_rows.append({
            "mode_num": MODE_NUM[mode],
            "mode_short": SHORT[mode],
            "runs": int(qm.notna().sum()),
            "quant_fail_ref_success": int(((ref == True) & (qm == False)).sum()),
            "quant_success_ref_fail": int(((ref == False) & (qm == True)).sum()),
            "both_fail": int(((ref == False) & (qm == False)).sum()),
            "both_success": int(((ref == True) & (qm == True)).sum()),
            "vel_ratio_median": float(np.nanmedian(mode_df["vel_rms_max_ratio"])) if "vel_rms_max_ratio" in mode_df else np.nan,
            "vel_ratio_p95": p95(mode_df["vel_rms_max_ratio"]) if "vel_rms_max_ratio" in mode_df and mode_df["vel_rms_max_ratio"].notna().sum() else np.nan,
            "endpoint_ratio_median": float(np.nanmedian(mode_df["endpoint_rms_max_ratio"])) if "endpoint_rms_max_ratio" in mode_df else np.nan,
            "endpoint_ratio_p95": p95(mode_df["endpoint_rms_max_ratio"]) if "endpoint_rms_max_ratio" in mode_df and mode_df["endpoint_rms_max_ratio"].notna().sum() else np.nan,
            "state_after_ratio_median": float(np.nanmedian(mode_df["state_after_rms_max_ratio"])) if "state_after_rms_max_ratio" in mode_df else np.nan,
            "state_after_ratio_p95": p95(mode_df["state_after_rms_max_ratio"]) if "state_after_rms_max_ratio" in mode_df and mode_df["state_after_rms_max_ratio"].notna().sum() else np.nan,
        })
    paired = pd.DataFrame(paired_rows).sort_values(["quant_fail_ref_success", "vel_ratio_p95"], ascending=[False, False])
    print(paired.to_string(index=False, float_format=lambda x: f"{x:,.3g}"))
    paired.to_csv(SUMMARIES / "printed_paired_success_sensitivity_plus_drift.csv", index=False)
else:
    print("Reference mode missing in done records; paired success sensitivity skipped.")

# ---------------------------------------------------------------------
# 7) Worst task/mode drift cases: where windows produce the largest continuous deviation.
# ---------------------------------------------------------------------
print("\n" + "=" * 120)
print("WORST TASK × MODE DRIFT CASES")
print("=" * 120)
worst_rows = []
for col in ["vel_rms_max_ratio", "endpoint_rms_max_ratio", "state_after_rms_max_ratio", "final_rms_mean_ratio"]:
    if col not in q.columns or q[col].notna().sum() == 0:
        continue
    tmp = q.groupby(["task_id", "mode_num", "mode_short"])[col].agg(["count", "median", p95, "max"]).reset_index()
    tmp["metric"] = col
    tmp = tmp.rename(columns={"count": "n", "median": "median", "p95": "p95", "max": "max"})
    worst_rows.append(tmp)
if worst_rows:
    worst = pd.concat(worst_rows, ignore_index=True)
    for col in ["vel_rms_max_ratio", "endpoint_rms_max_ratio", "state_after_rms_max_ratio", "final_rms_mean_ratio"]:
        sub = worst[worst["metric"] == col].sort_values("p95", ascending=False).head(20)
        if len(sub):
            print(f"\n--- Top task×mode by p95 {col} ---")
            print(sub.to_string(index=False, float_format=lambda x: f"{x:,.3g}"))
    worst.to_csv(SUMMARIES / "printed_worst_task_mode_drift_cases.csv", index=False)

# ---------------------------------------------------------------------
# 8) Optional request-level print if JSONL request records are present.
# ---------------------------------------------------------------------
print("\n" + "=" * 120)
print("REQUEST-LEVEL RAW DRIFT — optional extra power from every action request")
print("=" * 120)
req_records = []

def drift_key_ok(r):
    try:
        k = (str(r.get("suite")), int(r.get("task_id")), int(r.get("episode_id")), str(r.get("ablation_mode")))
        return k in manifest_keys
    except Exception:
        return False

if DRIFT_DIR.exists():
    files = [p for p in sorted(DRIFT_DIR.glob("*_request_records.jsonl")) if p.name != "all_modes_request_records.jsonl"]
    for p in files:
        try:
            for line in p.read_text(errors="ignore").splitlines():
                if not line.strip():
                    continue
                r = json.loads(line)
                if not drift_key_ok(r):
                    continue
                base = {k: v for k, v in r.items() if k not in ("step_records", "site_records")}
                req_records.append(base)
        except Exception as exc:
            print("JSONL_READ_WARN", p, exc)

req = pd.DataFrame(req_records)
print("REQUEST_RECORDS_MATCHED:", len(req))
if len(req):
    req["mode_num"] = req["ablation_mode"].map(MODE_NUM)
    req["mode_short"] = req["ablation_mode"].map(SHORT).fillna(req["ablation_mode"])
    for c in ["final_action_7d_rms", "final_action_7d_max_abs", "xyz_rms", "rot_rms", "gripper_threshold_mismatch_count", "gripper_boundary_count"]:
        if c in req.columns:
            req[c] = pd.to_numeric(req[c], errors="coerce")
    req_metric_cols = [c for c in ["final_action_7d_rms", "final_action_7d_max_abs", "xyz_rms", "rot_rms", "gripper_threshold_mismatch_count", "gripper_boundary_count"] if c in req.columns]
    if req_metric_cols:
        req_summary = req.groupby(["mode_num", "mode_short"])[req_metric_cols].agg(["count", "median", p95, "max"]).reset_index().sort_values("mode_num")
        req_summary.columns = ["_".join([str(x) for x in c if x]) if isinstance(c, tuple) else str(c) for c in req_summary.columns]
        print(req_summary.to_string(index=False, float_format=lambda x: f"{x:,.4g}"))
        req_summary.to_csv(SUMMARIES / "printed_request_level_raw_drift_by_mode.csv", index=False)
else:
    print("No request JSONL records found. Episode-level ratio tables above are still enough for Option 1.")

# ---------------------------------------------------------------------
# 9) Interpretation rule printed explicitly.
# ---------------------------------------------------------------------
print("\n" + "=" * 120)
print("HOW TO READ THIS")
print("=" * 120)
print("1) If success rates are similar but ratio p95/max differ by mode, windows still matter: they change internal trajectory risk.")
print("2) If a mode has high quant_fail_ref_success, it creates clean failures beyond task difficulty.")
print("3) If a task has high failure across reference and all quant modes, that is task difficulty, not clean quant sensitivity.")
print("4) For the next rerun, choose task/episode pairs with high drift p95 or quant_fail_ref_success, not just random failures.")
print("5) This cell does not modify model code, quantization scope, seeds, or sites.")

# Save main episode-level flattened table for later quick analysis.
ep_out = SUMMARIES / "printed_episode_level_drift_ratios_manifest_only.csv"
ep.to_csv(ep_out, index=False)
print("\nSAVED:", ep_out)
print("SAVED task×mode median CSVs and paired/worst tables under:", SUMMARIES)


CELL 14C — PRINTED DRIFT-RATIO WINDOW SENSITIVITY ANALYSIS
Goal: check whether windows matter using continuous drift, including successful episodes.
This is stronger than success/fail alone because every completed episode contributes drift metrics.
Option 3 / action_after site changes are NOT used here.
RUN_ROOT: /content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a
RESULTS: /content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a/results
SUMMARIES: /content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a/summaries
DRIFT_DIR: /content/drive/MyDrive/E

In [30]:
# CELL 14D — Three-layer sensitivity verdict: drift distributions + paired damage + task difficulty
# Run after Cell 14B and Cell 14C, or standalone after Cell 13B.
# This is the final printed decision layer: it does NOT change model code or quantization sites.

import json
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 260)

print("=" * 120)
print("CELL 14D — THREE-LAYER WINDOW SENSITIVITY VERDICT")
print("=" * 120)
print("Layer 1: continuous drift ratios on every episode, including successes.")
print("Layer 2: paired outcome damage where reference succeeds and quant mode fails.")
print("Layer 3: task/episode difficulty separation where reference also fails.")

REFERENCE_MODE = globals().get("REFERENCE_MODE", "v40_w8a16_reference")
MODE_ORDER = [
    "v40_w8a16_reference",
    "v40_w8a8_ALL_00_31",
    "v40_w8a8_R0_00_07",
    "v40_w8a8_R1_08_15",
    "v40_w8a8_R2_16_23",
    "v40_w8a8_R3_24_31",
    "v40_w8a8_CUM_00_15",
    "v40_w8a8_CUM_00_23",
]
SHORT = {m: m.replace("v40_w8a8_", "").replace("v40_w8a16_", "") for m in MODE_ORDER}

if "manifest" not in globals():
    manifest_path = Path(SUMMARIES) / "manifest_v40_linear130_split_ref5ep_windows_cumulative.json"
    if not manifest_path.exists():
        manifest_path = Path(SUMMARIES) / "manifest.json"
    manifest = json.loads(manifest_path.read_text())
manifest_stems = {str(x.get("result_stem", "")) for x in manifest}

records = []
for p in sorted(Path(RESULTS).glob("*/*.done.json")):
    d = json.loads(p.read_text())
    if str(d.get("result_stem", "")) not in manifest_stems:
        continue
    ratios = d.get("drift_ratio_vs_w8a16_reference", {}) or {}
    row = {
        "result_stem": d.get("result_stem"),
        "mode": d.get("ablation_mode"),
        "suite": d.get("suite"),
        "task_id": int(d.get("task_id", -1)),
        "episode_id": int(d.get("episode_id", -1)),
        "episode_seed": d.get("episode_seed", np.nan),
        "success": bool(d.get("success", False)),
        "task_fail": bool(d.get("task_fail", False)),
        "server_crash": bool(d.get("server_crash", False)),
        "timeout_or_missing": bool(d.get("timeout_or_missing", False)),
        "client_exception": bool(d.get("client_exception", False)),
        "final_rms_mean_ratio": ratios.get("final_rms_mean_ratio_vs_ref", np.nan),
        "vel_rms_max_ratio": ratios.get("vel_rms_max_ratio_vs_ref", np.nan),
        "endpoint_rms_max_ratio": ratios.get("endpoint_rms_max_ratio_vs_ref", np.nan),
        "state_after_rms_max_ratio": ratios.get("state_after_rms_max_ratio_vs_ref", np.nan),
        "grip_mismatch_total": d.get("grip_mismatch_total", 0),
        "grip_boundary_total": d.get("grip_boundary_total", 0),
    }
    records.append(row)

ep = pd.DataFrame(records).drop_duplicates(subset=["result_stem"], keep="last")
if ep.empty:
    raise RuntimeError("No manifest-matched done records found")

print("\nDATASET CHECK")
print("matched_episode_rows:", len(ep))
print("modes:", sorted(ep["mode"].unique()))
print("unique task/episode groups:", ep[["suite", "task_id", "episode_id"]].drop_duplicates().shape[0])

# Seed proof for the verdict layer.
if "episode_seed" in ep.columns and ep["episode_seed"].notna().any():
    seed_bad = ep.groupby(["suite", "task_id", "episode_id"])["episode_seed"].nunique(dropna=True).reset_index(name="n_unique_seeds")
    seed_bad = seed_bad[seed_bad["n_unique_seeds"] != 1]
    if len(seed_bad):
        print(seed_bad.to_string(index=False))
        raise RuntimeError("14D abort: fixed seed pairing failed")
    print("OK_FIXED_SEED_PAIRING_FOR_14D")
else:
    print("WARNING: no episode_seed proof available in done rows")

# Layer 1: continuous drift ratios.
ratio_cols = ["final_rms_mean_ratio", "vel_rms_max_ratio", "endpoint_rms_max_ratio", "state_after_rms_max_ratio"]
quant = ep[ep["mode"] != REFERENCE_MODE].copy()
for c in ratio_cols:
    quant[c] = pd.to_numeric(quant[c], errors="coerce")

print("\n" + "=" * 100)
print("LAYER 1 — CONTINUOUS DRIFT DISTRIBUTIONS BY MODE")
print("=" * 100)
layer1_rows = []
for mode, g in quant.groupby("mode"):
    row = {"mode": mode, "mode_short": SHORT.get(mode, mode), "n": len(g)}
    for c in ratio_cols:
        vals = g[c].dropna().astype(float)
        if len(vals):
            row[f"{c}_median"] = float(vals.median())
            row[f"{c}_p75"] = float(vals.quantile(0.75))
            row[f"{c}_p95"] = float(vals.quantile(0.95))
            row[f"{c}_max"] = float(vals.max())
        else:
            row[f"{c}_median"] = np.nan
            row[f"{c}_p75"] = np.nan
            row[f"{c}_p95"] = np.nan
            row[f"{c}_max"] = np.nan
    # A compact drift score for ranking only; do not treat as a paper metric.
    row["drift_score_med_log10"] = float(np.nanmean([
        np.log10(max(row.get("vel_rms_max_ratio_median", np.nan), 1e-9)),
        np.log10(max(row.get("endpoint_rms_max_ratio_median", np.nan), 1e-9)),
        np.log10(max(row.get("state_after_rms_max_ratio_median", np.nan), 1e-9)),
        np.log10(max(row.get("final_rms_mean_ratio_median", np.nan), 1e-9)),
    ]))
    layer1_rows.append(row)
layer1 = pd.DataFrame(layer1_rows)
layer1["mode_order"] = layer1["mode"].map({m:i for i,m in enumerate(MODE_ORDER)})
print(layer1.sort_values("drift_score_med_log10", ascending=False).to_string(index=False))

# Layer 2 + 3: paired outcome table.
print("\n" + "=" * 100)
print("LAYER 2/3 — PAIRED OUTCOME DAMAGE VS TASK DIFFICULTY")
print("=" * 100)
keys = ["suite", "task_id", "episode_id"]
wide = ep.pivot_table(index=keys, columns="mode", values="success", aggfunc="last")
if REFERENCE_MODE not in wide.columns:
    raise RuntimeError(f"Reference mode {REFERENCE_MODE} not found in paired table")
ref_success = wide[REFERENCE_MODE].astype(bool)

paired_rows = []
for mode in [m for m in MODE_ORDER if m != REFERENCE_MODE and m in wide.columns]:
    qsucc = wide[mode].astype(bool)
    both_success = int((ref_success & qsucc).sum())
    quant_damage = int((ref_success & (~qsucc)).sum())          # clean sensitivity: ref succeeds, quant fails
    task_difficulty = int(((~ref_success) & (~qsucc)).sum())    # hard/noisy episode: both fail
    quant_rescue_or_noise = int(((~ref_success) & qsucc).sum()) # ref fails, quant succeeds
    paired_rows.append({
        "mode": mode,
        "mode_short": SHORT.get(mode, mode),
        "groups": int(len(wide)),
        "both_success": both_success,
        "quant_damage_refS_quantF": quant_damage,
        "both_fail_task_difficulty": task_difficulty,
        "quant_rescue_or_noise_refF_quantS": quant_rescue_or_noise,
        "damage_rate_given_ref_success": quant_damage / max(int(ref_success.sum()), 1),
        "raw_success_rate": float(qsucc.mean()),
    })
paired = pd.DataFrame(paired_rows)
print(paired.sort_values(["damage_rate_given_ref_success", "quant_damage_refS_quantF"], ascending=False).to_string(index=False))

# Task difficulty layer: where reference itself fails.
print("\n" + "=" * 100)
print("LAYER 3 — HARD TASK/EPISODE GROUPS WHERE REFERENCE FAILS")
print("=" * 100)
ref_fail_groups = wide[~ref_success].reset_index()[keys]
print("reference_failed_groups:", len(ref_fail_groups))
if len(ref_fail_groups):
    print(ref_fail_groups.sort_values(keys).to_string(index=False))

task_layer = ep.groupby(["suite", "task_id"]).agg(
    runs=("success", "count"),
    success=("success", "sum"),
    task_fail=("task_fail", "sum"),
).reset_index()
task_layer["success_rate"] = task_layer["success"] / task_layer["runs"].clip(lower=1)
print("\nTASK DIFFICULTY SUMMARY, WORST FIRST")
print(task_layer.sort_values(["task_fail", "task_id"], ascending=[False, True]).to_string(index=False))

# Combined verdict table.
print("\n" + "=" * 100)
print("COMBINED VERDICT TABLE")
print("=" * 100)
verdict = paired.merge(
    layer1[["mode", "drift_score_med_log10", "vel_rms_max_ratio_median", "endpoint_rms_max_ratio_median", "state_after_rms_max_ratio_median", "final_rms_mean_ratio_median"]],
    on="mode", how="left"
)
# Sort worst first: paired damage first, then drift score.
verdict = verdict.sort_values(["damage_rate_given_ref_success", "drift_score_med_log10"], ascending=[False, False])
print(verdict.to_string(index=False))

# Save outputs.
out1 = Path(SUMMARIES) / "three_layer_drift_distributions_by_mode.csv"
out2 = Path(SUMMARIES) / "three_layer_paired_damage_by_mode.csv"
out3 = Path(SUMMARIES) / "three_layer_task_difficulty_summary.csv"
out4 = Path(SUMMARIES) / "three_layer_combined_verdict.csv"
layer1.to_csv(out1, index=False)
paired.to_csv(out2, index=False)
task_layer.to_csv(out3, index=False)
verdict.to_csv(out4, index=False)
print("\nSAVED")
print(out1)
print(out2)
print(out3)
print(out4)

print("\nINTERPRETATION RULE")
print("- High drift but no paired damage: window perturbs actions but may be behaviorally tolerated.")
print("- Ref succeeds + quant fails: strongest evidence of quant-window sensitivity.")
print("- Ref fails too: hard task/episode; do not blame quantization without rerun/repeat.")
print("- A real paper claim needs the same mode to show both elevated drift distribution and repeated paired damage.")


CELL 14D — THREE-LAYER WINDOW SENSITIVITY VERDICT
Layer 1: continuous drift ratios on every episode, including successes.
Layer 2: paired outcome damage where reference succeeds and quant mode fails.
Layer 3: task/episode difficulty separation where reference also fails.

DATASET CHECK
matched_episode_rows: 400
modes: ['v40_w8a16_reference', 'v40_w8a8_ALL_00_31', 'v40_w8a8_CUM_00_15', 'v40_w8a8_CUM_00_23', 'v40_w8a8_R0_00_07', 'v40_w8a8_R1_08_15', 'v40_w8a8_R2_16_23', 'v40_w8a8_R3_24_31']
unique task/episode groups: 50
OK_FIXED_SEED_PAIRING_FOR_14D

LAYER 1 — CONTINUOUS DRIFT DISTRIBUTIONS BY MODE
              mode mode_short  n  final_rms_mean_ratio_median  final_rms_mean_ratio_p75  final_rms_mean_ratio_p95  final_rms_mean_ratio_max  vel_rms_max_ratio_median  vel_rms_max_ratio_p75  vel_rms_max_ratio_p95  vel_rms_max_ratio_max  endpoint_rms_max_ratio_median  endpoint_rms_max_ratio_p75  endpoint_rms_max_ratio_p95  endpoint_rms_max_ratio_max  state_after_rms_max_ratio_median  state_afte

In [31]:
# CELL 14E — Paired statistical checks and bootstrap CIs for rigorous reporting
# Run after Cell 13B/13C and preferably after Cell 14B/14C/14D.
# Purpose:
#   Convert the three-layer evidence into paired statistics:
#   - paired binary outcome deltas vs reference (exact sign/binomial test on discordant pairs)
#   - bootstrap CIs for continuous drift ratios
#   - reference baseline quality warning
# This does NOT change quantization or rerun evaluation.

import json, math, hashlib
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 260)

print("=" * 120)
print("CELL 14E — PAIRED STATISTICAL CHECKS + BOOTSTRAP CIS")
print("=" * 120)

REFERENCE_MODE = globals().get("REFERENCE_MODE", "v40_w8a16_reference")
MODE_ORDER = [
    "v40_w8a16_reference",
    "v40_w8a8_ALL_00_31",
    "v40_w8a8_R0_00_07",
    "v40_w8a8_R1_08_15",
    "v40_w8a8_R2_16_23",
    "v40_w8a8_R3_24_31",
    "v40_w8a8_CUM_00_15",
    "v40_w8a8_CUM_00_23",
]
SHORT = {m: m.replace("v40_w8a8_", "").replace("v40_w8a16_", "") for m in MODE_ORDER}

if "manifest" not in globals():
    manifest_path = Path(SUMMARIES) / "manifest_v40_linear130_split_ref5ep_windows_cumulative.json"
    if not manifest_path.exists():
        raise RuntimeError(f"Expected manifest not found: {manifest_path}")
    manifest = json.loads(manifest_path.read_text())
manifest_stems = {str(x.get("result_stem", "")) for x in manifest}

rows = []
for p in sorted(Path(RESULTS).glob("*/*.done.json")):
    d = json.loads(p.read_text())
    if str(d.get("result_stem", "")) not in manifest_stems:
        continue
    ratios = d.get("drift_ratio_vs_w8a16_reference", {}) or {}
    rows.append({
        "result_stem": d.get("result_stem"),
        "mode": d.get("ablation_mode"),
        "suite": d.get("suite"),
        "task_id": int(d.get("task_id", -1)),
        "episode_id": int(d.get("episode_id", -1)),
        "episode_seed": d.get("episode_seed", np.nan),
        "success": bool(d.get("success", False)),
        "task_fail": bool(d.get("task_fail", False)),
        "server_crash": bool(d.get("server_crash", False)),
        "timeout_or_missing": bool(d.get("timeout_or_missing", False)),
        "client_exception": bool(d.get("client_exception", False)),
        "env_steps": float(d.get("env_steps", np.nan)),
        "outer_steps": float(d.get("outer_steps", np.nan)),
        "final_rms_mean_ratio": ratios.get("final_rms_mean_ratio_vs_ref", np.nan),
        "vel_rms_max_ratio": ratios.get("vel_rms_max_ratio_vs_ref", np.nan),
        "endpoint_rms_max_ratio": ratios.get("endpoint_rms_max_ratio_vs_ref", np.nan),
        "state_after_rms_max_ratio": ratios.get("state_after_rms_max_ratio_vs_ref", np.nan),
        "grip_mismatch_total": d.get("grip_mismatch_total", 0),
        "grip_boundary_total": d.get("grip_boundary_total", 0),
    })

ep = pd.DataFrame(rows).drop_duplicates(subset=["result_stem"], keep="last")
if len(ep) != len(manifest):
    raise RuntimeError(f"14E abort: manifest-matched rows != manifest length ({len(ep)} vs {len(manifest)}). Do not compute paired stats on incomplete data.")

_duplicate_pair_keys = ep.duplicated(subset=["suite", "task_id", "episode_id", "mode"], keep=False)
if _duplicate_pair_keys.any():
    print(ep.loc[_duplicate_pair_keys, ["suite", "task_id", "episode_id", "mode", "result_stem"]].sort_values(["suite", "task_id", "episode_id", "mode"]).to_string(index=False))
    raise RuntimeError("14E abort: duplicate suite/task/episode/mode rows remain after manifest filtering")

for c in ["success", "task_fail", "server_crash", "timeout_or_missing", "client_exception"]:
    ep[c] = ep[c].fillna(False).astype(bool)

group_cols = ["suite", "task_id", "episode_id"]
base_groups = ep[group_cols].drop_duplicates().shape[0]
print("RUN_ROOT:", RUN_ROOT)
print("matched rows:", len(ep))
print("paired task/episode groups:", base_groups)
print("modes:", sorted(ep["mode"].unique()))

# Check seed pairing.
if "episode_seed" in ep.columns and ep["episode_seed"].notna().any():
    seed_counts = ep.groupby(group_cols)["episode_seed"].nunique(dropna=True)
    bad_seed = seed_counts[seed_counts != 1]
    if len(bad_seed):
        print(bad_seed.head(30).to_string())
        raise RuntimeError("14E abort: seed pairing failed")
    print("OK_SEED_PAIRING_FOR_STATS")
else:
    print("WARNING: no episode_seed column available")

# Reference baseline quality.
ref = ep[ep["mode"] == REFERENCE_MODE].copy()
ref_success_rate = float(ref["success"].mean()) if len(ref) else np.nan
print("\nREFERENCE BASELINE QUALITY")
print("reference_mode:", REFERENCE_MODE)
print("reference_runs:", len(ref))
print("reference_success_rate:", ref_success_rate)
if ref_success_rate >= 0.95:
    print("REFERENCE_QUALITY: clean benchmark for quant damage")
elif ref_success_rate >= 0.85:
    print("REFERENCE_QUALITY: usable, but paired hard-task separation is required")
else:
    print("REFERENCE_QUALITY: noisy/stress suite; do not use raw success rate as main proof")

# Exact two-sided binomial sign test for discordant paired outcomes vs reference.
def exact_two_sided_binom_p(k, n):
    # H0 p=0.5. Two-sided probability of outcomes at least as extreme as min(k,n-k).
    if n <= 0:
        return np.nan
    lo = min(k, n-k)
    prob = sum(math.comb(n, i) for i in range(0, lo + 1)) / (2 ** n)
    return float(min(1.0, 2 * prob))

pivot_succ = ep.pivot_table(index=group_cols, columns="mode", values="success", aggfunc="first")
if REFERENCE_MODE not in pivot_succ.columns:
    raise RuntimeError(f"Reference mode {REFERENCE_MODE} not present in success pivot")

paired_rows = []
for m in [x for x in MODE_ORDER if x != REFERENCE_MODE and x in pivot_succ.columns]:
    sub = pivot_succ[[REFERENCE_MODE, m]].dropna()
    ref_s_q_fail = int(((sub[REFERENCE_MODE] == True) & (sub[m] == False)).sum())
    ref_f_q_succ = int(((sub[REFERENCE_MODE] == False) & (sub[m] == True)).sum())
    both_succ = int(((sub[REFERENCE_MODE] == True) & (sub[m] == True)).sum())
    both_fail = int(((sub[REFERENCE_MODE] == False) & (sub[m] == False)).sum())
    discordant = ref_s_q_fail + ref_f_q_succ
    p_exact = exact_two_sided_binom_p(ref_s_q_fail, discordant)
    paired_rows.append({
        "mode": m,
        "short": SHORT.get(m, m),
        "paired_groups": len(sub),
        "both_success": both_succ,
        "ref_success_quant_fail_clean_damage": ref_s_q_fail,
        "ref_fail_quant_success_possible_rescue_or_noise": ref_f_q_succ,
        "both_fail_hard_or_noisy": both_fail,
        "discordant_pairs": discordant,
        "paired_success_delta_quant_minus_ref": ref_f_q_succ - ref_s_q_fail,
        "exact_sign_p_two_sided": p_exact,
    })
paired = pd.DataFrame(paired_rows)
print("\nPAIRED BINARY OUTCOME STATS VS REFERENCE")
print(paired.to_string(index=False))
paired.to_csv(Path(SUMMARIES) / "paired_binary_outcome_stats_vs_reference.csv", index=False)

# Bootstrap CI helper.
def stable_seed(label):
    return int(hashlib.sha256(str(label).encode("utf-8")).hexdigest()[:8], 16)

def bootstrap_ci(values, stat_fn=np.median, n_boot=2000, ci=(2.5, 97.5), seed=123):
    vals = pd.to_numeric(pd.Series(values), errors="coerce").dropna().astype(float).values
    if len(vals) == 0:
        return (np.nan, np.nan, np.nan, 0)
    rng = np.random.default_rng(seed)
    boots = np.empty(n_boot, dtype=float)
    n = len(vals)
    for i in range(n_boot):
        boots[i] = stat_fn(rng.choice(vals, size=n, replace=True))
    lo, hi = np.percentile(boots, ci)
    return float(stat_fn(vals)), float(lo), float(hi), int(n)

ratio_cols = ["final_rms_mean_ratio", "vel_rms_max_ratio", "endpoint_rms_max_ratio", "state_after_rms_max_ratio"]
boot_rows = []
quant = ep[ep["mode"] != REFERENCE_MODE].copy()
for m in [x for x in MODE_ORDER if x != REFERENCE_MODE and x in quant["mode"].unique()]:
    sub = quant[quant["mode"] == m]
    for col in ratio_cols:
        vals = pd.to_numeric(sub[col], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
        med, lo, hi, n = bootstrap_ci(vals, stat_fn=np.median, seed=stable_seed(m + col + "median"))
        p95, p95lo, p95hi, _ = bootstrap_ci(vals, stat_fn=lambda x: np.percentile(x, 95), seed=stable_seed(m + col + "p95"))
        boot_rows.append({
            "mode": m,
            "short": SHORT.get(m, m),
            "metric": col,
            "n": n,
            "median": med,
            "median_ci_lo": lo,
            "median_ci_hi": hi,
            "p95": p95,
            "p95_ci_lo": p95lo,
            "p95_ci_hi": p95hi,
        })
boot = pd.DataFrame(boot_rows)
print("\nBOOTSTRAP CIS FOR DRIFT RATIOS BY MODE")
print(boot.to_string(index=False))
boot.to_csv(Path(SUMMARIES) / "bootstrap_ci_drift_ratios_by_mode.csv", index=False)

print("\nSAVED:")
print(Path(SUMMARIES) / "paired_binary_outcome_stats_vs_reference.csv")
print(Path(SUMMARIES) / "bootstrap_ci_drift_ratios_by_mode.csv")



CELL 14E — PAIRED STATISTICAL CHECKS + BOOTSTRAP CIS
RUN_ROOT: /content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a
matched rows: 400
paired task/episode groups: 50
modes: ['v40_w8a16_reference', 'v40_w8a8_ALL_00_31', 'v40_w8a8_CUM_00_15', 'v40_w8a8_CUM_00_23', 'v40_w8a8_R0_00_07', 'v40_w8a8_R1_08_15', 'v40_w8a8_R2_16_23', 'v40_w8a8_R3_24_31']
OK_SEED_PAIRING_FOR_STATS

REFERENCE BASELINE QUALITY
reference_mode: v40_w8a16_reference
reference_runs: 50
reference_success_rate: 0.88
REFERENCE_QUALITY: usable, but paired hard-task separation is required

PAIRED BINARY OUTCOME STATS VS REFERENCE
              mode     short  paired_groups  both_success  ref_success_quant_fail_clean_damage  ref_fail_quant_success_possible_rescue_or_noise  both_fail_hard_or_noisy  discordant_pairs  paired_success_delta_quant_minus_ref  exact_sign_p_two_sided
v40

In [32]:
# CELL 14F — Success-margin / efficiency analysis: env_steps, outer_steps, gripper risk
# Run after Cell 13B. This detects hidden degradation when all modes still succeed.
# Example: reference succeeds in 8 outer steps while a quant mode succeeds in 25 outer steps.

import json
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 260)

print("=" * 120)
print("CELL 14F — SUCCESS-MARGIN / EFFICIENCY ANALYSIS")
print("=" * 120)
print("Goal: if success rates are too high, use env_steps/outer_steps/gripper risk as margin degradation signals.")

REFERENCE_MODE = globals().get("REFERENCE_MODE", "v40_w8a16_reference")
if "manifest" not in globals():
    manifest_path = Path(SUMMARIES) / "manifest_v40_linear130_split_ref5ep_windows_cumulative.json"
    if not manifest_path.exists():
        manifest_path = Path(SUMMARIES) / "manifest.json"
    if not manifest_path.exists():
        raise RuntimeError(f"Expected manifest not found under {SUMMARIES}")
    manifest = json.loads(manifest_path.read_text())
manifest_stems = {str(x.get("result_stem", "")) for x in manifest}

rows = []
for p in sorted(Path(RESULTS).glob("*/*.done.json")):
    d = json.loads(p.read_text())
    if str(d.get("result_stem", "")) not in manifest_stems:
        continue
    rows.append({
        "result_stem": d.get("result_stem"),
        "mode": d.get("ablation_mode"),
        "suite": d.get("suite"),
        "task_id": int(d.get("task_id", -1)),
        "episode_id": int(d.get("episode_id", -1)),
        "success": bool(d.get("success", False)),
        "env_steps": pd.to_numeric(d.get("env_steps", np.nan), errors="coerce"),
        "outer_steps": pd.to_numeric(d.get("outer_steps", np.nan), errors="coerce"),
        "grip_mismatch_total": pd.to_numeric(d.get("grip_mismatch_total", 0), errors="coerce"),
        "grip_boundary_total": pd.to_numeric(d.get("grip_boundary_total", 0), errors="coerce"),
    })

df = pd.DataFrame(rows).drop_duplicates(subset=["result_stem"], keep="last")
if df.empty:
    raise RuntimeError("No manifest-matched done records found")

# Raw margin table by mode.
margin = df.groupby("mode").agg(
    runs=("success", "count"),
    success=("success", "sum"),
    env_steps_median=("env_steps", "median"),
    env_steps_p75=("env_steps", lambda x: np.nanpercentile(x, 75)),
    env_steps_p95=("env_steps", lambda x: np.nanpercentile(x, 95)),
    outer_steps_median=("outer_steps", "median"),
    outer_steps_p75=("outer_steps", lambda x: np.nanpercentile(x, 75)),
    outer_steps_p95=("outer_steps", lambda x: np.nanpercentile(x, 95)),
    grip_mismatch_sum=("grip_mismatch_total", "sum"),
    grip_boundary_sum=("grip_boundary_total", "sum"),
).reset_index()
margin["success_rate"] = margin["success"] / margin["runs"].clip(lower=1)
print("\nRAW SUCCESS-MARGIN TABLE BY MODE")
print(margin.sort_values(["success_rate", "outer_steps_median"], ascending=[False, True]).to_string(index=False))
margin.to_csv(Path(SUMMARIES) / "success_margin_by_mode.csv", index=False)

# Paired deltas vs reference for episodes where both reference and quant succeeded.
group_cols = ["suite", "task_id", "episode_id"]
piv_succ = df.pivot_table(index=group_cols, columns="mode", values="success", aggfunc="max")
piv_env = df.pivot_table(index=group_cols, columns="mode", values="env_steps", aggfunc="median")
piv_outer = df.pivot_table(index=group_cols, columns="mode", values="outer_steps", aggfunc="median")

paired_delta_rows = []
if REFERENCE_MODE not in piv_succ.columns:
    print("WARNING: reference mode missing; cannot compute paired deltas")
else:
    for m in [c for c in piv_succ.columns if c != REFERENCE_MODE]:
        both_succ = (piv_succ[REFERENCE_MODE] == True) & (piv_succ[m] == True)
        delta_env = (piv_env[m] - piv_env[REFERENCE_MODE])[both_succ].dropna()
        delta_outer = (piv_outer[m] - piv_outer[REFERENCE_MODE])[both_succ].dropna()
        paired_delta_rows.append({
            "mode": m,
            "paired_success_groups": int(both_succ.sum()),
            "delta_env_steps_median_quant_minus_ref": float(delta_env.median()) if len(delta_env) else np.nan,
            "delta_env_steps_p75": float(np.percentile(delta_env, 75)) if len(delta_env) else np.nan,
            "delta_env_steps_p95": float(np.percentile(delta_env, 95)) if len(delta_env) else np.nan,
            "delta_outer_steps_median_quant_minus_ref": float(delta_outer.median()) if len(delta_outer) else np.nan,
            "delta_outer_steps_p75": float(np.percentile(delta_outer, 75)) if len(delta_outer) else np.nan,
            "delta_outer_steps_p95": float(np.percentile(delta_outer, 95)) if len(delta_outer) else np.nan,
        })
    paired_delta = pd.DataFrame(paired_delta_rows)
    print("\nPAIRED SUCCESS-ONLY DELTAS VS REFERENCE")
    print(paired_delta.to_string(index=False))
    paired_delta.to_csv(Path(SUMMARIES) / "paired_success_margin_deltas_vs_reference.csv", index=False)

print("\nSAVED:")
print(Path(SUMMARIES) / "success_margin_by_mode.csv")
print(Path(SUMMARIES) / "paired_success_margin_deltas_vs_reference.csv")


CELL 14F — SUCCESS-MARGIN / EFFICIENCY ANALYSIS
Goal: if success rates are too high, use env_steps/outer_steps/gripper risk as margin degradation signals.

RAW SUCCESS-MARGIN TABLE BY MODE
               mode  runs  success  env_steps_median  env_steps_p75  env_steps_p95  outer_steps_median  outer_steps_p75  outer_steps_p95  grip_mismatch_sum  grip_boundary_sum  success_rate
 v40_w8a8_ALL_00_31    50       48             255.5         316.25          581.3                19.0            23.25            41.75                  0                  0          0.96
 v40_w8a8_CUM_00_15    50       47             261.5         321.50          815.4                19.0            23.50            58.40                  0                  0          0.94
  v40_w8a8_R1_08_15    50       46             262.5         326.25          990.0                19.0            23.75            71.00                  0                  0          0.92
  v40_w8a8_R3_24_31    50       46             261.0   

In [33]:
# CELL 14G — Duplicate-run reproducibility comparator (optional, for paper-level proof)
# Run this only after you have TWO fresh run folders from the same notebook/config/seeds.
# It compares success/fail, env_steps, outer_steps, and drift ratios for the same manifest keys.

import json
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 280)

print("=" * 120)
print("CELL 14G — DUPLICATE-RUN REPRODUCIBILITY COMPARATOR")
print("=" * 120)

# Fill these in after you have two finished runs. Leave RUN_ROOT_B=None to skip.
RUN_ROOT_A = Path(globals().get("RUN_ROOT", ""))
RUN_ROOT_B = None  # Example: Path('/content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/<second_run_id>')

if RUN_ROOT_B is None:
    print("SKIP: set RUN_ROOT_B to a second finished fresh run folder to compare reproducibility.")
else:
    RUN_ROOT_B = Path(RUN_ROOT_B)

    def load_run(root):
        root = Path(root)
        summaries = root / "summaries"
        results = root / "results"
        manifest_path = summaries / "manifest_v40_linear130_split_ref5ep_windows_cumulative.json"
        if not manifest_path.exists():
            raise RuntimeError(f"Expected manifest not found: {manifest_path}")
        manifest = json.loads(manifest_path.read_text())
        stems = {str(x.get("result_stem", "")) for x in manifest}
        rows = []
        for p in sorted(results.glob("*/*.done.json")):
            d = json.loads(p.read_text())
            if str(d.get("result_stem", "")) not in stems:
                continue
            ratios = d.get("drift_ratio_vs_w8a16_reference", {}) or {}
            rows.append({
                "key": (str(d.get("suite")), int(d.get("task_id", -1)), int(d.get("episode_id", -1)), str(d.get("ablation_mode"))),
                "suite": str(d.get("suite")),
                "task_id": int(d.get("task_id", -1)),
                "episode_id": int(d.get("episode_id", -1)),
                "mode": str(d.get("ablation_mode")),
                "episode_seed": d.get("episode_seed", np.nan),
                "success": bool(d.get("success", False)),
                "env_steps": float(d.get("env_steps", np.nan)),
                "outer_steps": float(d.get("outer_steps", np.nan)),
                "final_rms_mean_ratio": ratios.get("final_rms_mean_ratio_vs_ref", np.nan),
                "vel_rms_max_ratio": ratios.get("vel_rms_max_ratio_vs_ref", np.nan),
                "endpoint_rms_max_ratio": ratios.get("endpoint_rms_max_ratio_vs_ref", np.nan),
                "state_after_rms_max_ratio": ratios.get("state_after_rms_max_ratio_vs_ref", np.nan),
            })
        df = pd.DataFrame(rows).drop_duplicates(subset=["key"], keep="last")
        return df

    A = load_run(RUN_ROOT_A)
    B = load_run(RUN_ROOT_B)
    merged = A.merge(B, on="key", suffixes=("_A", "_B"), how="outer", indicator=True)

    print("RUN_ROOT_A:", RUN_ROOT_A)
    print("RUN_ROOT_B:", RUN_ROOT_B)
    print("rows_A:", len(A), "rows_B:", len(B), "merged:", len(merged))
    print("merge_status:")
    print(merged["_merge"].value_counts().to_string())

    both = merged[merged["_merge"] == "both"].copy()
    if both.empty:
        raise RuntimeError("No matching keys between runs")

    both["same_seed"] = both["episode_seed_A"].astype(str) == both["episode_seed_B"].astype(str)
    both["same_success"] = both["success_A"] == both["success_B"]
    both["env_steps_delta_B_minus_A"] = both["env_steps_B"] - both["env_steps_A"]
    both["outer_steps_delta_B_minus_A"] = both["outer_steps_B"] - both["outer_steps_A"]

    print("\nREPRO SUMMARY")
    print("same_seed_rate:", float(both["same_seed"].mean()))
    print("same_success_rate:", float(both["same_success"].mean()))
    print("env_steps_abs_delta_median:", float(both["env_steps_delta_B_minus_A"].abs().median()))
    print("outer_steps_abs_delta_median:", float(both["outer_steps_delta_B_minus_A"].abs().median()))

    flip = both[~both["same_success"]].copy()
    print("\nSUCCESS FLIPS BETWEEN RUNS")
    if flip.empty:
        print("No success/fail flips. Practical reproducibility is strong.")
    else:
        show_cols = ["suite_A", "task_id_A", "episode_id_A", "mode_A", "episode_seed_A", "success_A", "success_B", "env_steps_A", "env_steps_B", "outer_steps_A", "outer_steps_B"]
        print(flip[show_cols].to_string(index=False))

    by_mode = both.groupby("mode_A").agg(
        rows=("same_success", "count"),
        same_success_rate=("same_success", "mean"),
        success_flips=("same_success", lambda s: int((~s).sum())),
        env_abs_delta_median=("env_steps_delta_B_minus_A", lambda x: float(np.nanmedian(np.abs(x)))),
        outer_abs_delta_median=("outer_steps_delta_B_minus_A", lambda x: float(np.nanmedian(np.abs(x)))),
    ).reset_index()
    print("\nREPRO BY MODE")
    print(by_mode.to_string(index=False))

    out_dir = Path(RUN_ROOT_A) / "summaries"
    by_mode.to_csv(out_dir / "duplicate_run_repro_by_mode.csv", index=False)
    flip.to_csv(out_dir / "duplicate_run_success_flips.csv", index=False)
    print("\nSAVED:")
    print(out_dir / "duplicate_run_repro_by_mode.csv")
    print(out_dir / "duplicate_run_success_flips.csv")



CELL 14G — DUPLICATE-RUN REPRODUCIBILITY COMPARATOR
SKIP: set RUN_ROOT_B to a second finished fresh run folder to compare reproducibility.


In [34]:
# CELL 14H — Final claim audit: what this run can and cannot support
# Run after 14B/14C/14D/14E/14F. It prints conservative language for rigorous reporting.

import json
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 260)

print("=" * 120)
print("CELL 14H — FINAL CLAIM AUDIT / REPORTING LANGUAGE")
print("=" * 120)

REFERENCE_MODE = globals().get("REFERENCE_MODE", "v40_w8a16_reference")
summary_files = {
    "paired": Path(SUMMARIES) / "paired_binary_outcome_stats_vs_reference.csv",
    "boot": Path(SUMMARIES) / "bootstrap_ci_drift_ratios_by_mode.csv",
    "margin": Path(SUMMARIES) / "success_margin_by_mode.csv",
    "three_layer": Path(SUMMARIES) / "three_layer_combined_verdict.csv",
}
missing_summary_files = []
for name, path in summary_files.items():
    exists = path.exists()
    print(f"{name}_exists:", exists, path)
    if not exists:
        missing_summary_files.append((name, str(path)))
if missing_summary_files:
    raise RuntimeError(
        "14H abort: required upstream analysis files are missing. "
        "Run 14B/14C/14D/14E/14F before writing the final claim audit. Missing: "
        + repr(missing_summary_files)
    )

# Load current manifest-only episode records for baseline quality and suite label.
if "manifest" not in globals():
    manifest_path = Path(SUMMARIES) / "manifest_v40_linear130_split_ref5ep_windows_cumulative.json"
    if not manifest_path.exists():
        raise RuntimeError(f"Expected manifest not found: {manifest_path}")
    manifest = json.loads(manifest_path.read_text())
manifest_stems = {str(x.get("result_stem", "")) for x in manifest}
rows = []
for p in sorted(Path(RESULTS).glob("*/*.done.json")):
    d = json.loads(p.read_text())
    if str(d.get("result_stem", "")) not in manifest_stems:
        continue
    rows.append({
        "mode": d.get("ablation_mode"),
        "suite": d.get("suite"),
        "task_id": int(d.get("task_id", -1)),
        "episode_id": int(d.get("episode_id", -1)),
        "success": bool(d.get("success", False)),
    })

df = pd.DataFrame(rows)
if df.empty:
    raise RuntimeError("No manifest-matched records found")
if len(df) != len(manifest):
    raise RuntimeError(f"14H abort: manifest-matched records incomplete ({len(df)} vs {len(manifest)})")

suite_list = sorted(df["suite"].unique().tolist())
ref_rate = float(df[df["mode"] == REFERENCE_MODE]["success"].mean())
mode_summary = df.groupby("mode").agg(runs=("success", "count"), success=("success", "sum")).reset_index()
mode_summary["success_rate"] = mode_summary["success"] / mode_summary["runs"].clip(lower=1)

print("\nSUITES:", suite_list)
print("REFERENCE_SUCCESS_RATE:", ref_rate)
print("\nMODE SUCCESS SUMMARY")
print(mode_summary.sort_values("success_rate", ascending=False).to_string(index=False))

print("\nCLAIM AUDIT")
if ref_rate >= 0.95:
    print("- Reference is high: success/fail paired damage is interpretable as quantization damage when reference succeeds and quant fails.")
elif ref_rate >= 0.85:
    print("- Reference is moderate: use paired hard-task separation; raw success ranking alone is not enough.")
else:
    print("- Reference is low: treat this suite as a stress test; use continuous drift and paired cases, not raw success rate alone.")

print("- Valid main evidence layers:")
print("  1) Continuous drift distributions across all completed episodes, including successes.")
print("  2) Paired reference-success / quant-fail cases for clean damage.")
print("  3) Task difficulty separation where reference also fails.")
print("  4) Success-margin degradation via env_steps/outer_steps even when all modes succeed.")
print("  5) Optional duplicate-run reproducibility check if you run the same config twice.")

print("\nSAFE WORDING TEMPLATE")
print("Under a manifest-clean, fixed-seed paired evaluation on", suite_list, ", the evaluated A8 window placements showed the following behavior:")
print("- Report success rates only together with reference-success/quant-fail paired damage.")
print("- If success rates are similar, emphasize drift-ratio and margin distributions rather than claiming one mode is better.")
print("- Do not claim universal best window unless the same pattern repeats across duplicate runs and/or another suite.")
print("- Do not mix v34 and v40 as the same experiment; v40 has a different A8 scope and must be reported separately.")

# Save a small text report.
report_path = Path(SUMMARIES) / "final_claim_audit.txt"
with open(report_path, "w") as f:
    f.write("FINAL CLAIM AUDIT\n")
    f.write(f"RUN_ROOT: {RUN_ROOT}\n")
    f.write(f"SUITES: {suite_list}\n")
    f.write(f"REFERENCE_SUCCESS_RATE: {ref_rate}\n")
    f.write("\nMODE SUMMARY\n")
    f.write(mode_summary.sort_values("success_rate", ascending=False).to_string(index=False))
    f.write("\n\nSAFE CLAIM: Use drift distributions + paired damage + hard-task separation + margin, not raw success alone.\n")
print("\nSAVED:", report_path)



CELL 14H — FINAL CLAIM AUDIT / REPORTING LANGUAGE
paired_exists: True /content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a/summaries/paired_binary_outcome_stats_vs_reference.csv
boot_exists: True /content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a/summaries/bootstrap_ci_drift_ratios_by_mode.csv
margin_exists: True /content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a/summaries/success_margin_by_mode.csv
three_layer_exists: True /content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfol

# AUDIT NOTES — result-folder logic and reproducibility

This notebook keeps the original v40 analysis/print sections and adds stricter safeguards:

- Fresh run uses a versioned folder: `EXPERIMENT_ID__NOTEBOOK_BUILD_ID__timestamp__uuid` under `/content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/`.
- There is no automatic resume-latest behavior. Resuming requires explicit `USER_RUN_ID` and `FORCE_NEW_RUN=False`.
- Cell 12B aborts before evaluation if result/drift folders are dirty or if manifest counts do not match the dynamically expected suite×task×episode×mode shape.
- Cell 13B aborts after evaluation unless done JSONs match the manifest exactly, seeds match, and every mode has the dynamically expected number of rows.
- Cell 14 keeps the original comparison/risk output sections but loads only current-manifest rows.
- Cell 14B adds paired success/failure sensitivity.
- Cell 14C adds printed continuous drift-ratio sensitivity.

Reproducibility note: fixed episode seeds plus `task_suite.get_task_init_states(task_id)` and `env.set_init_state(initial_states[episode_id])` give a paired comparison across modes. They remove avoidable initial-condition/random-seed confounds, but GPU/simulator nondeterminism and closed-loop divergence after the first different action can still exist.

- Cell 13C adds a reproducibility proof report with script hashes, manifest/done counts, and client seed-code audit.
- Cell 14D combines the three-layer sensitivity method into one final printed verdict: drift distributions, paired quant damage, and hard-task separation.


Additional rigor cells added in this build:
- Cell 14E: paired binary stats + bootstrap CIs for drift ratios.
- Cell 14F: success-margin / efficiency degradation via env_steps and outer_steps.
- Cell 14G: optional duplicate-run reproducibility comparator.
- Cell 14H: final claim audit and conservative reporting language.
Default suite is now `libero_10`; override with `W8A8_SUITES` before Cell 02 if needed.



All-suite update:
- Default suite list is now `libero_spatial, libero_object, libero_goal, libero_10`.
- Expected default manifest size is dynamic but normally `4 × 10 × 5 × 8 = 1600` rows.
- Existing completed rows can be skipped only in explicit resume mode (`W8DYN_USER_RUN_ID` + `W8DYN_FORCE_NEW_RUN=0`). Fresh mode always creates a new empty folder.
- Cell 13 also removes stale request-level JSONL records for an incomplete row before rerunning that row in resume mode, preventing duplicate request records after interruption.




## Added patch note — resume result path verification

This copy is not a redesign. It preserves the existing generated server/client, manifest, run loop, and analysis cells. The only additions are:

- **Cell 01B**: sets the exact existing `libero_10` run folder and selected suites (`libero_spatial,libero_10`) before Cell 02.
- **Cell 02A**: verifies the correct saved-result location: `RUN_ROOT/results/<mode>/*.done.json`, prints existing summary, checks duplicate stems, seed consistency, and Drive-space warning before any resume run.

Cell 13 still performs the real resume/skip using exact `.done.json` paths under `RESULTS / mode / <result_stem>.done.json`.
